
# 06. Combine Tracking Measurement Validation & Cohort Comparability

## WR-MovementIQ: NFL Big Data Bowl 2027

**Research Project:** Beyond the 40-Yard Dash: Linking NFL Combine Receiver Movement Mechanics to Regular-Season Performance Using Player Tracking Data.

**Notebook:** 06 — Combine Tracking Measurement Validation & Cohort Comparability

**Competition:** NFL Big Data Bowl 2027 — Open Track

**Primary population:** Wide receivers (WR)

**Draft cohorts:** 2023, 2024 and 2025

**Environment:** Python / Google Colab

---

### Notebook purpose

This notebook evaluates the reliability, consistency and comparability of movement-direction measurements derived from NFL Combine tracking data.

Particular attention is given to the relationship between recorded tracking direction and coordinate-derived heading, accumulated turning measurements, speed-based filtering and potential differences across draft cohorts.

The objective is to establish a defensible measurement strategy before using Combine movement features in the integrated player-level modelling framework.



## 1. Research Objectives

The main objective is to establish whether movement-direction and turning characteristics derived from Combine tracking observations are sufficiently reliable for subsequent statistical analysis.

Specific objectives are:

1. Validate recorded direction against heading independently calculated from player coordinates.
2. Reconstruct turning measurements from raw tracking observations.
3. Evaluate the effect of different minimum-speed thresholds on turning estimates.
4. Quantify directional measurement error across draft cohorts.
5. Investigate whether measurement agreement differs across individual Combine drills.
6. Evaluate whether differences in speed distributions explain observed cohort differences in heading error.
7. Establish a primary turning-feature definition and appropriate sensitivity measures.

### Central methodological question

Are differences in calculated turning characteristics between draft cohorts attributable to genuine movement variation, differences in observation-speed composition, or differences in directional measurement behaviour?



## 2. Analytical Framework

This notebook follows five validation stages.

### Stage 1: Directional validation

Compare recorded direction with heading reconstructed from consecutive player coordinates.

### Stage 2: Turning reconstruction

Independently reconstruct recorded-direction and coordinate-derived turning measurements from raw tracking observations.

### Stage 3: Speed-threshold sensitivity

Compare turning estimates using minimum-speed thresholds of:

- 0.0
- 1.0
- 2.0

The objective is to determine how excluding low-speed observations changes accumulated turning measurements.

### Stage 4: Cohort and drill comparability

Evaluate directional error across:

- Draft years: 2023, 2024 and 2025.
- Eleven primary receiver skill drills.
- Multiple movement-speed bands.

### Stage 5: Speed-composition standardization

Determine whether differences in cohort-level heading error persist when all cohorts are evaluated using a common speed distribution.

The resulting evidence will inform the measurement strategy used in subsequent modelling.



## 3. Data Sources

The primary source is the NFL Combine tracking dataset.

### Raw data

`data/raw/nfl-big-data-bowl-2027/combine_tracking.csv`

Relevant variables:

| Variable | Description |
|---|---|
| draft_year | Draft cohort |
| event_id | Tracking event identifier |
| nfl_id | Player identifier |
| entity_type | Tracking entity classification |
| time | Tracking observation timestamp |
| drill_type | Broad drill category |
| drill_name | Specific Combine drill |
| attempt | Attempt identifier |
| x | Player X-coordinate |
| y | Player Y-coordinate |
| s | Recorded speed |
| a | Recorded acceleration |
| dis | Recorded displacement |
| dir | Recorded direction |

Only player observations belonging to the identified wide receiver Combine population are used for the primary analysis.

Ball observations are excluded.

### Supporting datasets

- `combine_results.csv`
- `wr_combine_attempt_features_v2.parquet`
- `wr_combine_turning_candidate_features.parquet`
- `wr_turning_reconstruction_validation.parquet`

### Primary drill selection

The analysis focuses on eleven high-coverage receiver skill drills:

1. GAUNTLET_DRILL
2. DAGGER_ROUTE_LEFT
3. SLANT_ROUTE_LEFT
4. CURL_ROUTE_RIGHT
5. SLOT_STRIKE_ROUTE_LEFT
6. OVER_SHOULDER_ADJUST
7. RED_ZONE_FADE_RIGHT
8. COMEBACK_ROUTE_RIGHT
9. SLOT_SAIL_ROUTE_RIGHT
10. GO_ROUTE_RIGHT
11. POST_CORNER_ROUTE_RIGHT

The 40-yard dash is treated separately as a traditional athletic benchmark.



## 4. Coordinate-Derived Heading Validation

Recorded tracking direction is compared with heading independently calculated from consecutive player coordinates.

For consecutive observations:

$$
\Delta x_t = x_t - x_{t-1}
$$

$$
\Delta y_t = y_t - y_{t-1}
$$

The coordinate-derived heading is:

$$
\theta_t =
\left[
\frac{180}{\pi}
\operatorname{atan2}(\Delta x_t,\Delta y_t)
\right]\bmod 360
$$

This convention measures heading clockwise from the positive Y-axis and was empirically validated against the recorded direction variable.

### Circular Angular Error

Because direction is circular, ordinary subtraction is inappropriate near the 0°/360° boundary.

The angular error is calculated as:

$$
E_t =
\left|
\left[
(\theta_t-dir_t+180)\bmod 360
\right]-180
\right|
$$

The resulting error ranges from 0° to 180°.

### Interpretation

- Small angular error indicates strong agreement.
- Large angular error indicates disagreement between recorded and coordinate-derived direction.
- Low-speed observations require special caution because small coordinate movements can produce unstable heading estimates.



## 5. Turning Measurement Reconstruction

Turning is estimated using consecutive directional observations.

The circular difference between consecutive headings is:

$$
\Delta\theta_t =
\left|
\left[
(\theta_t-\theta_{t-1}+180)\bmod 360
\right]-180
\right|
$$

Accumulated turning is:

$$
T=\sum_{t=1}^{n-1}\Delta\theta_t
$$

The analysis evaluates two independent measurement approaches.

### A. Recorded-Direction Turning

Uses the original recorded `dir` variable.

### B. Coordinate-Derived Turning

Uses heading calculated from consecutive X and Y coordinates.

### Pairwise Speed Filtering

A directional transition is retained when:

$$
\min(s_t,s_{t-1})\geq c
$$

Where $c$ is the minimum-speed threshold.

The primary threshold is 1.0, while 0.0 and 2.0 are used for sensitivity analysis.

### Important Distinction

Accumulated turning is not a direct measurement of agility, athletic ability or route quality.

It is a geometric movement characteristic that must be interpreted in the context of drill design, speed and measurement reliability.



## 6. Validation Metrics

The following metrics are used to evaluate measurement agreement and data quality.

### 6.1 Directional Agreement

The following statistics quantify the difference between recorded tracking direction and coordinate-derived heading:

- Median angular error.
- Mean angular error.
- 90th percentile angular error.
- 95th percentile angular error.

Angular error is measured in degrees (°), with values ranging from 0° to 180°.

### 6.2 Turning Reconstruction Agreement

The following metrics evaluate the agreement between stored and independently reconstructed turning measurements:

- Pearson correlation coefficient.
- Spearman rank correlation coefficient.
- Median absolute error.
- Mean absolute error.
- 95th percentile absolute error.

Correlation measures the strength of association, while absolute error measures the magnitude of differences between measurements.

### 6.3 Turning Coverage

Turning coverage measures the proportion of expected directional transitions for which valid turning measurements can be calculated.

$$
\text{Coverage} =
\frac{\text{Valid directional transitions}}
{\text{Expected directional transitions}}
\times 100
$$

Coverage is evaluated at three levels:

- Attempt level.
- Drill level.
- Draft cohort level.

### 6.4 Speed Consistency

Observation intervals and speed distributions are examined to identify potential differences in tracking quality and movement composition.

The analysis considers:

- Median sampling interval.
- Maximum sampling interval.
- Speed distribution across draft cohorts.
- Number of observations within each speed band.
- Heading error across speed bands.

### 6.5 Interpretation Principle

High correlation alone does not establish measurement equivalence.

Absolute differences, distributional behaviour, valid observation coverage and cohort-specific agreement must also be considered.

A measurement can have a high correlation with another measurement while still exhibiting systematic differences in magnitude.



## 7. Speed-Threshold Sensitivity Analysis

Low-speed observations can produce unstable coordinate-derived headings.

Three thresholds are evaluated:

| Threshold | Purpose |
|---|---|
| 0.0 | Retain all available transitions |
| 1.0 | Primary directional reliability threshold |
| 2.0 | Stricter sensitivity analysis |

The primary threshold of 1.0 was selected as a compromise between directional reliability and retaining sufficient movement transitions.

The analysis also examines:

- Number of valid transitions.
- Turning-pair coverage.
- Total accumulated turning.
- Mean turning per valid transition.
- Turning rate.
- Agreement between threshold-specific measurements.

Attempts with low turning coverage are flagged rather than automatically discarded.

Coverage categories:

| Coverage | Classification |
|---|---|
| Below 70% | Low |
| 70–80% | Moderate |
| 80% or above | Good |

These categories are quality-control indicators, not proof of measurement validity.



## 8. Cohort Comparability Analysis

A major concern is whether turning measurements are comparable across the 2023, 2024 and 2025 draft cohorts.

The analysis evaluates:

1. Recorded-direction and coordinate-heading agreement by cohort.
2. Heading error across movement-speed bands.
3. Heading error within individual Combine drills.
4. Turning reconstruction errors by cohort.
5. Turning-feature distributions by cohort and drill.
6. The effect of speed thresholds on cohort-level estimates.

### Speed bands

Observations are grouped into:

- ≤0.5
- 0.5–1
- 1–2
- 2–3
- >3

The speed-stratified analysis helps distinguish differences in heading error from differences in the proportion of observations occurring at low speeds.

### Important research precaution

A lower heading error in one cohort does not automatically establish that its tracking data are more accurate overall.

Differences may reflect observation composition, drill characteristics, movement behaviour or other measurement-related factors.



## 9. Speed-Composition Standardization

The cohorts may have different proportions of low-speed and high-speed observations.

Because heading error generally increases at low speeds, comparing unadjusted cohort-level mean errors can be misleading.

To address this issue, heading errors are standardized using a common pooled speed distribution.

For cohort $c$:

$$
E_c^{std} =
\sum_{b=1}^{B}w_b E_{c,b}
$$

Where:

- $E_{c,b}$ is the mean heading error for cohort $c$ in speed band $b$.
- $w_b$ is the pooled proportion of observations in speed band $b$.
- $B$ represents the number of speed bands.

The standardized estimates can then be compared with each cohort's observed mean heading error.

### Interpretation

If cohort differences become substantially smaller after standardization, speed composition may explain part of the observed discrepancy.

If differences remain, further investigation of measurement behaviour is warranted.

Standardization does not eliminate every possible source of measurement non-comparability.



## 10. Empirical Results

### 10.1 Directional validation

The initial validation demonstrates that recorded direction and coordinate-derived heading generally agree more closely at higher movement speeds.

At speeds above 3, median heading errors were:

| Draft cohort | Median error |
|---|---:|
| 2023 | 1.034° |
| 2024 | 1.046° |
| 2025 | 0.412° |

At speeds of 0.5 or below, directional errors were substantially higher.

### 10.2 Drill-level agreement

Heading agreement varies across individual drills.

The 2025 cohort generally exhibits lower errors across the primary drills, although the size of the difference varies by drill.

### 10.3 Measurement reconstruction

Independent reconstruction confirms that a speed threshold of 1.0 closely reproduces the stored recorded-direction and coordinate-derived turning features.

This supports the reproducibility of the selected calculation procedure.

### 10.4 Cohort differences

The initial results identify differences between recorded-direction and coordinate-derived turning measurements across draft cohorts.

These differences require careful interpretation and must not automatically be attributed to genuine differences in player movement ability.

### 10.5 Speed standardization

The speed-composition standardization results will be used to determine how much of the observed cohort-level heading-error difference is attributable to different speed distributions.

**Final interpretation should be updated after the standardized-error results have been evaluated.**



## 11. Measurement and Feature Selection Decisions

The following provisional measurement policy is adopted.

| Feature | Proposed treatment |
|---|---|
| Recorded total turn | Primary turning measure |
| Recorded mean turn | Supporting turning measure |
| Coordinate total turn | Sensitivity measure |
| Coordinate mean turn | Supporting sensitivity measure |
| Coordinate 95th percentile turn | Exploratory sensitivity measure |
| Turning rate | Exploratory measure |
| Legacy total direction change | Exclude from primary model pending precise definition |

### Primary threshold

Use a minimum speed threshold of 1.0.

### Sensitivity threshold

Repeat the analysis at a threshold of 2.0.

### Quality control

Retain turning-pair coverage flags and heading-error diagnostics.

### Research caution

Recorded-direction and coordinate-derived turning should not be treated as interchangeable merely because their correlation is high.

Their agreement must be evaluated at the appropriate cohort, drill and speed levels.



## 12. Limitations

Several limitations must be acknowledged.

1. Directional agreement does not establish the absolute accuracy of either measurement source.
2. Coordinate-derived heading becomes unstable when player displacement is very small.
3. Accumulated angular differences can amplify small per-observation discrepancies.
4. Different drills have different movement geometries and should not be interpreted as equivalent movement tests.
5. Cohort differences may reflect measurement behaviour, movement composition or other unobserved factors.
6. Speed-threshold filtering removes some genuine low-speed transitions.
7. The available three draft cohorts provide limited evidence for broad temporal generalization.
8. Turning characteristics are geometric descriptors and should not be interpreted as direct measures of agility or athletic superiority.

Consequently, all conclusions should be framed as measurement and observational findings rather than causal explanations.



## 13. Generated Outputs

### Data audit reports

Saved under:

`reports/data_audit/`

Expected outputs include:

- `turning_reconstruction_overall.csv`
- `turning_reconstruction_by_cohort.csv`
- `heading_error_by_cohort_speed.csv`
- `heading_error_by_cohort_drill.csv`
- `heading_speed_composition_by_cohort.csv`
- `heading_speed_standardized_error.csv`
- `heading_error_above_speed_1.csv`
- `turning_threshold_sensitivity_by_cohort_drill.csv`
- `player_turning_by_drill.csv`
- `player_turning_distribution_by_cohort_drill.csv`
- `player_turning_cohort_comparison.csv`

### Feature validation data

Saved under:

`data/features/`

- `wr_turning_reconstruction_validation.parquet`

### Notebook

`notebooks/06_combine_tracking_measurement_validation.ipynb`

All outputs should be reproducible from the raw data and documented processing procedures.



## 14. Conclusion and Transition to Modelling

This notebook establishes a measurement-validation framework for Combine receiver movement characteristics.

The analysis demonstrates why directional reliability, turning reconstruction, speed filtering and cohort comparability must be investigated before using accumulated turning measurements as explanatory variables.

The provisional measurement strategy prioritizes recorded-direction turning at a minimum-speed threshold of 1.0, with coordinate-derived turning retained for sensitivity analysis.

The speed-composition standardization analysis provides an additional diagnostic for understanding cohort-level differences.

### Transition to the next stage

The validated features will inform the subsequent modelling phase of WR-MovementIQ.

The next stage will investigate whether Combine movement characteristics explain variation in regular-season receiver movement characteristics.

The modelling framework will preserve player-level grouping during validation and distinguish exploratory associations from genuine out-of-sample predictive performance.

**Research principle:** Reliable feature construction must precede predictive modelling.


In [1]:

# ============================================================
# NOTEBOOK 06 — ENVIRONMENT SETUP
# WR-MovementIQ | NFL Big Data Bowl 2027
# ============================================================

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

MOUNT_PATH = Path("/content/gdrive_new")

if not (MOUNT_PATH / "MyDrive").exists():
    drive.mount(str(MOUNT_PATH))

# ------------------------------------------------------------
# 2. DEFINE PROJECT DIRECTORIES
# ------------------------------------------------------------

ROOT = MOUNT_PATH / "MyDrive" / "NFL-BigDataBowl-2027"

RAW_DIR = ROOT / "data" / "raw" / "nfl-big-data-bowl-2027"
FEATURE_DIR = ROOT / "data" / "features"
PROCESSED_DIR = ROOT / "data" / "processed"

NOTEBOOK_DIR = ROOT / "notebooks"

AUDIT_DIR = ROOT / "reports" / "data_audit"
FIGURE_DIR = ROOT / "reports" / "figures"
MODEL_DIR = ROOT / "reports" / "model_results"

# ------------------------------------------------------------
# 3. CREATE REQUIRED OUTPUT DIRECTORIES
# ------------------------------------------------------------

for directory in [
    AUDIT_DIR,
    FIGURE_DIR,
    MODEL_DIR,
    NOTEBOOK_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 4. VERIFY SOURCE DATASETS
# ------------------------------------------------------------

required_files = {
    "Combine tracking": RAW_DIR / "combine_tracking.csv",
    "Combine results": RAW_DIR / "combine_results.csv",
    "Players": RAW_DIR / "players.csv",
    "Game movement features": FEATURE_DIR / "wr_game_movement_features.parquet",
    "Combine attempt features": FEATURE_DIR / "wr_combine_attempt_features_enriched.parquet",
    "Turning candidates": FEATURE_DIR / "wr_combine_turning_candidate_features.parquet",
    "Turning reconstruction": FEATURE_DIR / "wr_turning_reconstruction_validation.parquet",
    "Combine player profiles": FEATURE_DIR / "wr_combine_player_profiles.parquet",
    "Integrated dataset": FEATURE_DIR / "wr_combine_game_integrated_v2.parquet"
}

verification = []

for name, path in required_files.items():
    verification.append({
        "dataset": name,
        "exists": path.exists(),
        "path": str(path)
    })

verification_df = pd.DataFrame(verification)

print("=" * 65)
print("WR-MOVEMENTIQ — NOTEBOOK 06 ENVIRONMENT CHECK")
print("=" * 65)

print("\nPROJECT ROOT:")
print(ROOT)

print("\nDATASET VERIFICATION:")
display(verification_df)

missing_files = verification_df[
    ~verification_df["exists"]
]

if len(missing_files) > 0:
    print("\nWARNING: SOME REQUIRED FILES ARE MISSING.")
    display(missing_files)
else:
    print("\nALL REQUIRED DATASETS VERIFIED.")

# ------------------------------------------------------------
# 5. VERIFY OUTPUT DIRECTORIES
# ------------------------------------------------------------

print("\nOUTPUT DIRECTORIES:")
for directory in [
    AUDIT_DIR,
    FIGURE_DIR,
    MODEL_DIR,
    NOTEBOOK_DIR
]:
    print(f"{directory}: {directory.exists()}")

# ------------------------------------------------------------
# 6. ENVIRONMENT SUMMARY
# ------------------------------------------------------------

print("\nENVIRONMENT:")
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

print("\nNOTEBOOK 06 SETUP COMPLETE.")


Mounted at /content/gdrive_new
WR-MOVEMENTIQ — NOTEBOOK 06 ENVIRONMENT CHECK

PROJECT ROOT:
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027

DATASET VERIFICATION:


,dataset,exists,path
0,Combine tracking,True,/content/gdrive_new/MyDrive/NFL-BigDataBowl-20...
1,Combine results,True,/content/gdrive_new/MyDrive/NFL-BigDataBowl-20...
2,Players,True,/content/gdrive_new/MyDrive/NFL-BigDataBowl-20...
3,Game movement features,True,/content/gdrive_new/MyDrive/NFL-BigDataBowl-20...
4,Combine attempt features,True,/content/gdrive_new/MyDrive/NFL-BigDataBowl-20...
5,Turning candidates,True,/content/gdrive_new/MyDrive/NFL-BigDataBowl-20...
6,Turning reconstruction,True,/content/gdrive_new/MyDrive/NFL-BigDataBowl-20...
7,Combine player profiles,True,/content/gdrive_new/MyDrive/NFL-BigDataBowl-20...
8,Integrated dataset,True,/content/gdrive_new/MyDrive/NFL-BigDataBowl-20...



ALL REQUIRED DATASETS VERIFIED.

OUTPUT DIRECTORIES:
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit: True
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/figures: True
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results: True
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/notebooks: True

ENVIRONMENT:
Python version: 3.13.18
Pandas version: 2.2.3
NumPy version: 2.4.6

NOTEBOOK 06 SETUP COMPLETE.


In [4]:

# ============================================================
# NOTEBOOK 06 — STEP 5
# RAW COMBINE TRACKING DATA INSPECTION
# ============================================================

# 1. LOAD DATASETS

combine_tracking = pd.read_csv(COMBINE_TRACKING_PATH)
combine_results = pd.read_csv(COMBINE_RESULTS_PATH)
players = pd.read_csv(PLAYERS_PATH)

print("=" * 70)
print("RAW COMBINE TRACKING DATA")
print("=" * 70)

# 2. DATASET DIMENSIONS

print("\nDATASET DIMENSIONS")
print("Tracking rows:", len(combine_tracking))
print("Tracking columns:", len(combine_tracking.columns))

# 3. COLUMN NAMES AND DATA TYPES

print("\nCOLUMN NAMES")
print(combine_tracking.columns.tolist())

print("\nDATA TYPES")
display(
    combine_tracking.dtypes.astype(str)
    .rename("data_type")
    .to_frame()
)

# 4. MISSING VALUE AUDIT

missing_audit = pd.DataFrame({
    "missing_count": combine_tracking.isna().sum(),
    "missing_pct": (
        combine_tracking.isna().mean() * 100
    )
}).sort_values("missing_pct", ascending=False)

print("\nMISSING VALUE AUDIT")
display(missing_audit.round(3))

# 5. DRAFT COHORT DISTRIBUTION

print("\nDRAFT COHORT DISTRIBUTION")

cohort_counts = (
    combine_tracking.groupby("draft_year")
    .agg(
        observations=("nfl_id", "size"),
        unique_players=("nfl_id", "nunique")
    )
    .reset_index()
)

display(cohort_counts)

# 6. ENTITY TYPE DISTRIBUTION

print("\nENTITY TYPE DISTRIBUTION")

entity_counts = (
    combine_tracking.groupby("entity_type")
    .agg(
        observations=("nfl_id", "size"),
        unique_players=("nfl_id", "nunique")
    )
    .reset_index()
)

display(entity_counts)

# 7. IDENTIFY WIDE RECEIVER COMBINE PLAYERS

wr_results = combine_results[
    combine_results["combine_position"]
    .astype(str).str.upper().eq("WR")
].copy()

wr_ids = set(wr_results["nfl_id"].dropna())

print("\nWIDE RECEIVER POPULATION")
print("WR Combine records:", len(wr_results))
print("Unique WR players:", len(wr_ids))

# 8. FILTER PLAYER TRACKING OBSERVATIONS

wr_tracking = combine_tracking[
    (combine_tracking["entity_type"] == "PLAYER") &
    (combine_tracking["nfl_id"].isin(wr_ids))
].copy()

print("\nWR TRACKING DATA")
print("Observations:", len(wr_tracking))
print("Unique players:", wr_tracking["nfl_id"].nunique())

# 9. DRILL DISTRIBUTION

print("\nDRILL DISTRIBUTION")

drill_summary = (
    wr_tracking.groupby("drill_name")
    .agg(
        observations=("nfl_id", "size"),
        players=("nfl_id", "nunique"),
        attempts=("attempt", "nunique")
    )
    .reset_index()
    .sort_values("players", ascending=False)
)

display(drill_summary)

# 10. DRILL COVERAGE BY DRAFT COHORT

print("\nDRILL COVERAGE BY DRAFT COHORT")

cohort_drill = (
    wr_tracking.groupby(
        ["draft_year", "drill_name"]
    )
    .agg(
        observations=("nfl_id", "size"),
        players=("nfl_id", "nunique")
    )
    .reset_index()
)

display(cohort_drill)

# 11. MOVEMENT VARIABLE COMPLETENESS

movement_columns = [
    "x", "y", "s", "a", "dis", "dir"
]

movement_columns = [
    col for col in movement_columns
    if col in wr_tracking.columns
]

movement_quality = pd.DataFrame({
    "variable": movement_columns,
    "observations": len(wr_tracking),
    "missing_count": [
        wr_tracking[col].isna().sum()
        for col in movement_columns
    ],
    "missing_pct": [
        wr_tracking[col].isna().mean() * 100
        for col in movement_columns
    ]
})

print("\nMOVEMENT VARIABLE COMPLETENESS")
display(movement_quality.round(4))

# 12. SAVE AUDIT OUTPUTS

cohort_counts.to_csv(
    AUDIT_DIR / "notebook06_cohort_distribution.csv",
    index=False
)

entity_counts.to_csv(
    AUDIT_DIR / "notebook06_entity_distribution.csv",
    index=False
)

drill_summary.to_csv(
    AUDIT_DIR / "notebook06_wr_drill_distribution.csv",
    index=False
)

cohort_drill.to_csv(
    AUDIT_DIR / "notebook06_drill_coverage_by_cohort.csv",
    index=False
)

movement_quality.to_csv(
    AUDIT_DIR / "notebook06_raw_movement_completeness.csv",
    index=False
)

# 13. FINAL CHECKS

print("\n" + "=" * 70)
print("STEP 5 COMPLETED")
print("=" * 70)

print("WR tracking observations:", len(wr_tracking))
print("WR players:", wr_tracking["nfl_id"].nunique())
print("Draft cohorts:", sorted(wr_tracking["draft_year"].dropna().unique()))

print("\nAUDIT REPORTS SAVED TO:")
print(AUDIT_DIR)


RAW COMBINE TRACKING DATA

DATASET DIMENSIONS
Tracking rows: 463189
Tracking columns: 14

COLUMN NAMES
['draft_year', 'event_id', 'nfl_id', 'entity_type', 'time', 'drill_type', 'drill_name', 'attempt', 'x', 'y', 's', 'a', 'dis', 'dir']

DATA TYPES


,data_type
draft_year,int64
event_id,object
nfl_id,int64
entity_type,object
time,object
drill_type,object
drill_name,object
attempt,int64
x,float64
y,float64



MISSING VALUE AUDIT


,missing_count,missing_pct
draft_year,0,0.0
event_id,0,0.0
nfl_id,0,0.0
entity_type,0,0.0
time,0,0.0
drill_type,0,0.0
drill_name,0,0.0
attempt,0,0.0
x,0,0.0
y,0,0.0



DRAFT COHORT DISTRIBUTION


,draft_year,observations,unique_players
0,2023,136097,160
1,2024,147815,177
2,2025,179277,173


... (5 outputs left out)

,draft_year,drill_name,observations,players
0,2023,COMEBACK_ROUTE_RIGHT,1983,36
1,2023,CURL_ROUTE_RIGHT,2294,37
2,2023,DAGGER_ROUTE_LEFT,2936,39
3,2023,FORTY_YARD_DASH,3123,33
4,2023,GAUNTLET_DRILL,6432,39
5,2023,GO_ROUTE_RIGHT,3036,35
6,2023,OVER_SHOULDER_ADJUST,2431,37
7,2023,POST_CORNER_ROUTE_LEFT,2190,39
8,2023,POST_CORNER_ROUTE_RIGHT,2505,36
9,2023,RED_ZONE_FADE_RIGHT,1987,36



MOVEMENT VARIABLE COMPLETENESS


,variable,observations,missing_count,missing_pct
0,x,109508,0,0.0
1,y,109508,0,0.0
2,s,109508,0,0.0
3,a,109508,0,0.0
4,dis,109508,0,0.0
5,dir,109508,0,0.0



STEP 5 COMPLETED
WR tracking observations: 109508
WR players: 107
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]

AUDIT REPORTS SAVED TO:
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit



## 5. Coordinate-Derived Heading Validation

### 5.1 Objective

This stage evaluates the agreement between recorded tracking direction and heading independently calculated from consecutive player coordinates.

### 5.2 Coordinate-derived heading

For consecutive observations belonging to the same player, drill, attempt and event:

$$
\Delta x_t=x_t-x_{t-1}
$$

$$
\Delta y_t=y_t-y_{t-1}
$$

Coordinate-derived heading is calculated as:

$$
\theta_t =
\left[
\frac{180}{\pi}
\operatorname{atan2}(\Delta x_t,\Delta y_t)
\right]\bmod 360
$$

This convention measures heading clockwise from the positive Y-axis.

### 5.3 Circular angular error

The difference between recorded direction and coordinate-derived heading is calculated using circular angular distance:

$$
E_t =
\left|
\left[
(\theta_t-dir_t+180)\bmod 360
\right]-180
\right|
$$

Angular error ranges from 0° to 180°.

### 5.4 Speed-dependent reliability

Heading estimates may become unstable when player displacement is very small.

The analysis therefore evaluates angular error across five speed bands:

- ≤0.5
- 0.5–1
- 1–2
- 2–3
- >3

The validation is also stratified by draft cohort and primary Combine drill.

### 5.5 Validation principle

Small angular errors indicate stronger agreement between the two measurement approaches.

However, directional agreement does not independently establish the absolute accuracy of either measurement.


In [5]:

# ============================================================
# NOTEBOOK 06 — STEP 6
# COORDINATE-DERIVED HEADING VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. DEFINE PRIMARY DRILLS
# ------------------------------------------------------------

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

# ------------------------------------------------------------
# 2. PREPARE TRACKING DATA
# ------------------------------------------------------------

heading_data = wr_tracking.copy()

heading_data["time"] = pd.to_datetime(
    heading_data["time"],
    errors="coerce"
)

heading_data = heading_data[
    heading_data["drill_name"].isin(PRIMARY_DRILLS)
].copy()

# Sort observations chronologically within each attempt.
GROUP_KEYS = [
    "draft_year",
    "nfl_id",
    "drill_type",
    "drill_name",
    "attempt",
    "event_id"
]

heading_data = heading_data.sort_values(
    GROUP_KEYS + ["time"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 3. CALCULATE COORDINATE DIFFERENCES
# ------------------------------------------------------------

heading_data["previous_x"] = (
    heading_data.groupby(GROUP_KEYS)["x"].shift(1)
)

heading_data["previous_y"] = (
    heading_data.groupby(GROUP_KEYS)["y"].shift(1)
)

heading_data["previous_speed"] = (
    heading_data.groupby(GROUP_KEYS)["s"].shift(1)
)

heading_data["dx"] = (
    heading_data["x"] - heading_data["previous_x"]
)

heading_data["dy"] = (
    heading_data["y"] - heading_data["previous_y"]
)

# ------------------------------------------------------------
# 4. RECONSTRUCT COORDINATE HEADING
# ------------------------------------------------------------

heading_data["coordinate_heading"] = (
    np.degrees(
        np.arctan2(
            heading_data["dx"],
            heading_data["dy"]
        )
    ) % 360
)

# The first observation in each attempt has no preceding
# coordinate observation and is excluded.

heading_data["heading_error"] = np.abs(
    (
        heading_data["coordinate_heading"]
        - heading_data["dir"]
        + 180
    ) % 360 - 180
)

# ------------------------------------------------------------
# 5. REMOVE INVALID TRANSITIONS
# ------------------------------------------------------------

valid_heading = heading_data[
    heading_data["previous_x"].notna()
    & heading_data["previous_y"].notna()
    & heading_data["heading_error"].notna()
    & heading_data["coordinate_heading"].notna()
].copy()

# ------------------------------------------------------------
# 6. VALIDATE DIRECTION AND SPEED
# ------------------------------------------------------------

print("=" * 70)
print("COORDINATE HEADING VALIDATION")
print("=" * 70)

print("\nOBSERVATION COUNTS")
print("Primary drill observations:", len(heading_data))
print("Valid directional transitions:", len(valid_heading))
print("Unique WR players:", valid_heading["nfl_id"].nunique())

print("\nRECORDED DIRECTION RANGE")
print("Minimum:", valid_heading["dir"].min())
print("Maximum:", valid_heading["dir"].max())

print("\nCOORDINATE HEADING RANGE")
print("Minimum:", valid_heading["coordinate_heading"].min())
print("Maximum:", valid_heading["coordinate_heading"].max())

# ------------------------------------------------------------
# 7. CREATE SPEED BANDS
# ------------------------------------------------------------

speed_bins = [-np.inf, 0.5, 1, 2, 3, np.inf]

speed_labels = [
    "<=0.5",
    "0.5-1",
    "1-2",
    "2-3",
    ">3"
]

valid_heading["speed_band"] = pd.cut(
    valid_heading["s"],
    bins=speed_bins,
    labels=speed_labels,
    right=True,
    include_lowest=True
)

# ------------------------------------------------------------
# 8. OVERALL HEADING ERROR
# ------------------------------------------------------------

overall_heading = pd.DataFrame({
    "observations": [len(valid_heading)],
    "median_error": [valid_heading["heading_error"].median()],
    "mean_error": [valid_heading["heading_error"].mean()],
    "p90_error": [valid_heading["heading_error"].quantile(0.90)],
    "p95_error": [valid_heading["heading_error"].quantile(0.95)]
})

print("\n1. OVERALL HEADING ERROR")
display(overall_heading.round(4))

# ------------------------------------------------------------
# 9. HEADING ERROR BY COHORT AND SPEED
# ------------------------------------------------------------

heading_by_speed = (
    valid_heading.groupby(
        ["draft_year", "speed_band"],
        observed=True
    )
    .agg(
        observations=("heading_error", "size"),
        median_error=("heading_error", "median"),
        mean_error=("heading_error", "mean"),
        p90_error=("heading_error", lambda x: x.quantile(.90)),
        p95_error=("heading_error", lambda x: x.quantile(.95))
    )
    .reset_index()
)

print("\n2. HEADING ERROR BY COHORT AND SPEED")
display(heading_by_speed.round(4))

# ------------------------------------------------------------
# 10. HEADING ERROR BY COHORT AND DRILL
#     Restrict to speed > 1
# ------------------------------------------------------------

high_speed_heading = valid_heading[
    valid_heading["s"] > 1
].copy()

heading_by_drill = (
    high_speed_heading.groupby(
        ["draft_year", "drill_name"]
    )
    .agg(
        observations=("heading_error", "size"),
        median_error=("heading_error", "median"),
        mean_error=("heading_error", "mean"),
        p90_error=("heading_error", lambda x: x.quantile(.90))
    )
    .reset_index()
)

print("\n3. HEADING ERROR BY COHORT AND DRILL (SPEED > 1)")
display(heading_by_drill.round(4))

# ------------------------------------------------------------
# 11. SAVE VALIDATION RESULTS
# ------------------------------------------------------------

overall_heading.to_csv(
    AUDIT_DIR / "heading_validation_overall.csv",
    index=False
)

heading_by_speed.to_csv(
    AUDIT_DIR / "heading_error_by_cohort_speed.csv",
    index=False
)

heading_by_drill.to_csv(
    AUDIT_DIR / "heading_error_by_cohort_drill.csv",
    index=False
)

# Save reconstructed observations for reproducibility.
valid_heading.to_parquet(
    FEATURE_DIR / "wr_heading_validation_observations.parquet",
    index=False
)

print("\n" + "=" * 70)
print("STEP 6 COMPLETED")
print("=" * 70)

print("\nOUTPUTS SAVED:")
print(AUDIT_DIR / "heading_validation_overall.csv")
print(AUDIT_DIR / "heading_error_by_cohort_speed.csv")
print(AUDIT_DIR / "heading_error_by_cohort_drill.csv")
print(FEATURE_DIR / "wr_heading_validation_observations.parquet")


COORDINATE HEADING VALIDATION

OBSERVATION COUNTS
Primary drill observations: 84064
Valid directional transitions: 82782
Unique WR players: 107

RECORDED DIRECTION RANGE
Minimum: 0.0
Maximum: 359.9996895

COORDINATE HEADING RANGE
Minimum: 0.0
Maximum: 359.99967727617366

1. OVERALL HEADING ERROR


,observations,median_error,mean_error,p90_error,p95_error
0,82782,1.0222,2.7733,3.9833,7.5651



2. HEADING ERROR BY COHORT AND SPEED


,draft_year,speed_band,observations,median_error,mean_error,p90_error,p95_error
0,2023,<=0.5,782,28.6213,44.6205,106.7986,137.3272
1,2023,0.5-1,846,9.2325,11.5147,23.8272,31.6356
2,2023,1-2,1755,4.3554,5.5404,11.9727,14.3200
3,2023,2-3,1713,2.7213,3.2660,6.7798,8.1683
4,2023,>3,25097,1.0515,1.3249,2.7923,3.4710
5,2024,<=0.5,832,35.1682,50.8987,126.3752,153.4349
6,2024,0.5-1,600,8.5919,11.3095,24.1844,31.8925
7,2024,1-2,1228,4.7612,5.7350,11.8969,14.5306
8,2024,2-3,1305,2.9024,3.5099,7.3677,8.6147
9,2024,>3,20457,1.0713,1.3506,2.8434,3.5514



3. HEADING ERROR BY COHORT AND DRILL (SPEED > 1)


,draft_year,drill_name,observations,median_error,mean_error,p90_error
0,2023,COMEBACK_ROUTE_RIGHT,1872,1.3426,2.0993,4.8540
1,2023,CURL_ROUTE_RIGHT,2167,1.8445,2.7102,6.1505
2,2023,DAGGER_ROUTE_LEFT,2802,1.3805,1.7414,3.2863
3,2023,GAUNTLET_DRILL,6150,0.9728,1.3562,2.8605
4,2023,GO_ROUTE_RIGHT,2900,0.7701,1.0839,2.0021
5,2023,OVER_SHOULDER_ADJUST,2373,1.0027,1.3838,2.7568
6,2023,POST_CORNER_ROUTE_RIGHT,2434,1.0508,1.3729,2.6703
7,2023,RED_ZONE_FADE_RIGHT,1736,1.3334,2.0723,4.5714
8,2023,SLANT_ROUTE_LEFT,1549,1.2371,1.7488,3.4655
9,2023,SLOT_SAIL_ROUTE_RIGHT,2523,1.4725,1.7424,3.3780



STEP 6 COMPLETED

OUTPUTS SAVED:
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/heading_validation_overall.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/heading_error_by_cohort_speed.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/heading_error_by_cohort_drill.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_heading_validation_observations.parquet



## 6. Independent Turning Reconstruction Validation

### 6.1 Objective

This stage independently reconstructs turning measurements from raw Combine tracking observations and evaluates their agreement with previously generated stored turning features.

### 6.2 Turning calculation

For consecutive directional observations, the absolute circular angular change is:

$$
\Delta\theta_t =
\left|
\left[
(\theta_t-\theta_{t-1}+180)\bmod 360
\right]-180
\right|
$$

The minimum-speed condition is:

$$
\min(s_t,s_{t-1}) \geq c
$$

where c represents the selected minimum-speed threshold.

The reconstructed total turning measure is:

$$
T = \sum_{t=2}^{n}
I\left(\min(s_t,s_{t-1})\geq c\right)
\Delta\theta_t
$$

### 6.3 Validation metrics

Reconstructed turning values are compared against stored turning features using:

- Pearson correlation.
- Spearman rank correlation.
- Median absolute error.
- Mean absolute error.
- 95th percentile absolute error.

### 6.4 Sensitivity analysis

Three speed thresholds are evaluated:

- 0.0: no minimum-speed restriction.
- 1.0: primary reconstruction threshold.
- 2.0: stricter sensitivity threshold.

### 6.5 Interpretation

High correlation indicates consistency in relative variation. Absolute errors and cohort-specific differences are also examined because correlation alone does not establish measurement equivalence.


In [6]:

# ============================================================
# NOTEBOOK 06 — STEP 7A
# INSPECT EXISTING TURNING RECONSTRUCTION
# ============================================================

reconstruction_path = (
    FEATURE_DIR / "wr_turning_reconstruction_validation.parquet"
)

reconstruction = pd.read_parquet(reconstruction_path)

print("=" * 70)
print("TURNING RECONSTRUCTION DATASET")
print("=" * 70)

print("\nDIMENSIONS")
print(reconstruction.shape)

print("\nCOLUMNS")
print(reconstruction.columns.tolist())

print("\nDATA TYPES")
display(
    reconstruction.dtypes.astype(str)
    .rename("data_type")
    .to_frame()
)

print("\nMISSING VALUES")
display(
    reconstruction.isna().sum()
    .rename("missing_count")
    .to_frame()
)

print("\nSAMPLE RECORDS")
display(reconstruction.head())

print("\nDRAFT COHORT DISTRIBUTION")
display(
    reconstruction.groupby("draft_year")
    .size()
    .rename("attempts")
    .reset_index()
)

print("\nSTEP 7A COMPLETED.")


TURNING RECONSTRUCTION DATASET

DIMENSIONS
(1282, 19)

COLUMNS
['draft_year', 'nfl_id', 'drill_type', 'drill_name', 'attempt', 'event_id', 'total_direction_change', 'recorded_total_turn', 'coordinate_total_turn', 'recorded_turn_t0', 'coordinate_turn_t0', 'recorded_turn_t1.0', 'coordinate_turn_t1.0', 'valid_pairs_t1.0', 'coverage_t1.0', 'recorded_turn_t2.0', 'coordinate_turn_t2.0', 'valid_pairs_t2.0', 'coverage_t2.0']

DATA TYPES


,data_type
draft_year,int64
nfl_id,int64
drill_type,object
drill_name,object
attempt,int64
event_id,object
total_direction_change,float64
recorded_total_turn,float64
coordinate_total_turn,float64
recorded_turn_t0,float64



MISSING VALUES


,missing_count
draft_year,0
nfl_id,0
drill_type,0
drill_name,0
attempt,0
event_id,0
total_direction_change,0
recorded_total_turn,0
coordinate_total_turn,0
recorded_turn_t0,0



SAMPLE RECORDS


,draft_year,nfl_id,drill_type,drill_name,attempt,event_id,total_direction_change,recorded_total_turn,coordinate_total_turn,recorded_turn_t0,coordinate_turn_t0,recorded_turn_t1.0,coordinate_turn_t1.0,valid_pairs_t1.0,coverage_t1.0,recorded_turn_t2.0,coordinate_turn_t2.0,valid_pairs_t2.0,coverage_t2.0
0,2023,55887,SKILL_DRILLS_WR,OVER_SHOULDER_ADJUST,1,6414e5fde3d8e72dcdc8d202-2023,39.240234,25.672927,84.981280,32.433994,111.546338,20.571961,70.945038,55,0.964912,14.300262,56.908795,52,0.912281
1,2023,55887,SKILL_DRILLS_WR,GAUNTLET_DRILL,2,6414e5fde3d8e72dcdc8d271-2023,107.753066,103.545034,208.127703,106.333251,261.257799,100.713467,208.127703,84,0.954545,99.120751,185.507841,81,0.920455
2,2023,55904,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,1,6414e684e3d8e72dcdcb4ec1-2023,384.820070,171.945925,208.715458,378.139621,523.715441,158.102728,204.316757,51,0.836066,129.055972,163.048160,46,0.754098
3,2023,55920,SKILL_DRILLS_WR,CURL_ROUTE_RIGHT,1,6414e684e3d8e72dcdcb4f32-2023,349.203045,346.196216,418.638762,347.798554,445.203816,344.332027,392.073707,55,0.964912,147.203541,188.997779,41,0.719298
4,2023,55920,SKILL_DRILLS_WR,RED_ZONE_FADE_RIGHT,1,6414e684e3d8e72dcdcb4ff6-2023,71.123641,66.058551,158.338885,69.510950,211.468986,64.685362,144.302642,41,0.911111,41.962493,96.576331,35,0.777778



DRAFT COHORT DISTRIBUTION


,draft_year,attempts
0,2023,468
1,2024,377
2,2025,437



STEP 7A COMPLETED.



## 7. Comparison of Stored and Reconstructed Turning Measurements

### 7.1 Objective

This section quantifies the agreement between previously stored turning measurements and independently reconstructed measurements.

### 7.2 Measurement comparisons

The following comparisons are evaluated:

1. Stored recorded turning against reconstructed recorded turning at thresholds 0, 1.0 and 2.0.
2. Stored coordinate turning against reconstructed coordinate turning at thresholds 0, 1.0 and 2.0.
3. Stored total direction change against unfiltered reconstructed recorded turning.

### 7.3 Statistical evaluation

For each comparison, the following statistics are calculated:

- Pearson correlation.
- Spearman rank correlation.
- Median absolute error.
- Mean absolute error.
- 95th percentile absolute error.

The analysis is repeated by draft cohort to identify possible differences in measurement agreement.

### 7.4 Interpretation

A strong correlation indicates that two measurements vary consistently. However, small absolute errors are also required to support numerical agreement.

Cohort-specific differences are examined before selecting the primary turning measurement for subsequent modelling.


In [7]:

# ============================================================
# NOTEBOOK 06 — STEP 7B
# STORED VS RECONSTRUCTED TURNING VALIDATION
# ============================================================

from scipy.stats import pearsonr, spearmanr

# ------------------------------------------------------------
# 1. VALIDATE ATTEMPT KEYS
# ------------------------------------------------------------

ATTEMPT_KEYS = [
    "draft_year",
    "nfl_id",
    "drill_type",
    "drill_name",
    "attempt",
    "event_id"
]

print("=" * 70)
print("TURNING RECONSTRUCTION AGREEMENT")
print("=" * 70)

print("\nDATASET VALIDATION")
print("Rows:", len(reconstruction))
print("Unique attempt keys:", reconstruction[ATTEMPT_KEYS].drop_duplicates().shape[0])
print("Unique players:", reconstruction["nfl_id"].nunique())

assert reconstruction[ATTEMPT_KEYS].duplicated().sum() == 0, (
    "Duplicate attempt keys detected."
)

# ------------------------------------------------------------
# 2. DEFINE COMPARISONS
# ------------------------------------------------------------

COMPARISONS = [
    {
        "stored": "recorded_total_turn",
        "reconstructed": "recorded_turn_t0",
        "comparison": "Stored recorded vs reconstructed recorded (threshold 0)"
    },
    {
        "stored": "recorded_total_turn",
        "reconstructed": "recorded_turn_t1.0",
        "comparison": "Stored recorded vs reconstructed recorded (threshold 1)"
    },
    {
        "stored": "recorded_total_turn",
        "reconstructed": "recorded_turn_t2.0",
        "comparison": "Stored recorded vs reconstructed recorded (threshold 2)"
    },
    {
        "stored": "coordinate_total_turn",
        "reconstructed": "coordinate_turn_t0",
        "comparison": "Stored coordinate vs reconstructed coordinate (threshold 0)"
    },
    {
        "stored": "coordinate_total_turn",
        "reconstructed": "coordinate_turn_t1.0",
        "comparison": "Stored coordinate vs reconstructed coordinate (threshold 1)"
    },
    {
        "stored": "coordinate_total_turn",
        "reconstructed": "coordinate_turn_t2.0",
        "comparison": "Stored coordinate vs reconstructed coordinate (threshold 2)"
    },
    {
        "stored": "total_direction_change",
        "reconstructed": "recorded_turn_t0",
        "comparison": "Total direction change vs reconstructed recorded (threshold 0)"
    }
]

# ------------------------------------------------------------
# 3. METRIC FUNCTION
# ------------------------------------------------------------

def calculate_agreement(data, stored_col, reconstructed_col):

    paired = data[
        [stored_col, reconstructed_col]
    ].dropna()

    x = paired[stored_col].astype(float)
    y = paired[reconstructed_col].astype(float)

    if len(paired) < 3:
        return None

    absolute_error = np.abs(x - y)

    return {
        "observations": len(paired),
        "pearson_r": pearsonr(x, y).statistic,
        "spearman_r": spearmanr(x, y).statistic,
        "median_absolute_error": absolute_error.median(),
        "mean_absolute_error": absolute_error.mean(),
        "p95_absolute_error": absolute_error.quantile(0.95)
    }

# ------------------------------------------------------------
# 4. OVERALL AGREEMENT
# ------------------------------------------------------------

overall_results = []

for comparison in COMPARISONS:

    metrics = calculate_agreement(
        reconstruction,
        comparison["stored"],
        comparison["reconstructed"]
    )

    overall_results.append({
        "comparison": comparison["comparison"],
        "stored_variable": comparison["stored"],
        "reconstructed_variable": comparison["reconstructed"],
        **metrics
    })

overall_agreement = pd.DataFrame(overall_results)

print("\n1. OVERALL RECONSTRUCTION AGREEMENT")
display(overall_agreement.round(4))

# ------------------------------------------------------------
# 5. COHORT-SPECIFIC AGREEMENT
# ------------------------------------------------------------

cohort_results = []

for comparison in COMPARISONS:

    for cohort in sorted(
        reconstruction["draft_year"].dropna().unique()
    ):

        cohort_data = reconstruction[
            reconstruction["draft_year"] == cohort
        ]

        metrics = calculate_agreement(
            cohort_data,
            comparison["stored"],
            comparison["reconstructed"]
        )

        cohort_results.append({
            "draft_year": cohort,
            "comparison": comparison["comparison"],
            "stored_variable": comparison["stored"],
            "reconstructed_variable": comparison["reconstructed"],
            **metrics
        })

cohort_agreement = pd.DataFrame(cohort_results)

print("\n2. AGREEMENT BY DRAFT COHORT")
display(cohort_agreement.round(4))

# ------------------------------------------------------------
# 6. PRIMARY THRESHOLD COMPARISON
# ------------------------------------------------------------

primary_comparison = overall_agreement[
    overall_agreement["reconstructed_variable"].isin([
        "recorded_turn_t1.0",
        "coordinate_turn_t1.0"
    ])
].copy()

print("\n3. PRIMARY THRESHOLD (1.0) VALIDATION")
display(primary_comparison.round(4))

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

overall_agreement.to_csv(
    AUDIT_DIR / "turning_reconstruction_agreement_overall.csv",
    index=False
)

cohort_agreement.to_csv(
    AUDIT_DIR / "turning_reconstruction_agreement_by_cohort.csv",
    index=False
)

primary_comparison.to_csv(
    AUDIT_DIR / "turning_reconstruction_primary_threshold.csv",
    index=False
)

# ------------------------------------------------------------
# 8. FINAL VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 7B COMPLETED")
print("=" * 70)

print("Overall comparisons:", len(overall_agreement))
print("Cohort comparisons:", len(cohort_agreement))

print("\nSAVED OUTPUTS:")
print(AUDIT_DIR / "turning_reconstruction_agreement_overall.csv")
print(AUDIT_DIR / "turning_reconstruction_agreement_by_cohort.csv")
print(AUDIT_DIR / "turning_reconstruction_primary_threshold.csv")


TURNING RECONSTRUCTION AGREEMENT

DATASET VALIDATION
Rows: 1282
Unique attempt keys: 1282
Unique players: 107

1. OVERALL RECONSTRUCTION AGREEMENT


,comparison,stored_variable,reconstructed_variable,observations,pearson_r,spearman_r,median_absolute_error,mean_absolute_error,p95_absolute_error
0,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t0,1282,0.7145,0.8397,5.8863,47.1565,199.4016
1,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t1.0,1282,0.9983,0.9987,1.4295,2.7222,11.3567
2,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t2.0,1282,0.8626,0.9189,4.5408,19.7990,106.8455
3,Stored coordinate vs reconstructed coordinate ...,coordinate_total_turn,coordinate_turn_t0,1282,0.4467,0.6780,29.1832,113.0455,469.5929
4,Stored coordinate vs reconstructed coordinate ...,coordinate_total_turn,coordinate_turn_t1.0,1282,0.9935,0.9912,4.3987,7.9307,23.4668
5,Stored coordinate vs reconstructed coordinate ...,coordinate_total_turn,coordinate_turn_t2.0,1282,0.8471,0.8766,22.6199,34.8773,128.3462
6,Total direction change vs reconstructed record...,total_direction_change,recorded_turn_t0,1282,0.9810,0.9757,4.1566,11.4566,46.0181



2. AGREEMENT BY DRAFT COHORT


,draft_year,comparison,stored_variable,reconstructed_variable,observations,pearson_r,spearman_r,median_absolute_error,mean_absolute_error,p95_absolute_error
0,2023,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t0,468,0.7954,0.8793,6.5994,45.8541,197.0950
1,2024,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t0,377,0.5736,0.7525,6.7206,47.2564,201.2019
2,2025,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t0,437,0.7563,0.8410,3.8292,48.4651,194.2038
3,2023,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t1.0,468,0.9984,0.9989,1.4013,2.7153,11.2758
4,2024,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t1.0,377,0.9986,0.9988,1.4890,2.3969,7.0580
5,2025,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t1.0,437,0.9979,0.9980,1.4073,3.0101,13.1375
6,2023,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t2.0,468,0.8352,0.9045,4.7859,20.9587,121.2123
7,2024,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t2.0,377,0.8922,0.9431,4.5420,17.4797,94.6048
8,2025,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t2.0,437,0.8678,0.9192,4.4377,20.5579,105.6075
9,2023,Stored coordinate vs reconstructed coordinate ...,coordinate_total_turn,coordinate_turn_t0,468,0.4917,0.6357,45.0000,122.7674,518.7195



3. PRIMARY THRESHOLD (1.0) VALIDATION


,comparison,stored_variable,reconstructed_variable,observations,pearson_r,spearman_r,median_absolute_error,mean_absolute_error,p95_absolute_error
1,Stored recorded vs reconstructed recorded (thr...,recorded_total_turn,recorded_turn_t1.0,1282,0.9983,0.9987,1.4295,2.7222,11.3567
4,Stored coordinate vs reconstructed coordinate ...,coordinate_total_turn,coordinate_turn_t1.0,1282,0.9935,0.9912,4.3987,7.9307,23.4668



STEP 7B COMPLETED
Overall comparisons: 7
Cohort comparisons: 21

SAVED OUTPUTS:
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/turning_reconstruction_agreement_overall.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/turning_reconstruction_agreement_by_cohort.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/turning_reconstruction_primary_threshold.csv



## 7. Independent Turning Reconstruction Results

### 7.1 Reconstruction validation

The independent reconstruction involved 1,282 Combine drill attempts from 107 wide receivers across the 2023, 2024 and 2025 draft cohorts.

All six attempt identifiers were unique, and no missing values were detected in the reconstruction dataset.

### 7.2 Recorded turning validation

The reconstruction using a minimum speed threshold of 1.0 achieved:

- Pearson correlation: 0.9983.
- Spearman correlation: 0.9987.
- Median absolute error: 1.4295 degrees.
- Mean absolute error: 2.7222 degrees.
- 95th percentile absolute error: 11.3567 degrees.

The results indicate very strong agreement between the stored recorded turning measurement and its independent reconstruction.

### 7.3 Coordinate turning validation

At the same threshold, coordinate-derived turning achieved:

- Pearson correlation: 0.9935.
- Spearman correlation: 0.9912.
- Median absolute error: 4.3987 degrees.
- Mean absolute error: 7.9307 degrees.
- 95th percentile absolute error: 23.4668 degrees.

This supports the reproducibility of the stored coordinate-based turning calculation.

### 7.4 Draft cohort consistency

Recorded turning reconstruction agreement remained consistently high across all cohorts:

- 2023 Pearson correlation: 0.9984.
- 2024 Pearson correlation: 0.9986.
- 2025 Pearson correlation: 0.9979.

Coordinate reconstruction correlations were also high, although absolute differences varied by cohort.

### 7.5 Threshold selection

The 1.0 minimum-speed threshold was selected as the primary reconstruction threshold because it most closely reproduced the stored recorded and coordinate turning features.

The 2.0 threshold is retained for sensitivity analysis. The zero threshold is retained as an unrestricted comparison.

### 7.6 Measurement limitations

High agreement between stored and reconstructed measurements establishes computational reproducibility, not equivalence between recorded-direction and coordinate-derived turning.

The cohort differences in heading error and accumulated turning measurements remain important measurement considerations.

### 7.7 Conclusion

The independent reconstruction supports the use of the 1.0 speed threshold for the primary turning feature pipeline.

Recorded and coordinate-derived turning measurements will be retained as distinct measures. Cohort-specific measurement differences will be reported rather than corrected without further evidence.


In [8]:

# ============================================================
# WR-MOVEMENTIQ — NOTEBOOK 06
# STEP 8A: FINAL COMBINE MOVEMENT FEATURE ENGINEERING
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

RAW_DIR = ROOT / "data" / "raw" / "nfl-big-data-bowl-2027"
FEATURE_DIR = ROOT / "data" / "features"
AUDIT_DIR = ROOT / "reports" / "data_audit"
MODEL_DIR = ROOT / "reports" / "model_results"
NOTEBOOK_DIR = ROOT / "notebooks"

for folder in [FEATURE_DIR, AUDIT_DIR, MODEL_DIR, NOTEBOOK_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. PRIMARY DRILL DEFINITION
# ------------------------------------------------------------

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

ATTEMPT_KEYS = [
    "draft_year",
    "nfl_id",
    "drill_type",
    "drill_name",
    "attempt",
    "event_id"
]

PLAYER_KEYS = ["draft_year", "nfl_id"]

# ------------------------------------------------------------
# 3. LOAD SOURCE DATASETS
# ------------------------------------------------------------

attempt_path = FEATURE_DIR / "wr_combine_attempt_features_enriched.parquet"
reconstruction_path = FEATURE_DIR / "wr_turning_reconstruction_validation.parquet"
combine_results_path = RAW_DIR / "combine_results.csv"

required_paths = [
    attempt_path,
    reconstruction_path,
    combine_results_path
]

missing_paths = [str(p) for p in required_paths if not p.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Required files are missing:\n" + "\n".join(missing_paths)
    )

attempts = pd.read_parquet(attempt_path)
turning = pd.read_parquet(reconstruction_path)
combine_results = pd.read_csv(combine_results_path)

print("=" * 70)
print("STEP 8A — SOURCE DATA VALIDATION")
print("=" * 70)

print("Combine attempt rows:", len(attempts))
print("Turning reconstruction rows:", len(turning))
print("Combine results rows:", len(combine_results))

# ------------------------------------------------------------
# 4. VALIDATE REQUIRED IDENTIFIERS
# ------------------------------------------------------------

for col in ATTEMPT_KEYS:
    if col not in attempts.columns:
        raise KeyError(f"Attempt dataset missing key: {col}")

    if col not in turning.columns:
        raise KeyError(f"Turning dataset missing key: {col}")

assert not attempts[ATTEMPT_KEYS].duplicated().any(), (
    "Duplicate keys found in attempt feature dataset."
)

assert not turning[ATTEMPT_KEYS].duplicated().any(), (
    "Duplicate keys found in turning reconstruction dataset."
)

# ------------------------------------------------------------
# 5. FILTER PRIMARY DRILLS
# ------------------------------------------------------------

primary_attempts = attempts[
    attempts["drill_name"].isin(PRIMARY_DRILLS)
].copy()

primary_turning = turning[
    turning["drill_name"].isin(PRIMARY_DRILLS)
].copy()

print("\nPRIMARY DRILL FILTER")
print("Selected attempts:", len(primary_attempts))
print("Turning reconstruction attempts:", len(primary_turning))
print("Players:", primary_attempts["nfl_id"].nunique())
print("Drills represented:", primary_attempts["drill_name"].nunique())

unexpected_drills = sorted(
    set(primary_attempts["drill_name"].dropna().unique())
    - set(PRIMARY_DRILLS)
)

assert len(unexpected_drills) == 0

# ------------------------------------------------------------
# 6. MERGE VERIFIED TURNING RECONSTRUCTION
# ------------------------------------------------------------

turning_columns = [
    "recorded_turn_t1.0",
    "coordinate_turn_t1.0",
    "valid_pairs_t1.0",
    "coverage_t1.0"
]

for col in turning_columns:
    if col not in primary_turning.columns:
        raise KeyError(f"Turning reconstruction missing: {col}")

turning_merge = primary_turning[
    ATTEMPT_KEYS + turning_columns
].copy()

primary_attempts = primary_attempts.merge(
    turning_merge,
    on=ATTEMPT_KEYS,
    how="left",
    validate="one_to_one",
    indicator=True
)

print("\nTURNING MERGE VALIDATION")
print(primary_attempts["_merge"].value_counts(dropna=False))

primary_attempts["turning_merge_status"] = primary_attempts["_merge"]
primary_attempts.drop(columns="_merge", inplace=True)

# Keep only attempts with successfully matched turning reconstruction.
primary_attempts["turning_reconstruction_available"] = (
    primary_attempts["recorded_turn_t1.0"].notna()
)

# ------------------------------------------------------------
# 7. IDENTIFY AVAILABLE MOVEMENT FEATURES
# ------------------------------------------------------------

# Different versions of the attempt feature pipeline may use
# slightly different column names. This mapping checks alternatives.

FEATURE_CANDIDATES = {
    "mean_speed": [
        "mean_speed",
        "mean_speed_raw",
        "attempt_mean_speed"
    ],
    "peak_speed": [
        "peak_speed",
        "peak_speed_raw",
        "attempt_peak_speed"
    ],
    "p95_speed": [
        "p95_speed",
        "p95_speed_raw",
        "attempt_p95_speed"
    ],
    "mean_acceleration": [
        "mean_acceleration",
        "mean_acceleration_raw",
        "attempt_mean_acceleration"
    ],
    "peak_acceleration": [
        "peak_acceleration",
        "peak_acceleration_raw",
        "attempt_peak_acceleration"
    ],
    "coordinate_path_length": [
        "coordinate_path_length",
        "coordinate_path_length_raw",
        "path_length"
    ],
    "net_displacement": [
        "net_displacement",
        "net_displacement_raw"
    ],
    "path_efficiency": [
        "path_efficiency"
    ],
    "coordinate_mean_turn": [
        "coordinate_mean_turn",
        "coordinate_mean_turn_t1.0"
    ],
    "coordinate_p95_turn": [
        "coordinate_p95_turn",
        "coordinate_p95_turn_t1.0"
    ],
    "turning_rate": [
        "turning_rate_degrees_per_second",
        "turning_rate"
    ]
}

resolved_features = {}

for standard_name, candidates in FEATURE_CANDIDATES.items():

    available = [
        candidate
        for candidate in candidates
        if candidate in primary_attempts.columns
    ]

    if available:
        resolved_features[standard_name] = available[0]

# Add independently reconstructed turning measurements.
resolved_features["recorded_total_turn"] = "recorded_turn_t1.0"
resolved_features["coordinate_total_turn"] = "coordinate_turn_t1.0"
resolved_features["turning_coverage"] = "coverage_t1.0"

print("\nRESOLVED MOVEMENT FEATURES")

for standard_name, source_name in resolved_features.items():
    print(f"{standard_name:30s} <- {source_name}")

if len(resolved_features) < 8:
    raise ValueError(
        "Too few movement features were identified. "
        "Inspect primary_attempts.columns before proceeding."
    )

# ------------------------------------------------------------
# 8. CREATE STANDARDIZED ATTEMPT-LEVEL TABLE
# ------------------------------------------------------------

selected_columns = ATTEMPT_KEYS + list(
    dict.fromkeys(resolved_features.values())
)

attempt_features = primary_attempts[selected_columns].copy()

# Rename movement columns to stable analytical names.
reverse_mapping = {
    source: standard
    for standard, source in resolved_features.items()
}

attempt_features.rename(
    columns=reverse_mapping,
    inplace=True
)

# Convert feature columns to numeric.
movement_columns = list(resolved_features.keys())

for col in movement_columns:
    attempt_features[col] = pd.to_numeric(
        attempt_features[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# 9. AGGREGATE ATTEMPTS WITHIN EACH PLAYER-DRILL
# ------------------------------------------------------------

# First aggregate repeated attempts within the same drill.
# This prevents drills with multiple attempts from dominating.

PLAYER_DRILL_KEYS = PLAYER_KEYS + ["drill_name"]

drill_medians = (
    attempt_features
    .groupby(PLAYER_DRILL_KEYS, as_index=False)[movement_columns]
    .median()
)

drill_attempt_counts = (
    attempt_features
    .groupby(PLAYER_DRILL_KEYS)
    .size()
    .reset_index(name="attempts_in_drill")
)

drill_profiles = drill_medians.merge(
    drill_attempt_counts,
    on=PLAYER_DRILL_KEYS,
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 10. AGGREGATE ACROSS DRILLS TO PLAYER LEVEL
# ------------------------------------------------------------

# Each drill contributes one player-drill median.
# Player-level medians and IQRs summarize typical performance
# and variation across the available primary drills.

player_medians = (
    drill_profiles
    .groupby(PLAYER_KEYS)[movement_columns]
    .median()
    .add_suffix("_median_across_drills")
)

player_iqrs = (
    drill_profiles
    .groupby(PLAYER_KEYS)[movement_columns]
    .quantile(0.75)
    .sub(
        drill_profiles.groupby(PLAYER_KEYS)[movement_columns]
        .quantile(0.25)
    )
    .add_suffix("_iqr_across_drills")
)

player_profiles = player_medians.join(player_iqrs).reset_index()

# Drill coverage and attempt counts.
player_coverage = (
    drill_profiles
    .groupby(PLAYER_KEYS)
    .agg(
        primary_drills_observed=("drill_name", "nunique"),
        primary_attempts_observed=("attempts_in_drill", "sum")
    )
    .reset_index()
)

player_profiles = player_profiles.merge(
    player_coverage,
    on=PLAYER_KEYS,
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 11. TURNING COVERAGE QUALITY FLAGS
# ------------------------------------------------------------

coverage_by_player = (
    drill_profiles
    .groupby(PLAYER_KEYS)
    .agg(
        median_turning_coverage=("turning_coverage", "median"),
        drills_with_turning_coverage_below_70_pct=(
            "turning_coverage",
            lambda x: int((x < 0.70).sum())
        ),
        drills_with_turning_coverage_below_80_pct=(
            "turning_coverage",
            lambda x: int((x < 0.80).sum())
        )
    )
    .reset_index()
)

player_profiles = player_profiles.merge(
    coverage_by_player,
    on=PLAYER_KEYS,
    how="left",
    validate="one_to_one"
)

player_profiles["turning_quality_flag"] = np.select(
    [
        player_profiles["median_turning_coverage"] < 0.70,
        player_profiles["median_turning_coverage"] < 0.80
    ],
    [
        "LOW",
        "MODERATE"
    ],
    default="GOOD"
)

# ------------------------------------------------------------
# 12. MERGE TRADITIONAL COMBINE MEASUREMENTS
# ------------------------------------------------------------

wr_results = combine_results[
    combine_results["combine_position"].astype(str).str.upper() == "WR"
].copy()

wr_results = wr_results.drop_duplicates(
    subset=["draft_year", "nfl_id"]
)

traditional_columns = [
    "ten_yd_split",
    "forty",
    "vertical",
    "broad_jump",
    "three_cone",
    "short_shuttle",
    "bench_reps",
    "ngs_athleticism_score",
    "ngs_college_production_score",
    "ngs_final_score"
]

available_traditional = [
    col for col in traditional_columns
    if col in wr_results.columns
]

for col in available_traditional:
    wr_results[col] = pd.to_numeric(
        wr_results[col],
        errors="coerce"
    )

traditional = wr_results[
    PLAYER_KEYS + available_traditional
].copy()

final_features = player_profiles.merge(
    traditional,
    on=PLAYER_KEYS,
    how="left",
    validate="one_to_one",
    indicator=True
)

print("\nTRADITIONAL COMBINE MERGE")
print(final_features["_merge"].value_counts(dropna=False))

final_features.drop(columns="_merge", inplace=True)

# ------------------------------------------------------------
# 13. FEATURE COMPLETENESS AUDIT
# ------------------------------------------------------------

excluded_identifiers = PLAYER_KEYS + [
    "primary_drills_observed",
    "primary_attempts_observed",
    "median_turning_coverage",
    "drills_with_turning_coverage_below_70_pct",
    "drills_with_turning_coverage_below_80_pct",
    "turning_quality_flag"
]

numeric_feature_columns = [
    col for col in final_features.select_dtypes(include=np.number).columns
    if col not in excluded_identifiers
]

missingness = pd.DataFrame({
    "feature": numeric_feature_columns,
    "observations": len(final_features),
    "missing_count": [
        final_features[col].isna().sum()
        for col in numeric_feature_columns
    ],
    "missing_pct": [
        final_features[col].isna().mean() * 100
        for col in numeric_feature_columns
    ],
    "unique_values": [
        final_features[col].nunique(dropna=True)
        for col in numeric_feature_columns
    ]
}).sort_values("missing_pct", ascending=False)

print("\nFEATURE COMPLETENESS")
display(missingness.round(2))

# ------------------------------------------------------------
# 14. FEATURE CORRELATION AUDIT
# ------------------------------------------------------------

correlation_data = final_features[numeric_feature_columns].copy()

# Exclude columns with no variation.
correlation_data = correlation_data.loc[
    :,
    correlation_data.nunique(dropna=True) > 1
]

spearman_matrix = correlation_data.corr(
    method="spearman"
)

upper_triangle = spearman_matrix.where(
    np.triu(np.ones(spearman_matrix.shape), k=1).astype(bool)
)

correlation_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

correlation_pairs.columns = [
    "feature_1",
    "feature_2",
    "spearman_correlation"
]

correlation_pairs["absolute_correlation"] = (
    correlation_pairs["spearman_correlation"].abs()
)

correlation_pairs = correlation_pairs.sort_values(
    "absolute_correlation",
    ascending=False
)

print("\nTOP 20 FEATURE CORRELATIONS")
display(correlation_pairs.head(20).round(3))

# ------------------------------------------------------------
# 15. FINAL FEATURE TABLE VALIDATION
# ------------------------------------------------------------

assert final_features[PLAYER_KEYS].duplicated().sum() == 0
assert final_features["nfl_id"].nunique() == len(final_features)

print("\n" + "=" * 70)
print("FINAL COMBINE FEATURE TABLE")
print("=" * 70)

print("Player rows:", len(final_features))
print("Unique players:", final_features["nfl_id"].nunique())
print("Draft cohorts:", sorted(final_features["draft_year"].unique()))
print("Movement features:", len(movement_columns))
print("Numeric features:", len(numeric_feature_columns))

print("\nPRIMARY DRILL COVERAGE")
display(
    final_features["primary_drills_observed"]
    .value_counts()
    .sort_index()
    .rename_axis("primary_drills_observed")
    .reset_index(name="players")
)

print("\nTURNING QUALITY FLAGS")
display(
    final_features["turning_quality_flag"]
    .value_counts()
    .rename_axis("quality_flag")
    .reset_index(name="players")
)

# ------------------------------------------------------------
# 16. SAVE FINAL DATASETS AND AUDITS
# ------------------------------------------------------------

final_parquet = FEATURE_DIR / "wr_combine_final_player_features.parquet"
final_csv = FEATURE_DIR / "wr_combine_final_player_features.csv"

drill_profile_path = FEATURE_DIR / "wr_combine_final_player_drill_profiles.parquet"

missingness_path = AUDIT_DIR / "combine_final_feature_missingness.csv"
correlation_path = AUDIT_DIR / "combine_final_feature_correlations.csv"

final_features.to_parquet(final_parquet, index=False)
final_features.to_csv(final_csv, index=False)

drill_profiles.to_parquet(drill_profile_path, index=False)

missingness.to_csv(missingness_path, index=False)
correlation_pairs.to_csv(correlation_path, index=False)

print("\n" + "=" * 70)
print("STEP 8A COMPLETED")
print("=" * 70)

for path in [
    final_parquet,
    final_csv,
    drill_profile_path,
    missingness_path,
    correlation_path
]:
    print(path, "| Exists:", path.exists())

print("\nDo not begin modelling until the feature audit has been reviewed.")


STEP 8A — SOURCE DATA VALIDATION
Combine attempt rows: 1771
Turning reconstruction rows: 1282
Combine results rows: 510

PRIMARY DRILL FILTER
Selected attempts: 1282
Turning reconstruction attempts: 1282
Players: 107
Drills represented: 11

TURNING MERGE VALIDATION
_merge
both          1282
left_only        0
right_only       0
Name: count, dtype: int64

RESOLVED MOVEMENT FEATURES
mean_speed                     <- mean_speed
peak_speed                     <- peak_speed
p95_speed                      <- p95_speed
mean_acceleration              <- mean_acceleration
peak_acceleration              <- peak_acceleration
coordinate_path_length         <- coordinate_path_length
net_displacement               <- net_displacement
path_efficiency                <- path_efficiency
coordinate_mean_turn           <- coordinate_mean_turn
coordinate_p95_turn            <- coordinate_p95_turn
turning_rate                   <- turning_rate_degrees_per_second
recorded_total_turn            <- recorded_tu

,feature,observations,missing_count,missing_pct,unique_values
34,bench_reps,107,79,73.83,11
32,three_cone,107,74,69.16,24
33,short_shuttle,107,69,64.49,24
31,broad_jump,107,12,11.21,21
30,vertical,107,11,10.28,26
28,ten_yd_split,107,10,9.35,14
29,forty,107,10,9.35,32
4,peak_acceleration_median_across_drills,107,0,0.00,107
5,coordinate_path_length_median_across_drills,107,0,0.00,107
2,p95_speed_median_across_drills,107,0,0.00,107



TOP 20 FEATURE CORRELATIONS


,feature_1,feature_2,spearman_correlation,absolute_correlation
450,peak_speed_iqr_across_drills,p95_speed_iqr_across_drills,0.959,0.959
269,coordinate_mean_turn_median_across_drills,turning_rate_median_across_drills,0.949,0.949
37,peak_speed_median_across_drills,p95_speed_median_across_drills,0.949,0.949
584,coordinate_mean_turn_iqr_across_drills,turning_rate_iqr_across_drills,0.908,0.908
532,coordinate_path_length_iqr_across_drills,net_displacement_iqr_across_drills,0.832,0.832
268,coordinate_mean_turn_median_across_drills,coordinate_p95_turn_median_across_drills,0.816,0.816
625,recorded_total_turn_iqr_across_drills,coordinate_total_turn_iqr_across_drills,0.793,0.793
271,coordinate_mean_turn_median_across_drills,coordinate_total_turn_median_across_drills,0.779,0.779
297,coordinate_p95_turn_median_across_drills,turning_rate_median_across_drills,0.779,0.779
326,turning_rate_median_across_drills,coordinate_total_turn_median_across_drills,0.763,0.763



FINAL COMBINE FEATURE TABLE
Player rows: 107
Unique players: 107
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]
Movement features: 14
Numeric features: 38

PRIMARY DRILL COVERAGE


,primary_drills_observed,players
0,4,1
1,5,3
2,7,2
3,8,3
4,9,3
5,10,16
6,11,79



TURNING QUALITY FLAGS


,quality_flag,players
0,GOOD,107



STEP 8A COMPLETED
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_final_player_features.parquet | Exists: True
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_final_player_features.csv | Exists: True
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_final_player_drill_profiles.parquet | Exists: True
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/combine_final_feature_missingness.csv | Exists: True
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/combine_final_feature_correlations.csv | Exists: True

Do not begin modelling until the feature audit has been reviewed.


In [9]:

# =====================================================================
# NOTEBOOK 06 — STEP 8B: FINAL COMBINE FEATURE AUDIT
# =====================================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# 1. PROJECT PATHS
# ---------------------------------------------------------------------

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

FEATURE_DIR = ROOT / "data" / "features"
AUDIT_DIR = ROOT / "reports" / "data_audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = FEATURE_DIR / "wr_combine_final_player_features.parquet"

df = pd.read_parquet(INPUT_FILE)

print("=" * 75)
print("STEP 8B — FINAL COMBINE FEATURE AUDIT")
print("=" * 75)

print("Dataset shape:", df.shape)
print("Unique players:", df["nfl_id"].nunique())
print("Draft cohorts:", sorted(df["draft_year"].dropna().unique()))


# ---------------------------------------------------------------------
# 2. IDENTIFY NUMERIC PREDICTOR COLUMNS
# ---------------------------------------------------------------------

excluded_columns = [
    "nfl_id",
    "draft_year"
]

numeric_columns = [
    c for c in df.select_dtypes(include=np.number).columns
    if c not in excluded_columns
]

print("\nNumeric feature count:", len(numeric_columns))


# ---------------------------------------------------------------------
# 3. FEATURE DISTRIBUTION AND VARIABILITY
# ---------------------------------------------------------------------

distribution = df[numeric_columns].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

distribution["missing_count"] = df[numeric_columns].isna().sum()
distribution["missing_pct"] = (
    df[numeric_columns].isna().mean() * 100
)

distribution["unique_values"] = df[numeric_columns].nunique()

distribution["zero_variance"] = (
    df[numeric_columns].nunique() <= 1
)

distribution.to_csv(
    AUDIT_DIR / "step8b_feature_distribution.csv"
)

print("\nFEATURE DISTRIBUTION")
display(distribution.round(3))


# ---------------------------------------------------------------------
# 4. HIGH-CORRELATION FEATURE PAIRS
# ---------------------------------------------------------------------

corr = df[numeric_columns].corr(method="spearman")

upper_triangle = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

correlation_pairs = (
    upper_triangle.stack()
    .reset_index()
)

correlation_pairs.columns = [
    "feature_1",
    "feature_2",
    "spearman_correlation"
]

correlation_pairs["absolute_correlation"] = (
    correlation_pairs["spearman_correlation"].abs()
)

correlation_pairs = correlation_pairs.sort_values(
    "absolute_correlation",
    ascending=False
)

correlation_pairs.to_csv(
    AUDIT_DIR / "step8b_feature_correlations.csv",
    index=False
)

print("\nTOP 30 FEATURE CORRELATIONS")
display(correlation_pairs.head(30).round(3))

print("\nHIGHLY CORRELATED PAIRS: ABSOLUTE CORRELATION >= 0.80")

high_corr = correlation_pairs[
    correlation_pairs["absolute_correlation"] >= 0.80
]

display(high_corr.round(3))


# ---------------------------------------------------------------------
# 5. COHORT-LEVEL FEATURE DISTRIBUTIONS
# ---------------------------------------------------------------------

cohort_summary = (
    df.groupby("draft_year")[numeric_columns]
    .agg(["count", "median", "mean", "std"])
)

cohort_summary.to_csv(
    AUDIT_DIR / "step8b_feature_distributions_by_cohort.csv"
)

print("\nCOHORT FEATURE DISTRIBUTIONS")
display(cohort_summary.round(3))


# ---------------------------------------------------------------------
# 6. COHORT MEDIAN COMPARISON
# ---------------------------------------------------------------------

cohort_medians = (
    df.groupby("draft_year")[numeric_columns]
    .median()
    .T
)

cohort_medians["2025_vs_2023_pct_change"] = np.where(
    cohort_medians[2023].abs() > 1e-10,
    (
        (cohort_medians[2025] - cohort_medians[2023])
        / cohort_medians[2023].abs()
    ) * 100,
    np.nan
)

cohort_medians.to_csv(
    AUDIT_DIR / "step8b_cohort_median_comparison.csv"
)

print("\nCOHORT MEDIAN COMPARISON")
display(cohort_medians.round(3))


# ---------------------------------------------------------------------
# 7. OUTLIER AUDIT USING THE IQR RULE
# ---------------------------------------------------------------------

outlier_records = []

for feature in numeric_columns:

    values = df[feature].dropna()

    if len(values) < 5:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (
        (df[feature] < lower_bound) |
        (df[feature] > upper_bound)
    )

    outlier_records.append({
        "feature": feature,
        "observations": len(values),
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": int(mask.sum()),
        "outlier_pct": mask.mean() * 100
    })

outlier_audit = pd.DataFrame(outlier_records)

outlier_audit = outlier_audit.sort_values(
    "outlier_pct",
    ascending=False
)

outlier_audit.to_csv(
    AUDIT_DIR / "step8b_outlier_audit.csv",
    index=False
)

print("\nOUTLIER AUDIT — IQR RULE")
display(outlier_audit.round(3))


# ---------------------------------------------------------------------
# 8. DRILL COVERAGE AND TURNING QUALITY
# ---------------------------------------------------------------------

coverage_columns = [
    c for c in [
        "primary_drills_observed",
        "primary_attempts_observed",
        "turning_coverage_median_across_drills",
        "drills_with_turning_coverage_below_70_pct"
    ]
    if c in df.columns
]

if coverage_columns:

    print("\nDRILL AND TURNING COVERAGE")

    display(
        df[coverage_columns].describe().round(3)
    )

if "primary_drills_observed" in df.columns:

    drill_coverage = (
        df.groupby("draft_year")["primary_drills_observed"]
        .agg(["count", "median", "mean", "min", "max"])
    )

    drill_coverage.to_csv(
        AUDIT_DIR / "step8b_drill_coverage_by_cohort.csv"
    )

    print("\nPRIMARY DRILL COVERAGE BY COHORT")
    display(drill_coverage.round(3))


# ---------------------------------------------------------------------
# 9. TURNING MEASUREMENT COMPARISON BY COHORT
# ---------------------------------------------------------------------

turning_columns = [
    c for c in [
        "recorded_total_turn_median_across_drills",
        "coordinate_total_turn_median_across_drills",
        "coordinate_mean_turn_median_across_drills",
        "coordinate_p95_turn_median_across_drills",
        "turning_rate_median_across_drills",
        "turning_coverage_median_across_drills"
    ]
    if c in df.columns
]

if turning_columns:

    turning_by_cohort = (
        df.groupby("draft_year")[turning_columns]
        .median()
    )

    turning_by_cohort.to_csv(
        AUDIT_DIR / "step8b_turning_features_by_cohort.csv"
    )

    print("\nTURNING FEATURES BY COHORT")
    display(turning_by_cohort.round(3))


# ---------------------------------------------------------------------
# 10. FINAL VALIDATION
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 8B VALIDATION SUMMARY")
print("=" * 75)

print("Player rows:", len(df))
print("Unique players:", df["nfl_id"].nunique())
print("Duplicate player IDs:", df["nfl_id"].duplicated().sum())
print("Numeric predictors:", len(numeric_columns))
print("Features with missing values:",
      int(df[numeric_columns].isna().any().sum()))
print("Zero-variance features:",
      int((df[numeric_columns].nunique() <= 1).sum()))
print("Highly correlated pairs (>= 0.80):", len(high_corr))

print("\nAUDIT REPORTS SAVED:")

for file in sorted(AUDIT_DIR.glob("step8b_*.csv")):
    print(file)

print("\nSTEP 8B COMPLETED.")


STEP 8B — FINAL COMBINE FEATURE AUDIT
Dataset shape: (107, 46)
Unique players: 107
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]

Numeric feature count: 43

FEATURE DISTRIBUTION


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_count,missing_pct,unique_values,zero_variance
mean_speed_median_across_drills,107.0,5.635,0.369,4.769,4.834,5.011,5.398,5.647,5.885,6.211,6.484,6.575,0,0.000,107,False
peak_speed_median_across_drills,107.0,8.285,0.244,7.693,7.703,7.908,8.116,8.270,8.441,8.660,8.728,9.113,0,0.000,101,False
p95_speed_median_across_drills,107.0,8.171,0.228,7.527,7.625,7.818,8.039,8.168,8.316,8.532,8.671,8.980,0,0.000,107,False
mean_acceleration_median_across_drills,107.0,3.256,0.309,2.565,2.575,2.708,3.064,3.279,3.483,3.745,3.806,3.821,0,0.000,107,False
peak_acceleration_median_across_drills,107.0,6.647,0.484,5.128,5.216,5.795,6.401,6.648,6.996,7.337,7.518,7.532,0,0.000,107,False
coordinate_path_length_median_across_drills,107.0,36.228,1.768,30.597,31.842,32.980,35.856,36.188,36.651,38.708,42.002,44.390,0,0.000,107,False
net_displacement_median_across_drills,107.0,33.069,2.729,25.555,26.301,27.738,31.234,34.253,34.889,35.745,36.408,37.092,0,0.000,107,False
path_efficiency_median_across_drills,107.0,0.898,0.017,0.834,0.850,0.873,0.889,0.898,0.905,0.925,0.934,0.953,0,0.000,107,False
coordinate_mean_turn_median_across_drills,107.0,2.929,0.522,1.657,1.828,2.003,2.516,3.023,3.281,3.653,3.777,3.891,0,0.000,107,False
coordinate_p95_turn_median_across_drills,107.0,7.299,1.396,4.272,4.859,5.239,5.961,7.623,8.290,9.189,10.439,11.310,0,0.000,107,False



TOP 30 FEATURE CORRELATIONS


,feature_1,feature_2,spearman_correlation,absolute_correlation
484,turning_coverage_median_across_drills,median_turning_coverage,1.000,1.000
525,peak_speed_iqr_across_drills,p95_speed_iqr_across_drills,0.959,0.959
309,coordinate_mean_turn_median_across_drills,turning_rate_median_across_drills,0.949,0.949
42,peak_speed_median_across_drills,p95_speed_median_across_drills,0.949,0.949
694,coordinate_mean_turn_iqr_across_drills,turning_rate_iqr_across_drills,0.908,0.908
627,coordinate_path_length_iqr_across_drills,net_displacement_iqr_across_drills,0.832,0.832
308,coordinate_mean_turn_median_across_drills,coordinate_p95_turn_median_across_drills,0.816,0.816
750,recorded_total_turn_iqr_across_drills,coordinate_total_turn_iqr_across_drills,0.793,0.793
311,coordinate_mean_turn_median_across_drills,coordinate_total_turn_median_across_drills,0.779,0.779
342,coordinate_p95_turn_median_across_drills,turning_rate_median_across_drills,0.779,0.779



HIGHLY CORRELATED PAIRS: ABSOLUTE CORRELATION >= 0.80


,feature_1,feature_2,spearman_correlation,absolute_correlation
484,turning_coverage_median_across_drills,median_turning_coverage,1.000,1.000
525,peak_speed_iqr_across_drills,p95_speed_iqr_across_drills,0.959,0.959
309,coordinate_mean_turn_median_across_drills,turning_rate_median_across_drills,0.949,0.949
42,peak_speed_median_across_drills,p95_speed_median_across_drills,0.949,0.949
694,coordinate_mean_turn_iqr_across_drills,turning_rate_iqr_across_drills,0.908,0.908
627,coordinate_path_length_iqr_across_drills,net_displacement_iqr_across_drills,0.832,0.832
308,coordinate_mean_turn_median_across_drills,coordinate_p95_turn_median_across_drills,0.816,0.816


... (9 outputs left out)

,count,median,mean,min,max
draft_year,,,,,
2023,39,11.0,10.359,5,11
2024,31,11.0,10.484,5,11
2025,37,11.0,10.378,4,11



TURNING FEATURES BY COHORT


,recorded_total_turn_median_across_drills,coordinate_total_turn_median_across_drills,coordinate_mean_turn_median_across_drills,coordinate_p95_turn_median_across_drills,turning_rate_median_across_drills,turning_coverage_median_across_drills
draft_year,,,,,,
2023,119.011,173.342,3.182,8.130,28.839,0.950
2024,122.331,181.582,3.251,8.122,30.798,0.935
2025,111.927,115.382,2.361,5.594,22.859,0.953



STEP 8B VALIDATION SUMMARY
Player rows: 107
Unique players: 107
Duplicate player IDs: 0
Numeric predictors: 43
Features with missing values: 7
Zero-variance features: 0
Highly correlated pairs (>= 0.80): 7

AUDIT REPORTS SAVED:
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8b_cohort_median_comparison.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8b_drill_coverage_by_cohort.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8b_feature_correlations.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8b_feature_distribution.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8b_feature_distributions_by_cohort.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8b_outlier_audit.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8b_turning_features_by_cohort.csv

STEP 8B COMPLETED.


In [10]:

# =====================================================================
# NOTEBOOK 06 — STEP 8C: DRILL-LEVEL COHORT COMPARABILITY
# =====================================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

FEATURE_DIR = ROOT / "data" / "features"
AUDIT_DIR = ROOT / "reports" / "data_audit"

drill_path = (
    FEATURE_DIR /
    "wr_combine_final_player_drill_profiles.parquet"
)

drills = pd.read_parquet(drill_path)

print("=" * 75)
print("STEP 8C — DRILL-LEVEL COHORT COMPARABILITY")
print("=" * 75)

print("Dataset shape:", drills.shape)
print("Unique players:", drills["nfl_id"].nunique())

print("\nCOLUMNS")
for col in drills.columns:
    print(f"{col}: {drills[col].dtype}")

print("\nDRAFT COHORTS")
print(sorted(drills["draft_year"].dropna().unique()))

print("\nDRILL COUNTS BY COHORT")
display(
    drills.groupby(["draft_year", "drill_name"])
    .size()
    .rename("players")
    .reset_index()
    .pivot(
        index="drill_name",
        columns="draft_year",
        values="players"
    )
)

print("\nTURNING FEATURE CANDIDATES")

turning_candidates = [
    c for c in drills.columns
    if any(term in c.lower() for term in [
        "recorded_total_turn",
        "coordinate_total_turn",
        "coordinate_mean_turn",
        "coordinate_p95_turn",
        "turning_rate",
        "turning_coverage"
    ])
]

print(turning_candidates)

print("\nSTEP 8C SCHEMA CHECK COMPLETED.")
print("Share this output before running the drill-level comparison.")


STEP 8C — DRILL-LEVEL COHORT COMPARABILITY
Dataset shape: (1113, 18)
Unique players: 107

COLUMNS
draft_year: int64
nfl_id: int64
drill_name: object
mean_speed: float64
peak_speed: float64
p95_speed: float64
mean_acceleration: float64
peak_acceleration: float64
coordinate_path_length: float64
net_displacement: float64
path_efficiency: float64
coordinate_mean_turn: float64
coordinate_p95_turn: float64
turning_rate: float64
recorded_total_turn: float64
coordinate_total_turn: float64
turning_coverage: float64
attempts_in_drill: int64

DRAFT COHORTS
[np.int64(2023), np.int64(2024), np.int64(2025)]

DRILL COUNTS BY COHORT


draft_year,2023,2024,2025
drill_name,,,
COMEBACK_ROUTE_RIGHT,36,28,34
CURL_ROUTE_RIGHT,37,30,36
DAGGER_ROUTE_LEFT,39,31,37
GAUNTLET_DRILL,39,31,37
GO_ROUTE_RIGHT,35,29,34
OVER_SHOULDER_ADJUST,37,29,34
POST_CORNER_ROUTE_RIGHT,36,29,31
RED_ZONE_FADE_RIGHT,36,29,33
SLANT_ROUTE_LEFT,39,31,37



TURNING FEATURE CANDIDATES
['coordinate_mean_turn', 'coordinate_p95_turn', 'turning_rate', 'recorded_total_turn', 'coordinate_total_turn', 'turning_coverage']

STEP 8C SCHEMA CHECK COMPLETED.
Share this output before running the drill-level comparison.


In [11]:

# =====================================================================
# NOTEBOOK 06 — STEP 8C: DRILL-LEVEL COHORT COMPARABILITY AUDIT
# =====================================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

FEATURE_DIR = ROOT / "data" / "features"
AUDIT_DIR = ROOT / "reports" / "data_audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = (
    FEATURE_DIR /
    "wr_combine_final_player_drill_profiles.parquet"
)

drills = pd.read_parquet(INPUT_FILE)

# ---------------------------------------------------------------------
# 1. VALIDATE THE DRILL-LEVEL DATA
# ---------------------------------------------------------------------

KEYS = ["draft_year", "nfl_id", "drill_name"]

assert not drills.duplicated(KEYS).any(), (
    "Duplicate player-drill records detected."
)

assert drills["nfl_id"].nunique() == 107
assert set(drills["draft_year"].unique()) == {2023, 2024, 2025}

TURNING_FEATURES = [
    "recorded_total_turn",
    "coordinate_total_turn",
    "coordinate_mean_turn",
    "coordinate_p95_turn",
    "turning_rate",
    "turning_coverage"
]

OTHER_FEATURES = [
    "mean_speed",
    "peak_speed",
    "mean_acceleration",
    "path_efficiency"
]

FEATURES = TURNING_FEATURES + OTHER_FEATURES

missing_columns = [c for c in FEATURES if c not in drills.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("=" * 75)
print("STEP 8C — DRILL-LEVEL COHORT COMPARABILITY")
print("=" * 75)

print("Rows:", len(drills))
print("Players:", drills["nfl_id"].nunique())
print("Drills:", drills["drill_name"].nunique())


# ---------------------------------------------------------------------
# 2. COHORT SUMMARY FOR EACH DRILL
# ---------------------------------------------------------------------

summary = (
    drills.groupby(["drill_name", "draft_year"])[FEATURES]
    .agg(["count", "median", "mean", "std"])
)

summary.to_csv(
    AUDIT_DIR / "step8c_drill_cohort_summary.csv"
)

print("\nDRILL-LEVEL COHORT SUMMARY")
display(summary.round(3))


# ---------------------------------------------------------------------
# 3. 2025 VERSUS 2023 MEDIAN COMPARISON
# ---------------------------------------------------------------------

comparison_records = []

for drill_name, group in drills.groupby("drill_name"):

    row = {"drill_name": drill_name}

    for feature in FEATURES:

        medians = group.groupby("draft_year")[feature].median()
        counts = group.groupby("draft_year")[feature].count()

        median_2023 = medians.get(2023, np.nan)
        median_2024 = medians.get(2024, np.nan)
        median_2025 = medians.get(2025, np.nan)

        row[f"{feature}_n_2023"] = counts.get(2023, 0)
        row[f"{feature}_n_2024"] = counts.get(2024, 0)
        row[f"{feature}_n_2025"] = counts.get(2025, 0)

        row[f"{feature}_median_2023"] = median_2023
        row[f"{feature}_median_2024"] = median_2024
        row[f"{feature}_median_2025"] = median_2025

        if pd.notna(median_2023) and abs(median_2023) > 1e-10:
            row[f"{feature}_2025_vs_2023_pct"] = (
                (median_2025 - median_2023)
                / abs(median_2023)
            ) * 100
        else:
            row[f"{feature}_2025_vs_2023_pct"] = np.nan

    comparison_records.append(row)

drill_comparison = pd.DataFrame(comparison_records)

drill_comparison.to_csv(
    AUDIT_DIR / "step8c_drill_2025_vs_2023_comparison.csv",
    index=False
)

print("\n2025 VERSUS 2023 — TURNING COMPARISON")

turning_comparison_columns = [
    "drill_name",
    "recorded_total_turn_median_2023",
    "recorded_total_turn_median_2025",
    "recorded_total_turn_2025_vs_2023_pct",
    "coordinate_total_turn_median_2023",
    "coordinate_total_turn_median_2025",
    "coordinate_total_turn_2025_vs_2023_pct",
    "coordinate_mean_turn_2025_vs_2023_pct",
    "coordinate_p95_turn_2025_vs_2023_pct",
    "turning_rate_2025_vs_2023_pct"
]

display(
    drill_comparison[turning_comparison_columns]
    .sort_values("coordinate_total_turn_2025_vs_2023_pct")
    .round(2)
)


# ---------------------------------------------------------------------
# 4. IDENTIFY DRILLS WITH LARGE COORDINATE TURNING CHANGES
# ---------------------------------------------------------------------

coordinate_change = drill_comparison[
    [
        "drill_name",
        "coordinate_total_turn_2025_vs_2023_pct",
        "coordinate_mean_turn_2025_vs_2023_pct",
        "coordinate_p95_turn_2025_vs_2023_pct",
        "recorded_total_turn_2025_vs_2023_pct"
    ]
].copy()

coordinate_change["large_coordinate_decline"] = (
    coordinate_change["coordinate_total_turn_2025_vs_2023_pct"] <= -20
)

coordinate_change["recorded_coordinate_disagreement"] = (
    (
        coordinate_change["coordinate_total_turn_2025_vs_2023_pct"]
        -
        coordinate_change["recorded_total_turn_2025_vs_2023_pct"]
    ).abs() >= 15
)

coordinate_change.to_csv(
    AUDIT_DIR / "step8c_turning_change_flags.csv",
    index=False
)

print("\nDRILLS WITH COORDINATE TURNING DECLINE OF 20% OR MORE")
display(
    coordinate_change[
        coordinate_change["large_coordinate_decline"]
    ].round(2)
)

print("\nDRILLS WITH RECORDED-COORDINATE CHANGE DISAGREEMENT")
display(
    coordinate_change[
        coordinate_change["recorded_coordinate_disagreement"]
    ].round(2)
)


# ---------------------------------------------------------------------
# 5. WITHIN-DRILL RECORDED VS COORDINATE TURNING CORRELATION
# ---------------------------------------------------------------------

correlation_records = []

for (draft_year, drill_name), group in drills.groupby(
    ["draft_year", "drill_name"]
):

    valid = group[
        ["recorded_total_turn", "coordinate_total_turn"]
    ].dropna()

    if len(valid) >= 5:
        corr = valid["recorded_total_turn"].corr(
            valid["coordinate_total_turn"],
            method="spearman"
        )
    else:
        corr = np.nan

    correlation_records.append({
        "draft_year": draft_year,
        "drill_name": drill_name,
        "observations": len(valid),
        "spearman_recorded_vs_coordinate": corr
    })

turning_correlations = pd.DataFrame(correlation_records)

turning_correlations.to_csv(
    AUDIT_DIR / "step8c_recorded_coordinate_correlations.csv",
    index=False
)

print("\nRECORDED VS COORDINATE TURNING CORRELATIONS")
display(
    turning_correlations.pivot(
        index="drill_name",
        columns="draft_year",
        values="spearman_recorded_vs_coordinate"
    ).round(3)
)


# ---------------------------------------------------------------------
# 6. CHECK WHETHER SPEED OR COVERAGE ALSO SHIFTED
# ---------------------------------------------------------------------

control_features = [
    "mean_speed",
    "peak_speed",
    "mean_acceleration",
    "path_efficiency",
    "turning_coverage"
]

control_records = []

for drill_name, group in drills.groupby("drill_name"):

    row = {"drill_name": drill_name}

    for feature in control_features:

        medians = group.groupby("draft_year")[feature].median()

        m23 = medians.get(2023, np.nan)
        m25 = medians.get(2025, np.nan)

        row[f"{feature}_median_2023"] = m23
        row[f"{feature}_median_2025"] = m25

        row[f"{feature}_2025_vs_2023_pct"] = (
            ((m25 - m23) / abs(m23)) * 100
            if pd.notna(m23) and abs(m23) > 1e-10
            else np.nan
        )

    control_records.append(row)

control_comparison = pd.DataFrame(control_records)

control_comparison.to_csv(
    AUDIT_DIR / "step8c_speed_coverage_controls.csv",
    index=False
)

print("\nSPEED, ACCELERATION, PATH EFFICIENCY AND COVERAGE COMPARISON")
display(control_comparison.round(2))


# ---------------------------------------------------------------------
# 7. FINAL AUDIT SUMMARY
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 8C — AUDIT SUMMARY")
print("=" * 75)

print("Drills analysed:", drills["drill_name"].nunique())

print(
    "Drills with coordinate total-turn decline >=20%:",
    int(coordinate_change["large_coordinate_decline"].sum())
)

print(
    "Drills with recorded-coordinate disagreement >=15 percentage points:",
    int(coordinate_change["recorded_coordinate_disagreement"].sum())
)

print(
    "Mean recorded-coordinate Spearman correlation:",
    round(
        turning_correlations[
            "spearman_recorded_vs_coordinate"
        ].mean(),
        3
    )
)

print("\nSAVED REPORTS")

for file in sorted(AUDIT_DIR.glob("step8c_*.csv")):
    print(file)

print("\nSTEP 8C COMPLETED.")


STEP 8C — DRILL-LEVEL COHORT COMPARABILITY
Rows: 1113
Players: 107
Drills: 11

DRILL-LEVEL COHORT SUMMARY


recorded_total_turn                    \
                                                 count   median     mean   
drill_name              draft_year                                         
COMEBACK_ROUTE_RIGHT    2023                        36  136.000  141.994   
                        2024                        28  138.936  150.648   
                        2025                        34  176.474  175.122   
CURL_ROUTE_RIGHT        2023                        37  235.506  244.313   
                        2024                        30  271.177  265.955   
                        2025                        36  253.772  252.654   
DAGGER_ROUTE_LEFT       2023                        39  146.987  149.838   
                        2024                        31  146.039  144.073   
                        2025                        37  140.360  143.093   
GAUNTLET_DRILL          2023                        39  113.793  113.942   
                        2024                        31  118.967  117.168   
                        2025                        37  111.927  114.370   
GO_ROUTE_RIGHT          2023                        35   38.552   39.077   
                        2024                        29   31.556   34.893   
                        2025                        34   31.228   32.958   
OVER_SHOULDER_ADJUST    2023                        37   23.799   27.255   
                        2024                        29   24.906   28.636   
                        2025                        34   32.343   36.107   
POST_CORNER_ROUTE_RIGHT 2023                        36  107.503  108.529   
                        2024                        29  128.196  131.520   
                        2025                        31   80.125   81.323   
RED_ZONE_FADE_RIGHT     2023                        36   60.502   57.694   
                        2024                        29   58.011   64.309   
                        2025                        33   51.600   49.413   
SLANT_ROUTE_LEFT        2023                        39   91.012   92.389   
                        2024                        31   89.021   90.286   
                        2025                        37   63.757   63.943   
SLOT_SAIL_ROUTE_RIGHT   2023                        33  195.506  181.636   
                        2024                        30  176.070  166.761   
                        2025                        35  186.606  175.107   
SLOT_STRIKE_ROUTE_LEFT  2023                        37  177.335  200.270   
                        2024                        28  171.457  192.526   
                        2025                        36  160.125  178.078   

                                           coordinate_total_turn           \
                                       std                 count   median   
drill_name              draft_year                                          
COMEBACK_ROUTE_RIGHT    2023        19.573                    36  193.304   
                        2024        37.965                    28  192.197   
                        2025        40.439                    34  181.319   
CURL_ROUTE_RIGHT        2023        71.011                    37  292.453   
                        2024        34.568                    30  316.232   
                        2025        33.932                    36  260.342   
DAGGER_ROUTE_LEFT       2023        27.548                    39  189.332   
                        2024        19.151                    31  196.425   
                        2025        20.392                    37  145.680   
GAUNTLET_DRILL          2023         7.925                    39  192.811   
                        2024        12.095                    31  183.924   
                        2025         8.245                    37  115.636   
GO_ROUTE_RIGHT          2023        16.460                    35  104.864   
                        2024        11.859                    29  


2025 VERSUS 2023 — TURNING COMPARISON


,drill_name,recorded_total_turn_median_2023,recorded_total_turn_median_2025,recorded_total_turn_2025_vs_2023_pct,coordinate_total_turn_median_2023,coordinate_total_turn_median_2025,coordinate_total_turn_2025_vs_2023_pct,coordinate_mean_turn_2025_vs_2023_pct,coordinate_p95_turn_2025_vs_2023_pct,turning_rate_2025_vs_2023_pct
5,OVER_SHOULDER_ADJUST,23.80,32.34,35.90,120.74,38.01,-68.52,-69.53,-71.46,-71.70
4,GO_ROUTE_RIGHT,38.55,31.23,-19.00,104.86,37.27,-64.46,-64.48,-65.96,-64.07
7,RED_ZONE_FADE_RIGHT,60.50,51.60,-14.71,130.83,54.91,-58.03,-53.43,-61.43,-47.77
6,POST_CORNER_ROUTE_RIGHT,107.50,80.13,-25.47,152.74,87.25,-42.87,-49.31,-54.28,-48.55
8,SLANT_ROUTE_LEFT,91.01,63.76,-29.95,115.82,67.55,-41.68,-12.93,-40.54,-1.58
3,GAUNTLET_DRILL,113.79,111.93,-1.64,192.81,115.64,-40.03,-39.55,-23.80,-42.12
10,SLOT_STRIKE_ROUTE_LEFT,177.33,160.12,-9.71,233.56,162.34,-30.49,-21.60,-43.87,-20.23
2,DAGGER_ROUTE_LEFT,146.99,140.36,-4.51,189.33,145.68,-23.06,-22.11,-28.16,-21.60
9,SLOT_SAIL_ROUTE_RIGHT,195.51,186.61,-4.55,227.67,189.42,-16.80,-15.22,-21.83,-8.77
1,CURL_ROUTE_RIGHT,235.51,253.77,7.76,292.45,260.34,-10.98,-13.00,-6.41,-12.01



DRILLS WITH COORDINATE TURNING DECLINE OF 20% OR MORE


,drill_name,coordinate_total_turn_2025_vs_2023_pct,coordinate_mean_turn_2025_vs_2023_pct,coordinate_p95_turn_2025_vs_2023_pct,recorded_total_turn_2025_vs_2023_pct,large_coordinate_decline,recorded_coordinate_disagreement
2,DAGGER_ROUTE_LEFT,-23.06,-22.11,-28.16,-4.51,True,True
3,GAUNTLET_DRILL,-40.03,-39.55,-23.80,-1.64,True,True
4,GO_ROUTE_RIGHT,-64.46,-64.48,-65.96,-19.00,True,True
5,OVER_SHOULDER_ADJUST,-68.52,-69.53,-71.46,35.90,True,True
6,POST_CORNER_ROUTE_RIGHT,-42.87,-49.31,-54.28,-25.47,True,True
7,RED_ZONE_FADE_RIGHT,-58.03,-53.43,-61.43,-14.71,True,True
8,SLANT_ROUTE_LEFT,-41.68,-12.93,-40.54,-29.95,True,False
10,SLOT_STRIKE_ROUTE_LEFT,-30.49,-21.60,-43.87,-9.71,True,True


... (3 outputs left out)

draft_year,2023,2024,2025
drill_name,,,
COMEBACK_ROUTE_RIGHT,0.514,0.822,0.968
CURL_ROUTE_RIGHT,0.932,0.896,0.989
DAGGER_ROUTE_LEFT,0.827,0.617,0.981
GAUNTLET_DRILL,0.249,0.317,0.986
GO_ROUTE_RIGHT,0.638,0.438,0.842
OVER_SHOULDER_ADJUST,-0.063,0.376,0.986
POST_CORNER_ROUTE_RIGHT,0.847,0.755,0.931
RED_ZONE_FADE_RIGHT,0.294,0.643,0.972
SLANT_ROUTE_LEFT,0.724,0.681,0.964



SPEED, ACCELERATION, PATH EFFICIENCY AND COVERAGE COMPARISON


,drill_name,mean_speed_median_2023,mean_speed_median_2025,mean_speed_2025_vs_2023_pct,peak_speed_median_2023,peak_speed_median_2025,peak_speed_2025_vs_2023_pct,mean_acceleration_median_2023,mean_acceleration_median_2025,mean_acceleration_2025_vs_2023_pct,path_efficiency_median_2023,path_efficiency_median_2025,path_efficiency_2025_vs_2023_pct,turning_coverage_median_2023,turning_coverage_median_2025,turning_coverage_2025_vs_2023_pct
0,COMEBACK_ROUTE_RIGHT,5.05,4.53,-10.28,8.23,7.97,-3.22,3.98,3.96,-0.46,0.68,0.79,16.46,0.94,0.91,-3.99
1,CURL_ROUTE_RIGHT,4.03,4.20,4.41,7.06,7.17,1.50,4.42,4.47,1.10,0.77,0.76,-1.38,0.96,0.96,-0.07
2,DAGGER_ROUTE_LEFT,6.29,6.54,3.96,8.17,8.48,3.75,3.80,4.02,5.72,0.89,0.88,-1.01,0.97,0.97,0.06
3,GAUNTLET_DRILL,6.80,6.73,-0.93,8.77,8.92,1.68,2.38,2.44,2.41,0.89,0.90,0.46,0.95,0.92,-3.47
4,GO_ROUTE_RIGHT,7.77,7.87,1.37,10.29,10.12,-1.65,2.01,1.96,-2.32,0.99,1.00,0.16,0.96,0.97,1.49
5,OVER_SHOULDER_ADJUST,5.86,5.40,-7.90,7.60,7.40,-2.67,1.44,1.61,12.05,1.00,0.99,-0.27,0.98,0.90,-8.22
6,POST_CORNER_ROUTE_RIGHT,7.07,7.68,8.65,9.25,9.84,6.39,3.01,2.81,-6.55,0.92,0.93,0.68,0.97,0.97,0.40
7,RED_ZONE_FADE_RIGHT,3.91,4.63,18.46,6.44,6.53,1.42,2.08,2.22,7.13,0.99,0.99,0.41,0.89,0.97,9.80
8,SLANT_ROUTE_LEFT,5.23,4.95,-5.40,8.59,7.63,-11.21,3.14,3.49,11.19,0.96,0.94,-2.15,0.84,0.93,10.42
9,SLOT_SAIL_ROUTE_RIGHT,6.08,6.51,7.01,8.32,8.55,2.79,4.01,4.17,4.08,0.76,0.78,2.70,0.89,0.95,7.58



STEP 8C — AUDIT SUMMARY
Drills analysed: 11
Drills with coordinate total-turn decline >=20%: 8
Drills with recorded-coordinate disagreement >=15 percentage points: 9
Mean recorded-coordinate Spearman correlation: 0.738

SAVED REPORTS
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8c_drill_2025_vs_2023_comparison.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8c_drill_cohort_summary.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8c_recorded_coordinate_correlations.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8c_speed_coverage_controls.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8c_turning_change_flags.csv

STEP 8C COMPLETED.


In [12]:

# =====================================================================
# NOTEBOOK 06 — STEP 8D
# RAW TRACKING RECONSTRUCTION OF RECORDED VS COORDINATE TURNING
# =====================================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

RAW_DIR = ROOT / "data" / "raw" / "nfl-big-data-bowl-2027"
FEATURE_DIR = ROOT / "data" / "features"
AUDIT_DIR = ROOT / "reports" / "data_audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

TRACKING_FILE = RAW_DIR / "combine_tracking.csv"
RESULTS_FILE = RAW_DIR / "combine_results.csv"

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

GROUP_KEYS = [
    "draft_year",
    "nfl_id",
    "drill_type",
    "drill_name",
    "attempt",
    "event_id"
]

# ---------------------------------------------------------------------
# 1. LOAD ONLY WR PLAYER OBSERVATIONS IN THE PRIMARY DRILLS
# ---------------------------------------------------------------------

wr = pd.read_csv(RESULTS_FILE)

wr_ids = set(
    wr.loc[
        wr["combine_position"].astype(str).str.upper().eq("WR"),
        "nfl_id"
    ].dropna().astype(int)
)

tracking = pd.read_csv(TRACKING_FILE)

required = GROUP_KEYS + [
    "entity_type", "time", "x", "y", "s", "dir"
]

missing = [c for c in required if c not in tracking.columns]

if missing:
    raise ValueError(f"Missing raw tracking columns: {missing}")

tracking["nfl_id"] = pd.to_numeric(
    tracking["nfl_id"], errors="coerce"
)

tracking = tracking[
    tracking["entity_type"].eq("PLAYER")
    & tracking["nfl_id"].isin(wr_ids)
    & tracking["drill_name"].isin(PRIMARY_DRILLS)
].copy()

for col in ["x", "y", "s", "dir", "draft_year", "attempt"]:
    tracking[col] = pd.to_numeric(
        tracking[col], errors="coerce"
    )

tracking["time"] = pd.to_datetime(
    tracking["time"], errors="coerce"
)

tracking = tracking.dropna(
    subset=GROUP_KEYS + ["time", "x", "y", "s", "dir"]
)

tracking = tracking.sort_values(
    GROUP_KEYS + ["time"]
).reset_index(drop=True)

print("Raw WR primary-drill observations:", len(tracking))
print("Players:", tracking["nfl_id"].nunique())
print("Attempts:", tracking.groupby(GROUP_KEYS).ngroups)
print("Cohorts:", sorted(tracking["draft_year"].unique()))


# ---------------------------------------------------------------------
# 2. ANGULAR DIFFERENCE FUNCTION
# ---------------------------------------------------------------------

def signed_angle_difference(angle_a, angle_b):
    """
    Smallest signed angular difference, in degrees,
    constrained to [-180, 180).
    """
    return (angle_a - angle_b + 180) % 360 - 180


# ---------------------------------------------------------------------
# 3. RECONSTRUCT BOTH TURNING MEASURES PER ATTEMPT
# ---------------------------------------------------------------------

attempt_records = []

for key, group in tracking.groupby(GROUP_KEYS, sort=False):

    group = group.sort_values("time").copy()

    x = group["x"].to_numpy(dtype=float)
    y = group["y"].to_numpy(dtype=float)
    speed = group["s"].to_numpy(dtype=float)
    recorded_dir = group["dir"].to_numpy(dtype=float)

    # Coordinate-derived heading:
    # clockwise from positive Y axis.
    dx = np.diff(x, prepend=np.nan)
    dy = np.diff(y, prepend=np.nan)

    coordinate_heading = (
        np.degrees(np.arctan2(dx, dy)) % 360
    )

    # Compare direction estimates at the same observation.
    heading_error = np.abs(
        signed_angle_difference(
            coordinate_heading,
            recorded_dir
        )
    )

    # Heading change between consecutive observations.
    recorded_turn = np.abs(
        signed_angle_difference(
            recorded_dir[1:],
            recorded_dir[:-1]
        )
    )

    coordinate_turn = np.abs(
        signed_angle_difference(
            coordinate_heading[1:],
            coordinate_heading[:-1]
        )
    )

    # Use the same speed threshold and same transition pairs.
    pair_speed = np.minimum(speed[1:], speed[:-1])
    valid_pair = (
        np.isfinite(recorded_turn)
        & np.isfinite(coordinate_turn)
        & (pair_speed >= 1.0)
    )

    rec = recorded_turn[valid_pair]
    coord = coordinate_turn[valid_pair]

    # Direction agreement only at moving observations.
    moving = speed >= 1.0
    moving_errors = heading_error[moving]
    moving_errors = moving_errors[np.isfinite(moving_errors)]

    record = dict(zip(GROUP_KEYS, key))

    record.update({
        "raw_observations": len(group),
        "valid_pairs_speed_ge_1": int(valid_pair.sum()),

        "recorded_turn_sum_t1": float(rec.sum()) if len(rec) else np.nan,
        "coordinate_turn_sum_t1": float(coord.sum()) if len(coord) else np.nan,

        "recorded_turn_mean_t1": float(rec.mean()) if len(rec) else np.nan,
        "coordinate_turn_mean_t1": float(coord.mean()) if len(coord) else np.nan,

        "median_pair_turn_difference": (
            float(np.median(np.abs(rec - coord)))
            if len(rec) else np.nan
        ),

        "p95_pair_turn_difference": (
            float(np.percentile(np.abs(rec - coord), 95))
            if len(rec) else np.nan
        ),

        "median_heading_error_speed_ge_1": (
            float(np.median(moving_errors))
            if len(moving_errors) else np.nan
        ),

        "p95_heading_error_speed_ge_1": (
            float(np.percentile(moving_errors, 95))
            if len(moving_errors) else np.nan
        ),

        "mean_heading_error_speed_ge_1": (
            float(np.mean(moving_errors))
            if len(moving_errors) else np.nan
        )
    })

    attempt_records.append(record)

raw_attempt = pd.DataFrame(attempt_records)

raw_attempt.to_parquet(
    FEATURE_DIR / "wr_raw_turning_reconstruction_step8d.parquet",
    index=False
)

raw_attempt.to_csv(
    FEATURE_DIR / "wr_raw_turning_reconstruction_step8d.csv",
    index=False
)

print("\nRECONSTRUCTED ATTEMPT DATASET")
print("Shape:", raw_attempt.shape)
print("Unique players:", raw_attempt["nfl_id"].nunique())
print("Unique attempts:", raw_attempt.groupby(GROUP_KEYS).ngroups)


# ---------------------------------------------------------------------
# 4. COMPARE COHORTS WITHIN EACH DRILL
# ---------------------------------------------------------------------

metrics = [
    "recorded_turn_sum_t1",
    "coordinate_turn_sum_t1",
    "recorded_turn_mean_t1",
    "coordinate_turn_mean_t1",
    "median_pair_turn_difference",
    "median_heading_error_speed_ge_1",
    "p95_heading_error_speed_ge_1"
]

cohort_summary = (
    raw_attempt.groupby(["drill_name", "draft_year"])[metrics]
    .agg(["count", "median", "mean", "std"])
)

cohort_summary.to_csv(
    AUDIT_DIR / "step8d_raw_turning_cohort_summary.csv"
)

print("\nRAW RECONSTRUCTION — COHORT SUMMARY")
display(cohort_summary.round(3))


# ---------------------------------------------------------------------
# 5. 2025 VS 2023 RAW ATTEMPT-LEVEL MEDIAN COMPARISON
# ---------------------------------------------------------------------

comparison = []

for drill_name, group in raw_attempt.groupby("drill_name"):

    row = {"drill_name": drill_name}

    for metric in metrics:

        medians = group.groupby("draft_year")[metric].median()

        m23 = medians.get(2023, np.nan)
        m24 = medians.get(2024, np.nan)
        m25 = medians.get(2025, np.nan)

        row[f"{metric}_2023"] = m23
        row[f"{metric}_2024"] = m24
        row[f"{metric}_2025"] = m25

        row[f"{metric}_change_2025_vs_2023_pct"] = (
            ((m25 - m23) / abs(m23)) * 100
            if pd.notna(m23) and abs(m23) > 1e-10
            else np.nan
        )

    comparison.append(row)

raw_comparison = pd.DataFrame(comparison)

raw_comparison.to_csv(
    AUDIT_DIR / "step8d_raw_turning_2025_vs_2023.csv",
    index=False
)

print("\nRAW TURNING COMPARISON — 2025 VS 2023")

display(
    raw_comparison[
        [
            "drill_name",
            "recorded_turn_sum_t1_2023",
            "recorded_turn_sum_t1_2025",
            "recorded_turn_sum_t1_change_2025_vs_2023_pct",
            "coordinate_turn_sum_t1_2023",
            "coordinate_turn_sum_t1_2025",
            "coordinate_turn_sum_t1_change_2025_vs_2023_pct",
            "median_heading_error_speed_ge_1_2023",
            "median_heading_error_speed_ge_1_2025",
            "median_pair_turn_difference_2023",
            "median_pair_turn_difference_2025"
        ]
    ].round(2)
)


# ---------------------------------------------------------------------
# 6. RECONSTRUCTION AGREEMENT BY COHORT
# ---------------------------------------------------------------------

agreement_records = []

for (year, drill), group in raw_attempt.groupby(
    ["draft_year", "drill_name"]
):

    valid = group[
        [
            "recorded_turn_sum_t1",
            "coordinate_turn_sum_t1"
        ]
    ].dropna()

    agreement_records.append({
        "draft_year": year,
        "drill_name": drill,
        "attempts": len(valid),
        "spearman_recorded_vs_coordinate": (
            valid["recorded_turn_sum_t1"].corr(
                valid["coordinate_turn_sum_t1"],
                method="spearman"
            ) if len(valid) >= 5 else np.nan
        ),
        "pearson_recorded_vs_coordinate": (
            valid["recorded_turn_sum_t1"].corr(
                valid["coordinate_turn_sum_t1"],
                method="pearson"
            ) if len(valid) >= 5 else np.nan
        )
    })

agreement = pd.DataFrame(agreement_records)

agreement.to_csv(
    AUDIT_DIR / "step8d_raw_turning_agreement.csv",
    index=False
)

print("\nRAW RECONSTRUCTION AGREEMENT")
display(
    agreement.pivot(
        index="drill_name",
        columns="draft_year",
        values="spearman_recorded_vs_coordinate"
    ).round(3)
)


# ---------------------------------------------------------------------
# 7. QUALITY CHECKS
# ---------------------------------------------------------------------

print("\nQUALITY CHECKS")

print(
    "Attempts with zero valid pairs:",
    int((raw_attempt["valid_pairs_speed_ge_1"] == 0).sum())
)

print(
    "Attempts with fewer than 10 valid pairs:",
    int((raw_attempt["valid_pairs_speed_ge_1"] < 10).sum())
)

print(
    "Missing reconstructed recorded turn:",
    int(raw_attempt["recorded_turn_sum_t1"].isna().sum())
)

print(
    "Missing reconstructed coordinate turn:",
    int(raw_attempt["coordinate_turn_sum_t1"].isna().sum())
)

print(
    "Median valid pairs per attempt:",
    raw_attempt["valid_pairs_speed_ge_1"].median()
)

print("\nSAVED OUTPUTS")

for file in sorted(AUDIT_DIR.glob("step8d_*.csv")):
    print(file)

print("\nSTEP 8D COMPLETED.")


Raw WR primary-drill observations: 84064
Players: 107
Attempts: 1282
Cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]

RECONSTRUCTED ATTEMPT DATASET
Shape: (1282, 17)
Unique players: 107
Unique attempts: 1282

RAW RECONSTRUCTION — COHORT SUMMARY


recorded_turn_sum_t1                    \
                                                  count   median     mean   
drill_name              draft_year                                          
COMEBACK_ROUTE_RIGHT    2023                         37  137.257  144.353   
                        2024                         33  139.662  149.674   
                        2025                         40  181.117  177.028   
CURL_ROUTE_RIGHT        2023                         43  242.854  235.894   
                        2024                         31  272.328  268.603   
                        2025                         36  253.887  254.140   
DAGGER_ROUTE_LEFT       2023                         43  148.428  151.477   
                        2024                         32  145.805  144.438   
                        2025                         40  140.703  143.330   
GAUNTLET_DRILL          2023                         71  115.119  115.764   
                        2024                         58  120.615  119.143   
                        2025                         71  113.070  115.669   
GO_ROUTE_RIGHT          2023                         41   39.575   39.894   
                        2024                         30   32.637   36.455   
                        2025                         34   31.563   33.993   
OVER_SHOULDER_ADJUST    2023                         38   25.222   28.158   
                        2024                         29   27.198   30.118   
                        2025                         34   33.665   37.442   
POST_CORNER_ROUTE_RIGHT 2023                         38  109.062  109.507   
                        2024                         33  128.978  132.240   
                        2025                         40   80.279   82.319   
RED_ZONE_FADE_RIGHT     2023                         43   63.146   60.673   
                        2024                         36   62.720   66.086   
                        2025                         33   57.585   54.004   
SLANT_ROUTE_LEFT        2023                         39   93.170   93.922   
                        2024                         34   90.648   92.332   
                        2025                         37   64.754   65.481   
SLOT_SAIL_ROUTE_RIGHT   2023                         38  182.008  177.417   
                        2024                         33  176.603  166.142   
                        2025                         35  187.123  176.502   
SLOT_STRIKE_ROUTE_LEFT  2023                         37  197.751  213.010   
                        2024                         28  193.031  202.994   
                        2025                         37  178.645  191.947   

                                           coordinate_turn_sum_t1           \
                                       std                  count   median   
drill_name              draft_year                                           
COMEBACK_ROUTE_RIGHT    2023        19.425                     37  200.617   
                        2024        36.192                     33  201.320   
                        2025        47.183                     40  187.424   
CURL_ROUTE_RIGHT        2023        84.399                     43  303.750   
                        2024        35.624                     31  319.879   
                        2025        33.479                     36  261.753   
DAGGER_ROUTE_LEFT       2023        31.320                     43  196.918   
                        2024        20.499                     32  199.789   
                        2025        25.964                     40  147.667   
GAUNTLET_DRILL          2023         9.970                     71  199.298   
                        2024        13.517                     58  194.369   
                        2025        10.699                     71  117.002   
GO_ROUTE_RIGHT          2023        17.397                     41  117.211   
              


RAW TURNING COMPARISON — 2025 VS 2023


,drill_name,recorded_turn_sum_t1_2023,recorded_turn_sum_t1_2025,recorded_turn_sum_t1_change_2025_vs_2023_pct,coordinate_turn_sum_t1_2023,coordinate_turn_sum_t1_2025,coordinate_turn_sum_t1_change_2025_vs_2023_pct,median_heading_error_speed_ge_1_2023,median_heading_error_speed_ge_1_2025,median_pair_turn_difference_2023,median_pair_turn_difference_2025
0,COMEBACK_ROUTE_RIGHT,137.26,181.12,31.95,200.62,187.42,-6.58,1.34,0.58,1.66,0.25
1,CURL_ROUTE_RIGHT,242.85,253.89,4.54,303.75,261.75,-13.83,1.88,1.25,2.08,0.40
2,DAGGER_ROUTE_LEFT,148.43,140.70,-5.20,196.92,147.67,-25.01,1.44,1.12,1.32,0.19
3,GAUNTLET_DRILL,115.12,113.07,-1.78,199.30,117.00,-41.29,0.96,0.20,1.43,0.11
4,GO_ROUTE_RIGHT,39.58,31.56,-20.25,117.21,40.87,-65.13,0.76,0.18,1.11,0.09
5,OVER_SHOULDER_ADJUST,25.22,33.66,33.48,130.47,40.06,-69.29,1.00,0.21,1.37,0.09
6,POST_CORNER_ROUTE_RIGHT,109.06,80.28,-26.39,159.20,88.47,-44.43,1.05,0.57,1.18,0.10
7,RED_ZONE_FADE_RIGHT,63.15,57.59,-8.81,140.86,62.14,-55.88,1.35,0.42,1.74,0.22
8,SLANT_ROUTE_LEFT,93.17,64.75,-30.50,123.66,70.36,-43.10,1.25,1.54,1.47,0.27
9,SLOT_SAIL_ROUTE_RIGHT,182.01,187.12,2.81,229.18,190.99,-16.66,1.46,1.22,1.39,0.22



RAW RECONSTRUCTION AGREEMENT


draft_year,2023,2024,2025
drill_name,,,
COMEBACK_ROUTE_RIGHT,0.354,0.782,0.967
CURL_ROUTE_RIGHT,0.934,0.907,0.978
DAGGER_ROUTE_LEFT,0.804,0.596,0.979
GAUNTLET_DRILL,0.198,0.212,0.977
GO_ROUTE_RIGHT,0.655,0.498,0.821
OVER_SHOULDER_ADJUST,-0.051,0.395,0.980
POST_CORNER_ROUTE_RIGHT,0.766,0.715,0.931
RED_ZONE_FADE_RIGHT,0.204,0.445,0.968
SLANT_ROUTE_LEFT,0.666,0.648,0.930



QUALITY CHECKS
Attempts with zero valid pairs: 0
Attempts with fewer than 10 valid pairs: 0
Missing reconstructed recorded turn: 0
Missing reconstructed coordinate turn: 0
Median valid pairs per attempt: 60.0

SAVED OUTPUTS
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8d_raw_turning_2025_vs_2023.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8d_raw_turning_agreement.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8d_raw_turning_cohort_summary.csv

STEP 8D COMPLETED.


In [14]:

# =====================================================================
# NOTEBOOK 06 — STEP 8E (CORRECTED)
# PLAYER-WEIGHTED TURNING + SPEED THRESHOLD SENSITIVITY
# =====================================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

RAW_DIR = ROOT / "data/raw/nfl-big-data-bowl-2027"
FEATURE_DIR = ROOT / "data/features"
AUDIT_DIR = ROOT / "reports/data_audit"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

GROUP_KEYS = [
    "draft_year", "nfl_id", "drill_type",
    "drill_name", "attempt", "event_id"
]

THRESHOLDS = [0.5, 1.0, 2.0, 3.0]


# ---------------------------------------------------------------------
# 1. LOAD RAW TRACKING
# ---------------------------------------------------------------------

wr = pd.read_csv(RAW_DIR / "combine_results.csv")

wr_ids = set(
    wr.loc[
        wr["combine_position"].astype(str).str.upper().eq("WR"),
        "nfl_id"
    ].dropna().astype(int)
)

tracking = pd.read_csv(RAW_DIR / "combine_tracking.csv")

tracking["nfl_id"] = pd.to_numeric(
    tracking["nfl_id"], errors="coerce"
)

for col in ["x", "y", "s", "dir", "draft_year", "attempt"]:
    tracking[col] = pd.to_numeric(
        tracking[col], errors="coerce"
    )

tracking["time"] = pd.to_datetime(
    tracking["time"], errors="coerce"
)

tracking = tracking[
    tracking["entity_type"].eq("PLAYER")
    & tracking["nfl_id"].isin(wr_ids)
    & tracking["drill_name"].isin(PRIMARY_DRILLS)
].copy()

tracking = tracking.dropna(
    subset=GROUP_KEYS + ["time", "x", "y", "s", "dir"]
)

tracking = tracking.sort_values(
    GROUP_KEYS + ["time"]
).reset_index(drop=True)

print("Filtered raw observations:", len(tracking))


# ---------------------------------------------------------------------
# 2. RECONSTRUCT TURNING AT FOUR SPEED THRESHOLDS
# ---------------------------------------------------------------------

def angular_difference(a, b):
    return (a - b + 180) % 360 - 180


attempt_records = []

for key, group in tracking.groupby(GROUP_KEYS, sort=False):

    group = group.sort_values("time")

    x = group["x"].to_numpy(dtype=float)
    y = group["y"].to_numpy(dtype=float)
    speed = group["s"].to_numpy(dtype=float)
    recorded_dir = group["dir"].to_numpy(dtype=float)

    if len(group) < 3:
        continue

    dx = np.diff(x)
    dy = np.diff(y)

    # Coordinate heading for each consecutive movement segment.
    coordinate_heading = (
        np.degrees(np.arctan2(dx, dy)) % 360
    )

    # Recorded direction at the endpoint of each segment.
    recorded_heading = recorded_dir[1:]

    # Turning between consecutive movement segments.
    coordinate_turn = np.abs(
        angular_difference(
            coordinate_heading[1:],
            coordinate_heading[:-1]
        )
    )

    recorded_turn = np.abs(
        angular_difference(
            recorded_heading[1:],
            recorded_heading[:-1]
        )
    )

    # Speed at the two observations defining each turning pair.
    pair_speed = np.minimum(speed[1:-1], speed[2:])

    row = dict(zip(GROUP_KEYS, key))
    row["raw_observations"] = len(group)

    for threshold in THRESHOLDS:

        valid = (
            np.isfinite(coordinate_turn)
            & np.isfinite(recorded_turn)
            & np.isfinite(pair_speed)
            & (pair_speed >= threshold)
        )

        c = coordinate_turn[valid]
        r = recorded_turn[valid]

        suffix = str(threshold).replace(".", "_")

        row[f"valid_pairs_t{suffix}"] = int(valid.sum())

        row[f"coordinate_turn_sum_t{suffix}"] = (
            float(c.sum()) if len(c) else np.nan
        )

        row[f"recorded_turn_sum_t{suffix}"] = (
            float(r.sum()) if len(r) else np.nan
        )

        row[f"coordinate_turn_mean_t{suffix}"] = (
            float(c.mean()) if len(c) else np.nan
        )

        row[f"recorded_turn_mean_t{suffix}"] = (
            float(r.mean()) if len(r) else np.nan
        )

    # CRITICAL FIX: save each reconstructed attempt.
    attempt_records.append(row)


# ---------------------------------------------------------------------
# 3. CREATE AND VALIDATE ATTEMPT DATASET
# ---------------------------------------------------------------------

attempt_sensitivity = pd.DataFrame(attempt_records)

if attempt_sensitivity.empty:
    raise RuntimeError(
        "No attempts reconstructed. Inspect tracking filters and grouping keys."
    )

print("\nATTEMPT DATASET")
print("Shape:", attempt_sensitivity.shape)
print("Unique attempts:", attempt_sensitivity.groupby(GROUP_KEYS).ngroups)
print("Unique players:", attempt_sensitivity["nfl_id"].nunique())

assert not attempt_sensitivity.duplicated(GROUP_KEYS).any()

attempt_sensitivity.to_parquet(
    FEATURE_DIR / "wr_turning_threshold_attempts_step8e.parquet",
    index=False
)


# ---------------------------------------------------------------------
# 4. AGGREGATE ATTEMPTS TO ONE PLAYER-DRILL RECORD
# ---------------------------------------------------------------------

metric_columns = []

for threshold in THRESHOLDS:

    suffix = str(threshold).replace(".", "_")

    metric_columns.extend([
        f"coordinate_turn_sum_t{suffix}",
        f"recorded_turn_sum_t{suffix}",
        f"coordinate_turn_mean_t{suffix}",
        f"recorded_turn_mean_t{suffix}"
    ])

aggregation = {
    col: (col, "median")
    for col in metric_columns
}

aggregation["attempts_in_drill"] = ("attempt", "nunique")

player_drill = (
    attempt_sensitivity
    .groupby(
        ["draft_year", "nfl_id", "drill_name"],
        as_index=False
    )
    .agg(**aggregation)
)

assert not player_drill.duplicated(
    ["draft_year", "nfl_id", "drill_name"]
).any()

player_drill.to_parquet(
    FEATURE_DIR / "wr_turning_threshold_player_drill_step8e.parquet",
    index=False
)

player_drill.to_csv(
    FEATURE_DIR / "wr_turning_threshold_player_drill_step8e.csv",
    index=False
)

print("\nPLAYER-DRILL DATASET")
print("Shape:", player_drill.shape)
print("Players:", player_drill["nfl_id"].nunique())
print("Player-drill records:", len(player_drill))


# ---------------------------------------------------------------------
# 5. PLAYER-WEIGHTED COHORT SUMMARY
# ---------------------------------------------------------------------

summary_records = []

for (drill, year), group in player_drill.groupby(
    ["drill_name", "draft_year"]
):

    row = {
        "drill_name": drill,
        "draft_year": year,
        "players": group["nfl_id"].nunique()
    }

    for col in metric_columns:
        row[f"{col}_median"] = group[col].median()

    summary_records.append(row)

cohort_summary = pd.DataFrame(summary_records)

cohort_summary.to_csv(
    AUDIT_DIR / "step8e_player_weighted_cohort_summary.csv",
    index=False
)


# ---------------------------------------------------------------------
# 6. COMPARE 2025 WITH 2023 AT EACH SPEED THRESHOLD
# ---------------------------------------------------------------------

change_records = []

for drill, group in player_drill.groupby("drill_name"):

    row = {"drill_name": drill}

    for threshold in THRESHOLDS:

        suffix = str(threshold).replace(".", "_")

        for measure in [
            "coordinate_turn_sum",
            "recorded_turn_sum",
            "coordinate_turn_mean",
            "recorded_turn_mean"
        ]:

            col = f"{measure}_t{suffix}"

            medians = group.groupby("draft_year")[col].median()

            m23 = medians.get(2023, np.nan)
            m25 = medians.get(2025, np.nan)

            row[f"{col}_2023"] = m23
            row[f"{col}_2025"] = m25

            row[f"{col}_change_pct"] = (
                ((m25 - m23) / abs(m23)) * 100
                if pd.notna(m23) and abs(m23) > 1e-10
                else np.nan
            )

    change_records.append(row)

threshold_comparison = pd.DataFrame(change_records)

threshold_comparison.to_csv(
    AUDIT_DIR / "step8e_player_weighted_threshold_comparison.csv",
    index=False
)

print("\nPLAYER-WEIGHTED COORDINATE TURNING CHANGE (%)")

display(
    threshold_comparison[
        [
            "drill_name",
            "coordinate_turn_sum_t0_5_change_pct",
            "coordinate_turn_sum_t1_0_change_pct",
            "coordinate_turn_sum_t2_0_change_pct",
            "coordinate_turn_sum_t3_0_change_pct"
        ]
    ].round(2)
)

print("\nPLAYER-WEIGHTED RECORDED TURNING CHANGE (%)")

display(
    threshold_comparison[
        [
            "drill_name",
            "recorded_turn_sum_t0_5_change_pct",
            "recorded_turn_sum_t1_0_change_pct",
            "recorded_turn_sum_t2_0_change_pct",
            "recorded_turn_sum_t3_0_change_pct"
        ]
    ].round(2)
)


# ---------------------------------------------------------------------
# 7. THRESHOLD COVERAGE
# ---------------------------------------------------------------------

coverage_records = []

for threshold in THRESHOLDS:

    suffix = str(threshold).replace(".", "_")
    col = f"valid_pairs_t{suffix}"

    for year, group in attempt_sensitivity.groupby("draft_year"):

        coverage_records.append({
            "draft_year": year,
            "threshold": threshold,
            "attempts": len(group),
            "median_valid_pairs": group[col].median(),
            "attempts_with_20_or_more_pairs": int(
                (group[col] >= 20).sum()
            ),
            "attempts_with_40_or_more_pairs": int(
                (group[col] >= 40).sum()
            )
        })

coverage = pd.DataFrame(coverage_records)

coverage.to_csv(
    AUDIT_DIR / "step8e_threshold_coverage.csv",
    index=False
)

print("\nTHRESHOLD COVERAGE")
display(coverage.round(2))


# ---------------------------------------------------------------------
# 8. FINAL VALIDATION
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 8E — FINAL VALIDATION")
print("=" * 70)

print("Attempt rows:", len(attempt_sensitivity))
print("Player-drill rows:", len(player_drill))
print("Players:", player_drill["nfl_id"].nunique())
print("Drills:", player_drill["drill_name"].nunique())

print(
    "Duplicate player-drill keys:",
    player_drill.duplicated(
        ["draft_year", "nfl_id", "drill_name"]
    ).sum()
)

print("\nSAVED REPORTS")

for file in sorted(AUDIT_DIR.glob("step8e_*.csv")):
    print(file)

print("\nSTEP 8E COMPLETED SUCCESSFULLY.")


Filtered raw observations: 84064

ATTEMPT DATASET
Shape: (1282, 27)
Unique attempts: 1282
Unique players: 107

PLAYER-DRILL DATASET
Shape: (1113, 20)
Players: 107
Player-drill records: 1113

PLAYER-WEIGHTED COORDINATE TURNING CHANGE (%)


,drill_name,coordinate_turn_sum_t0_5_change_pct,coordinate_turn_sum_t1_0_change_pct,coordinate_turn_sum_t2_0_change_pct,coordinate_turn_sum_t3_0_change_pct
0,COMEBACK_ROUTE_RIGHT,10.56,-5.32,-7.49,-48.59
1,CURL_ROUTE_RIGHT,-19.34,-13.83,-14.57,26.36
2,DAGGER_ROUTE_LEFT,-29.87,-26.38,-20.50,-17.57
3,GAUNTLET_DRILL,-43.36,-41.11,-36.74,-34.21
4,GO_ROUTE_RIGHT,-66.66,-65.56,-63.15,-61.84
5,OVER_SHOULDER_ADJUST,-62.24,-69.14,-67.72,-66.98
6,POST_CORNER_ROUTE_RIGHT,-47.21,-43.94,-40.41,-36.63
7,RED_ZONE_FADE_RIGHT,-63.22,-55.03,-54.16,-46.67
8,SLANT_ROUTE_LEFT,-50.20,-43.10,-37.89,-37.88
9,SLOT_SAIL_ROUTE_RIGHT,-24.69,-19.50,-13.68,-9.41



PLAYER-WEIGHTED RECORDED TURNING CHANGE (%)


,drill_name,recorded_turn_sum_t0_5_change_pct,recorded_turn_sum_t1_0_change_pct,recorded_turn_sum_t2_0_change_pct,recorded_turn_sum_t3_0_change_pct
0,COMEBACK_ROUTE_RIGHT,55.12,33.36,11.74,-39.54
1,CURL_ROUTE_RIGHT,-0.25,7.73,-1.51,74.83
2,DAGGER_ROUTE_LEFT,-3.88,-4.77,-2.33,-3.58
3,GAUNTLET_DRILL,-2.90,-2.14,-0.94,-0.45
4,GO_ROUTE_RIGHT,-28.14,-21.71,-16.47,-18.54
5,OVER_SHOULDER_ADJUST,53.49,31.13,35.44,42.20
6,POST_CORNER_ROUTE_RIGHT,-27.18,-25.57,-23.41,-22.53
7,RED_ZONE_FADE_RIGHT,-17.04,-8.60,-1.48,16.25
8,SLANT_ROUTE_LEFT,-29.96,-30.50,-28.71,-29.55
9,SLOT_SAIL_ROUTE_RIGHT,-5.15,-4.79,-4.30,-3.77



THRESHOLD COVERAGE


,draft_year,threshold,attempts,median_valid_pairs,attempts_with_20_or_more_pairs,attempts_with_40_or_more_pairs
0,2023,0.5,468,61.0,468,439
1,2024,0.5,377,61.0,377,352
2,2025,0.5,437,64.0,436,383
3,2023,1.0,468,59.0,468,411
4,2024,1.0,377,60.0,377,328
5,2025,1.0,437,62.0,435,380
6,2023,2.0,468,57.0,468,375
7,2024,2.0,377,57.0,377,316
8,2025,2.0,437,58.0,434,363
9,2023,3.0,468,54.0,468,335



STEP 8E — FINAL VALIDATION
Attempt rows: 1282
Player-drill rows: 1113
Players: 107
Drills: 11
Duplicate player-drill keys: 0

SAVED REPORTS
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8e_player_weighted_cohort_summary.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8e_player_weighted_threshold_comparison.csv
/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8e_threshold_coverage.csv

STEP 8E COMPLETED SUCCESSFULLY.


In [17]:

# =====================================================================
# NOTEBOOK 06 — STEP 8F
# RAW MOVEMENT MEASUREMENT DIAGNOSTICS
#
# Objectives:
# 1. Compare coordinate displacement with recorded displacement.
# 2. Compare coordinate-derived speed with recorded speed.
# 3. Validate coordinate heading against recorded direction.
# 4. Compare these relationships across draft cohorts and drills.
# 5. Save reproducible audit datasets and reports.
# =====================================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# 0. PROJECT PATHS
# ---------------------------------------------------------------------

ROOT = Path(
    "/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027"
)

RAW_DIR = ROOT / "data/raw/nfl-big-data-bowl-2027"
FEATURE_DIR = ROOT / "data/features"
AUDIT_DIR = ROOT / "reports/data_audit"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw data directory:", RAW_DIR)


# ---------------------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------------------

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

GROUP_KEYS = [
    "draft_year",
    "nfl_id",
    "drill_type",
    "drill_name",
    "attempt",
    "event_id"
]

SPEED_THRESHOLDS = [0.5, 1.0, 2.0, 3.0]


# ---------------------------------------------------------------------
# 2. LOAD COMBINE RESULTS AND IDENTIFY WIDE RECEIVERS
# ---------------------------------------------------------------------

combine = pd.read_csv(
    RAW_DIR / "combine_results.csv"
)

combine["nfl_id"] = pd.to_numeric(
    combine["nfl_id"],
    errors="coerce"
)

wr_ids = set(
    combine.loc[
        combine["combine_position"]
        .astype(str)
        .str.upper()
        .eq("WR"),
        "nfl_id"
    ].dropna().astype(int)
)

print("\nCOMBINE WR VALIDATION")
print("Wide receiver IDs:", len(wr_ids))


# ---------------------------------------------------------------------
# 3. LOAD RAW COMBINE TRACKING
# ---------------------------------------------------------------------

df = pd.read_csv(
    RAW_DIR / "combine_tracking.csv"
)

numeric_columns = [
    "nfl_id",
    "draft_year",
    "attempt",
    "x",
    "y",
    "s",
    "a",
    "dis",
    "dir"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df["time"] = pd.to_datetime(
    df["time"],
    errors="coerce"
)


# ---------------------------------------------------------------------
# 4. FILTER PLAYER-LEVEL WR PRIMARY DRILLS
# ---------------------------------------------------------------------

df = df[
    (df["entity_type"] == "PLAYER")
    & (df["nfl_id"].isin(wr_ids))
    & (df["drill_name"].isin(PRIMARY_DRILLS))
].copy()

df = df.dropna(
    subset=GROUP_KEYS + [
        "time",
        "x",
        "y",
        "s",
        "dis",
        "dir"
    ]
)

df = df.sort_values(
    GROUP_KEYS + ["time"]
).reset_index(drop=True)

print("\nRAW TRACKING FILTER")
print("Filtered observations:", len(df))
print("Unique players:", df["nfl_id"].nunique())
print("Draft cohorts:", sorted(df["draft_year"].unique()))
print("Drills:", df["drill_name"].nunique())


# ---------------------------------------------------------------------
# 5. RECONSTRUCT INTERVAL-LEVEL MEASUREMENTS
# ---------------------------------------------------------------------

attempt_rows = []

for key, g in df.groupby(
    GROUP_KEYS,
    sort=False
):

    g = g.sort_values("time")

    x = g["x"].to_numpy(dtype=float)
    y = g["y"].to_numpy(dtype=float)

    t = (
        g["time"]
        .astype("int64")
        .to_numpy(dtype=float) / 1e9
    )

    recorded_speed = g["s"].to_numpy(dtype=float)
    recorded_dis = g["dis"].to_numpy(dtype=float)
    recorded_dir = g["dir"].to_numpy(dtype=float)

    if len(g) < 3:
        continue

    # -------------------------------------------------------------
    # 5A. COORDINATE MOVEMENT BETWEEN CONSECUTIVE OBSERVATIONS
    # -------------------------------------------------------------

    dx = np.diff(x)
    dy = np.diff(y)
    dt = np.diff(t)

    coordinate_step = np.hypot(dx, dy)

    valid_dt = (
        np.isfinite(dt)
        & (dt > 0)
    )

    coordinate_speed = np.full(
        len(dt),
        np.nan
    )

    coordinate_speed[valid_dt] = (
        coordinate_step[valid_dt] / dt[valid_dt]
    )

    # -------------------------------------------------------------
    # 5B. ALIGN RECORDED MEASUREMENTS TO INTERVAL ENDPOINTS
    # -------------------------------------------------------------

    endpoint_speed = recorded_speed[1:]
    endpoint_dis = recorded_dis[1:]
    endpoint_dir = recorded_dir[1:]

    # -------------------------------------------------------------
    # 5C. COORDINATE-DERIVED HEADING
    # Direction clockwise from positive Y axis.
    # -------------------------------------------------------------

    coordinate_heading = (
        np.degrees(
            np.arctan2(dx, dy)
        ) % 360
    )

    heading_error = np.abs(
        (
            coordinate_heading
            - endpoint_dir
            + 180
        ) % 360 - 180
    )

    # -------------------------------------------------------------
    # 5D. COMMON VALIDITY MASK FOR MOVEMENT MEASUREMENTS
    # -------------------------------------------------------------

    valid_geometry = (
        valid_dt
        & np.isfinite(coordinate_step)
        & np.isfinite(endpoint_dis)
        & np.isfinite(coordinate_speed)
        & np.isfinite(endpoint_speed)
    )

    row = dict(
        zip(GROUP_KEYS, key)
    )

    row["observations"] = len(g)
    row["valid_intervals"] = int(
        valid_geometry.sum()
    )

    # -------------------------------------------------------------
    # 5E. DISPLACEMENT AND SPEED DIAGNOSTICS
    # -------------------------------------------------------------

    if valid_geometry.any():

        cstep = coordinate_step[valid_geometry]
        rdis = endpoint_dis[valid_geometry]

        cspeed = coordinate_speed[valid_geometry]
        rspeed = endpoint_speed[valid_geometry]

        gaps = dt[valid_geometry]

        row["median_coordinate_step"] = float(
            np.median(cstep)
        )

        row["median_recorded_displacement"] = float(
            np.median(rdis)
        )

        row["median_coordinate_speed"] = float(
            np.median(cspeed)
        )

        row["median_recorded_speed"] = float(
            np.median(rspeed)
        )

        row["median_sampling_interval"] = float(
            np.median(gaps)
        )

        # CRITICAL: use a shared mask for paired ratios.
        valid_dis = (
            np.isfinite(cstep)
            & np.isfinite(rdis)
            & (rdis > 0)
        )

        valid_speed = (
            np.isfinite(cspeed)
            & np.isfinite(rspeed)
            & (rspeed > 0)
        )

        row["median_coordinate_to_recorded_dis_ratio"] = (
            float(
                np.median(
                    cstep[valid_dis] / rdis[valid_dis]
                )
            )
            if valid_dis.any()
            else np.nan
        )

        row["median_coordinate_to_recorded_speed_ratio"] = (
            float(
                np.median(
                    cspeed[valid_speed] / rspeed[valid_speed]
                )
            )
            if valid_speed.any()
            else np.nan
        )

        row["valid_displacement_ratio_pairs"] = int(
            valid_dis.sum()
        )

        row["valid_speed_ratio_pairs"] = int(
            valid_speed.sum()
        )

        # Absolute differences provide additional diagnostics.
        row["median_absolute_displacement_difference"] = float(
            np.median(
                np.abs(cstep - rdis)
            )
        )

        row["median_absolute_speed_difference"] = float(
            np.median(
                np.abs(cspeed - rspeed)
            )
        )

    else:

        for col in [
            "median_coordinate_step",
            "median_recorded_displacement",
            "median_coordinate_speed",
            "median_recorded_speed",
            "median_sampling_interval",
            "median_coordinate_to_recorded_dis_ratio",
            "median_coordinate_to_recorded_speed_ratio",
            "median_absolute_displacement_difference",
            "median_absolute_speed_difference"
        ]:
            row[col] = np.nan

        row["valid_displacement_ratio_pairs"] = 0
        row["valid_speed_ratio_pairs"] = 0

    # -------------------------------------------------------------
    # 5F. HEADING AGREEMENT AT DIFFERENT SPEED THRESHOLDS
    # -------------------------------------------------------------

    for threshold in SPEED_THRESHOLDS:

        suffix = str(threshold).replace(".", "_")

        heading_mask = (
            np.isfinite(heading_error)
            & np.isfinite(endpoint_speed)
            & (endpoint_speed >= threshold)
        )

        row[f"heading_pairs_t{suffix}"] = int(
            heading_mask.sum()
        )

        row[f"heading_error_median_t{suffix}"] = (
            float(
                np.median(
                    heading_error[heading_mask]
                )
            )
            if heading_mask.any()
            else np.nan
        )

        row[f"heading_error_p90_t{suffix}"] = (
            float(
                np.percentile(
                    heading_error[heading_mask],
                    90
                )
            )
            if heading_mask.any()
            else np.nan
        )

    # -------------------------------------------------------------
    # 5G. SAVE ATTEMPT RESULT
    # -------------------------------------------------------------

    attempt_rows.append(row)


# ---------------------------------------------------------------------
# 6. BUILD ATTEMPT-LEVEL DIAGNOSTIC DATASET
# ---------------------------------------------------------------------

attempt_audit = pd.DataFrame(
    attempt_rows
)

if attempt_audit.empty:
    raise RuntimeError(
        "No attempt diagnostics generated. Inspect the filters."
    )

assert not attempt_audit.duplicated(
    GROUP_KEYS
).any()

attempt_audit.to_parquet(
    FEATURE_DIR
    / "wr_raw_measurement_diagnostics_step8f.parquet",
    index=False
)

attempt_audit.to_csv(
    AUDIT_DIR
    / "step8f_attempt_measurement_diagnostics.csv",
    index=False
)

print("\nATTEMPT DIAGNOSTICS")
print("Shape:", attempt_audit.shape)
print("Attempts:", len(attempt_audit))
print("Players:", attempt_audit["nfl_id"].nunique())
print("Drills:", attempt_audit["drill_name"].nunique())


# ---------------------------------------------------------------------
# 7. PLAYER-DRILL AGGREGATION
# Each player receives equal weight within each drill and cohort.
# ---------------------------------------------------------------------

diagnostic_cols = [
    "median_coordinate_step",
    "median_recorded_displacement",
    "median_coordinate_speed",
    "median_recorded_speed",
    "median_sampling_interval",
    "median_coordinate_to_recorded_dis_ratio",
    "median_coordinate_to_recorded_speed_ratio",
    "median_absolute_displacement_difference",
    "median_absolute_speed_difference",
    "valid_displacement_ratio_pairs",
    "valid_speed_ratio_pairs"
]

for threshold in SPEED_THRESHOLDS:

    suffix = str(threshold).replace(".", "_")

    diagnostic_cols.extend([
        f"heading_error_median_t{suffix}",
        f"heading_error_p90_t{suffix}"
    ])

player_drill_audit = (
    attempt_audit
    .groupby(
        [
            "draft_year",
            "nfl_id",
            "drill_name"
        ],
        as_index=False
    )[diagnostic_cols]
    .median()
)

assert not player_drill_audit.duplicated(
    ["draft_year", "nfl_id", "drill_name"]
).any()

player_drill_audit.to_parquet(
    FEATURE_DIR
    / "wr_raw_measurement_player_drill_step8f.parquet",
    index=False
)

player_drill_audit.to_csv(
    FEATURE_DIR
    / "wr_raw_measurement_player_drill_step8f.csv",
    index=False
)

print("\nPLAYER-DRILL DATASET")
print("Shape:", player_drill_audit.shape)
print("Players:", player_drill_audit["nfl_id"].nunique())
print("Records:", len(player_drill_audit))


# ---------------------------------------------------------------------
# 8. PLAYER-WEIGHTED COHORT SUMMARY
# ---------------------------------------------------------------------

cohort_aggregation = {
    "players": ("nfl_id", "nunique")
}

for col in diagnostic_cols:

    cohort_aggregation[
        f"{col}_cohort_median"
    ] = (col, "median")

cohort_summary = (
    player_drill_audit
    .groupby(
        ["draft_year", "drill_name"],
        as_index=False
    )
    .agg(**cohort_aggregation)
)

cohort_summary.to_csv(
    AUDIT_DIR
    / "step8f_player_weighted_cohort_summary.csv",
    index=False
)


# ---------------------------------------------------------------------
# 9. OVERALL COHORT MEDIANS
# ---------------------------------------------------------------------

overall = (
    player_drill_audit
    .groupby("draft_year")[diagnostic_cols]
    .median()
    .round(4)
)

print("\n" + "=" * 70)
print("OVERALL PLAYER-WEIGHTED COHORT MEDIANS")
print("=" * 70)

display(overall)


# ---------------------------------------------------------------------
# 10. HEADING AGREEMENT BY DRILL AND COHORT
# ---------------------------------------------------------------------

heading_columns = [
    "heading_error_median_t1_0",
    "heading_error_p90_t1_0",
    "heading_error_median_t3_0",
    "heading_error_p90_t3_0"
]

heading_aggregation = {
    "players": ("nfl_id", "nunique")
}

for col in heading_columns:

    heading_aggregation[
        f"{col}_median"
    ] = (col, "median")

heading_by_drill = (
    player_drill_audit
    .groupby(
        ["drill_name", "draft_year"],
        as_index=False
    )
    .agg(**heading_aggregation)
)

heading_by_drill.to_csv(
    AUDIT_DIR
    / "step8f_heading_agreement_by_drill.csv",
    index=False
)

print("\n" + "=" * 70)
print("HEADING AGREEMENT BY DRILL AND COHORT")
print("=" * 70)

display(
    heading_by_drill.round(3)
)


# ---------------------------------------------------------------------
# 11. COORDINATE / RECORDED RATIOS BY DRILL AND COHORT
# ---------------------------------------------------------------------

ratio_cols = [
    "median_coordinate_to_recorded_dis_ratio",
    "median_coordinate_to_recorded_speed_ratio"
]

ratio_aggregation = {
    "players": ("nfl_id", "nunique")
}

for col in ratio_cols:

    ratio_aggregation[
        f"{col}_median"
    ] = (col, "median")

ratio_summary = (
    player_drill_audit
    .groupby(
        ["draft_year", "drill_name"],
        as_index=False
    )
    .agg(**ratio_aggregation)
)

ratio_summary.to_csv(
    AUDIT_DIR
    / "step8f_coordinate_recorded_ratios_by_drill.csv",
    index=False
)

print("\n" + "=" * 70)
print("COORDINATE / RECORDED MEASUREMENT RATIOS")
print("=" * 70)

display(
    ratio_summary.round(4)
)


# ---------------------------------------------------------------------
# 12. VALIDATION SUMMARY
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 8F — FINAL VALIDATION")
print("=" * 70)

print("Raw observations:", len(df))
print("Attempt rows:", len(attempt_audit))
print("Player-drill rows:", len(player_drill_audit))
print("Unique players:", player_drill_audit["nfl_id"].nunique())
print("Unique drills:", player_drill_audit["drill_name"].nunique())

print(
    "Duplicate attempt keys:",
    attempt_audit.duplicated(GROUP_KEYS).sum()
)

print(
    "Duplicate player-drill keys:",
    player_drill_audit.duplicated(
        ["draft_year", "nfl_id", "drill_name"]
    ).sum()
)

print(
    "Attempts with no valid geometry intervals:",
    (attempt_audit["valid_intervals"] == 0).sum()
)

print("\nSAVED OUTPUTS")

outputs = [
    FEATURE_DIR / "wr_raw_measurement_diagnostics_step8f.parquet",
    FEATURE_DIR / "wr_raw_measurement_player_drill_step8f.parquet",
    FEATURE_DIR / "wr_raw_measurement_player_drill_step8f.csv",
    AUDIT_DIR / "step8f_attempt_measurement_diagnostics.csv",
    AUDIT_DIR / "step8f_player_weighted_cohort_summary.csv",
    AUDIT_DIR / "step8f_heading_agreement_by_drill.csv",
    AUDIT_DIR / "step8f_coordinate_recorded_ratios_by_drill.csv"
]

for path in outputs:
    print(
        "VERIFIED" if path.exists() else "MISSING",
        path
    )

assert all(path.exists() for path in outputs)

print("\nSTEP 8F COMPLETED SUCCESSFULLY.")


Project root: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027
Raw data directory: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/raw/nfl-big-data-bowl-2027

COMBINE WR VALIDATION
Wide receiver IDs: 107

RAW TRACKING FILTER
Filtered observations: 84064
Unique players: 107
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]
Drills: 11

ATTEMPT DIAGNOSTICS
Shape: (1282, 31)
Attempts: 1282
Players: 107
Drills: 11

PLAYER-DRILL DATASET
Shape: (1113, 22)
Players: 107
Records: 1113

OVERALL PLAYER-WEIGHTED COHORT MEDIANS


,median_coordinate_step,median_recorded_displacement,median_coordinate_speed,median_recorded_speed,median_sampling_interval,median_coordinate_to_recorded_dis_ratio,median_coordinate_to_recorded_speed_ratio,median_absolute_displacement_difference,median_absolute_speed_difference,valid_displacement_ratio_pairs,valid_speed_ratio_pairs,heading_error_median_t0_5,heading_error_p90_t0_5,heading_error_median_t1_0,heading_error_p90_t1_0,heading_error_median_t2_0,heading_error_p90_t2_0,heading_error_median_t3_0,heading_error_p90_t3_0
draft_year,,,,,,,,,,,,,,,,,,,
2023,0.6469,0.3600,6.4687,6.4967,0.1,1.7647,0.9956,0.2348,0.1233,62.0,62.0,1.3231,3.5296,1.2605,3.3025,1.1860,2.9010,1.1140,2.6888
2024,0.6667,0.3768,6.6667,6.7033,0.1,1.7299,0.9965,0.2381,0.1230,62.0,62.0,1.3132,3.6687,1.2770,3.3690,1.2024,2.9693,1.1367,2.7811
2025,0.6209,0.3544,6.2090,6.1900,0.1,1.8072,0.9974,0.2334,0.0665,65.0,65.0,0.7285,2.1628,0.7192,2.0978,0.6503,1.9630,0.6041,1.8876



HEADING AGREEMENT BY DRILL AND COHORT


,drill_name,draft_year,players,heading_error_median_t1_0_median,heading_error_p90_t1_0_median,heading_error_median_t3_0_median,heading_error_p90_t3_0_median
0,COMEBACK_ROUTE_RIGHT,2023,36,1.360,4.506,1.157,3.952
1,COMEBACK_ROUTE_RIGHT,2024,28,1.394,4.876,1.156,3.763
2,COMEBACK_ROUTE_RIGHT,2025,34,0.577,3.395,0.350,1.712
3,CURL_ROUTE_RIGHT,2023,37,1.861,5.686,1.447,3.590
4,CURL_ROUTE_RIGHT,2024,30,1.730,5.386,1.464,3.788
5,CURL_ROUTE_RIGHT,2025,36,1.246,4.459,1.094,3.118
6,DAGGER_ROUTE_LEFT,2023,39,1.439,3.192,1.328,2.922
7,DAGGER_ROUTE_LEFT,2024,31,1.414,3.354,1.349,2.925
8,DAGGER_ROUTE_LEFT,2025,37,1.122,2.074,1.122,2.033
9,GAUNTLET_DRILL,2023,39,0.975,2.853,0.915,2.491



COORDINATE / RECORDED MEASUREMENT RATIOS


,draft_year,drill_name,players,median_coordinate_to_recorded_dis_ratio_median,median_coordinate_to_recorded_speed_ratio_median
0,2023,COMEBACK_ROUTE_RIGHT,36,2.1642,1.0001
1,2023,CURL_ROUTE_RIGHT,37,3.0000,0.9953
2,2023,DAGGER_ROUTE_LEFT,39,1.6674,0.9969
3,2023,GAUNTLET_DRILL,39,1.3346,0.9963
4,2023,GO_ROUTE_RIGHT,35,1.1918,0.9957
5,2023,OVER_SHOULDER_ADJUST,37,1.5789,0.9939
6,2023,POST_CORNER_ROUTE_RIGHT,36,1.5402,0.9950
7,2023,RED_ZONE_FADE_RIGHT,36,2.7428,0.9933
8,2023,SLANT_ROUTE_LEFT,39,2.0716,0.9879
9,2023,SLOT_SAIL_ROUTE_RIGHT,33,1.6122,0.9984



STEP 8F — FINAL VALIDATION
Raw observations: 84064
Attempt rows: 1282
Player-drill rows: 1113
Unique players: 107
Unique drills: 11
Duplicate attempt keys: 0
Duplicate player-drill keys: 0
Attempts with no valid geometry intervals: 0

SAVED OUTPUTS
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_raw_measurement_diagnostics_step8f.parquet
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_raw_measurement_player_drill_step8f.parquet
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_raw_measurement_player_drill_step8f.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8f_attempt_measurement_diagnostics.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8f_player_weighted_cohort_summary.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8f_heading_agreement_by_drill.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-Big

In [18]:

# =====================================================================
# NOTEBOOK 06 — STEP 8G
# RECORDED DIRECTION TIMESTAMP ALIGNMENT AUDIT
# =====================================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
RAW_DIR = ROOT / "data/raw/nfl-big-data-bowl-2027"
FEATURE_DIR = ROOT / "data/features"
AUDIT_DIR = ROOT / "reports/data_audit"

PRIMARY_DRILLS = [
    "GAUNTLET_DRILL",
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "SLOT_STRIKE_ROUTE_LEFT",
    "OVER_SHOULDER_ADJUST",
    "RED_ZONE_FADE_RIGHT",
    "COMEBACK_ROUTE_RIGHT",
    "SLOT_SAIL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "POST_CORNER_ROUTE_RIGHT"
]

GROUP_KEYS = [
    "draft_year", "nfl_id", "drill_type",
    "drill_name", "attempt", "event_id"
]

LAGS = [-2, -1, 0, 1, 2]

# ---------------------------------------------------------------------
# 1. LOAD AND FILTER
# ---------------------------------------------------------------------

combine = pd.read_csv(RAW_DIR / "combine_results.csv")

combine["nfl_id"] = pd.to_numeric(
    combine["nfl_id"], errors="coerce"
)

wr_ids = set(
    combine.loc[
        combine["combine_position"].astype(str).str.upper().eq("WR"),
        "nfl_id"
    ].dropna().astype(int)
)

df = pd.read_csv(RAW_DIR / "combine_tracking.csv")

for col in ["nfl_id", "draft_year", "attempt",
            "x", "y", "s", "dir"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["time"] = pd.to_datetime(df["time"], errors="coerce")

df = df[
    df["entity_type"].eq("PLAYER")
    & df["nfl_id"].isin(wr_ids)
    & df["drill_name"].isin(PRIMARY_DRILLS)
].dropna(
    subset=GROUP_KEYS + ["time", "x", "y", "s", "dir"]
).copy()

df = df.sort_values(
    GROUP_KEYS + ["time"]
).reset_index(drop=True)

print("Filtered observations:", len(df))


# ---------------------------------------------------------------------
# 2. RECONSTRUCT HEADINGS AND TEST LAGS
# ---------------------------------------------------------------------

records = []

for key, g in df.groupby(GROUP_KEYS, sort=False):

    g = g.sort_values("time")

    x = g["x"].to_numpy(float)
    y = g["y"].to_numpy(float)
    speed = g["s"].to_numpy(float)
    direction = g["dir"].to_numpy(float)

    if len(g) < 5:
        continue

    dx = np.diff(x)
    dy = np.diff(y)

    coordinate_heading = (
        np.degrees(np.arctan2(dx, dy)) % 360
    )

    # Each coordinate heading corresponds to the endpoint
    # of its displacement interval.
    recorded_endpoint_direction = direction[1:]
    endpoint_speed = speed[1:]

    row_base = dict(zip(GROUP_KEYS, key))

    for lag in LAGS:

        # Positive lag compares coordinate heading at time t
        # against recorded direction at a later observation.
        if lag >= 0:
            c = coordinate_heading[:len(coordinate_heading)-lag or None]
            r = recorded_endpoint_direction[lag:]
            s = endpoint_speed[:len(endpoint_speed)-lag or None]
        else:
            k = abs(lag)
            c = coordinate_heading[k:]
            r = recorded_endpoint_direction[:-k]
            s = endpoint_speed[k:]

        n = min(len(c), len(r), len(s))
        c, r, s = c[:n], r[:n], s[:n]

        error = np.abs((c - r + 180) % 360 - 180)

        valid = (
            np.isfinite(error)
            & np.isfinite(s)
            & (s >= 1.0)
        )

        if not valid.any():
            continue

        records.append({
            **row_base,
            "lag": lag,
            "valid_pairs": int(valid.sum()),
            "median_heading_error": float(
                np.median(error[valid])
            ),
            "p90_heading_error": float(
                np.percentile(error[valid], 90)
            )
        })


lag_attempts = pd.DataFrame(records)

if lag_attempts.empty:
    raise RuntimeError("No lag diagnostic records generated.")

lag_attempts.to_parquet(
    FEATURE_DIR / "wr_direction_lag_attempts_step8g.parquet",
    index=False
)


# ---------------------------------------------------------------------
# 3. PLAYER-WEIGHTED LAG SUMMARY
# ---------------------------------------------------------------------

player_lag = (
    lag_attempts
    .groupby(
        ["draft_year", "nfl_id", "drill_name", "lag"],
        as_index=False
    )
    .agg(
        median_heading_error=("median_heading_error", "median"),
        median_p90_error=("p90_heading_error", "median"),
        median_valid_pairs=("valid_pairs", "median")
    )
)

cohort_lag = (
    player_lag
    .groupby(["draft_year", "lag"], as_index=False)
    .agg(
        players=("nfl_id", "nunique"),
        median_heading_error=("median_heading_error", "median"),
        median_p90_error=("median_p90_error", "median"),
        median_valid_pairs=("median_valid_pairs", "median")
    )
)

cohort_lag.to_csv(
    AUDIT_DIR / "step8g_direction_lag_by_cohort.csv",
    index=False
)

print("\nDIRECTION ALIGNMENT BY COHORT")
display(cohort_lag.round(4))


# ---------------------------------------------------------------------
# 4. LAG SUMMARY BY DRILL AND COHORT
# ---------------------------------------------------------------------

drill_lag = (
    player_lag
    .groupby(
        ["drill_name", "draft_year", "lag"],
        as_index=False
    )
    .agg(
        players=("nfl_id", "nunique"),
        median_heading_error=("median_heading_error", "median"),
        median_p90_error=("median_p90_error", "median")
    )
)

drill_lag.to_csv(
    AUDIT_DIR / "step8g_direction_lag_by_drill.csv",
    index=False
)


# ---------------------------------------------------------------------
# 5. IDENTIFY BEST LAG PER COHORT
# ---------------------------------------------------------------------

best_lag = (
    cohort_lag
    .sort_values(
        ["draft_year", "median_heading_error"]
    )
    .groupby("draft_year", as_index=False)
    .first()
)

print("\nBEST DIRECTION ALIGNMENT BY COHORT")
display(best_lag.round(4))


# ---------------------------------------------------------------------
# 6. FINAL VALIDATION
# ---------------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 8G — FINAL VALIDATION")
print("=" * 65)

print("Attempt-lag records:", len(lag_attempts))
print("Players:", lag_attempts["nfl_id"].nunique())
print("Drills:", lag_attempts["drill_name"].nunique())
print("Lags:", sorted(lag_attempts["lag"].unique()))

print("\nSAVED FILES")

for path in [
    FEATURE_DIR / "wr_direction_lag_attempts_step8g.parquet",
    AUDIT_DIR / "step8g_direction_lag_by_cohort.csv",
    AUDIT_DIR / "step8g_direction_lag_by_drill.csv"
]:
    print("VERIFIED" if path.exists() else "MISSING", path)

print("\nSTEP 8G COMPLETED.")


Filtered observations: 84064

DIRECTION ALIGNMENT BY COHORT


,draft_year,lag,players,median_heading_error,median_p90_error,median_valid_pairs
0,2023,-2,39,2.7879,7.9488,58.0
1,2023,-1,39,1.4888,3.9227,59.0
2,2023,0,39,1.2605,3.3025,59.0
3,2023,1,39,2.6549,7.0518,58.0
4,2023,2,39,4.0274,11.4887,57.0
5,2024,-2,31,2.8148,7.7678,59.0
6,2024,-1,31,1.5631,4.0095,60.0
7,2024,0,31,1.2770,3.3690,60.0
8,2024,1,31,2.5958,6.9661,59.0
9,2024,2,31,3.8893,11.3241,58.0



BEST DIRECTION ALIGNMENT BY COHORT


,draft_year,lag,players,median_heading_error,median_p90_error,median_valid_pairs
0,2023,0,39,1.2605,3.3025,59.0
1,2024,0,31,1.2770,3.3690,60.0
2,2025,0,37,0.7192,2.0978,61.0



STEP 8G — FINAL VALIDATION
Attempt-lag records: 6410
Players: 107
Drills: 11
Lags: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2)]

SAVED FILES
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_direction_lag_attempts_step8g.parquet
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8g_direction_lag_by_cohort.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8g_direction_lag_by_drill.csv

STEP 8G COMPLETED.


In [19]:

# =====================================================================
# NOTEBOOK 06 — STEP 8H
# FINAL COMBINE FEATURE GOVERNANCE AND APPROVAL REGISTER
# =====================================================================

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

FEATURE_DIR = ROOT / "data/features"
AUDIT_DIR = ROOT / "reports/data_audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = (
    FEATURE_DIR /
    "wr_combine_final_player_features.parquet"
)

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)

df = pd.read_parquet(SOURCE)

print("=" * 70)
print("STEP 8H — FEATURE GOVERNANCE")
print("=" * 70)

print("Source:", SOURCE)
print("Shape:", df.shape)
print("Unique players:", df["nfl_id"].nunique())

assert df["nfl_id"].nunique() == len(df), (
    "Expected one row per Combine player."
)


# ---------------------------------------------------------------------
# 1. DEFINE FEATURE REGISTER
# ---------------------------------------------------------------------

feature_specs = [

    # Traditional Combine predictors
    ("forty", "Traditional athletic testing",
     "PRIMARY", "Standard 40-yard dash benchmark"),

    ("vertical", "Traditional athletic testing",
     "PRIMARY", "Vertical jump performance"),

    ("broad_jump", "Traditional athletic testing",
     "PRIMARY", "Horizontal jump performance"),

    ("ten_yd_split", "Traditional athletic testing",
     "SECONDARY", "Early acceleration benchmark; assess redundancy with forty"),

    # Tracking-derived speed
    ("combine_mean_speed_median_across_drills",
     "Movement speed", "PRIMARY",
     "Median player-level mean speed across drills"),

    ("combine_peak_speed_median_across_drills",
     "Movement speed", "PRIMARY",
     "Median player-level peak speed across drills"),

    # Tracking-derived acceleration
    ("combine_mean_acceleration_median_across_drills",
     "Movement acceleration", "PRIMARY",
     "Median player-level mean acceleration across drills"),

    ("combine_peak_acceleration_median_across_drills",
     "Movement acceleration", "SECONDARY",
     "Peak acceleration; assess redundancy with mean acceleration"),

    # Movement efficiency
    ("combine_path_efficiency_median_across_drills",
     "Movement efficiency", "PRIMARY",
     "Net displacement relative to coordinate path length"),

    # Movement distance
    ("combine_coordinate_path_length_median_across_drills",
     "Movement distance", "SECONDARY",
     "Coordinate-derived distance; verify measurement scale"),

    ("combine_net_displacement_median_across_drills",
     "Movement distance", "SECONDARY",
     "Endpoint displacement; assess redundancy with path length"),

    # NGS comparator features
    ("ngs_athleticism_score", "NGS scores",
     "COMPARATOR", "External athleticism score"),

    ("ngs_college_production_score", "NGS scores",
     "COMPARATOR", "College production score; not a pure movement measure"),

    ("ngs_final_score", "NGS scores",
     "COMPARATOR", "Combined NGS score; assess overlap with component scores"),

    # Low-coverage traditional tests
    ("three_cone", "Low-coverage tests",
     "SENSITIVITY", "Limited player coverage"),

    ("short_shuttle", "Low-coverage tests",
     "SENSITIVITY", "Limited player coverage"),

    ("bench_reps", "Low-coverage tests",
     "SENSITIVITY", "Limited player coverage"),

    # Turning and direction
    ("combine_recorded_total_turn_median_across_drills",
     "Turning mechanics", "EXPLORATORY",
     "Recorded-direction turning; cohort comparability unresolved"),

    ("combine_coordinate_total_turn_median_across_drills",
     "Turning mechanics", "EXPLORATORY",
     "Coordinate turning; cohort comparability unresolved"),

    ("combine_coordinate_mean_turn_median_across_drills",
     "Turning mechanics", "EXPLORATORY",
     "Coordinate mean turn; cohort comparability unresolved"),

    ("combine_coordinate_p95_turn_median_across_drills",
     "Turning mechanics", "EXPLORATORY",
     "Coordinate upper-tail turning; cohort comparability unresolved"),

    ("combine_turning_rate_median_across_drills",
     "Turning mechanics", "EXPLORATORY",
     "Turning rate; cohort comparability unresolved")
]


# ---------------------------------------------------------------------
# 2. CHECK ACTUAL SOURCE COLUMNS
# ---------------------------------------------------------------------

register = []

for feature, family, status, rationale in feature_specs:

    exists = feature in df.columns

    if exists:
        values = pd.to_numeric(
            df[feature],
            errors="coerce"
        )

        missing_pct = values.isna().mean() * 100
        unique_values = values.nunique(dropna=True)

        if unique_values <= 1:
            variance_status = "ZERO_OR_ONE_UNIQUE_VALUE"
        else:
            variance_status = "VARIABLE"

    else:
        missing_pct = np.nan
        unique_values = 0
        variance_status = "COLUMN_NOT_FOUND"

    register.append({
        "feature": feature,
        "feature_family": family,
        "proposed_status": status,
        "available": exists,
        "missing_pct": missing_pct,
        "unique_values": unique_values,
        "variance_status": variance_status,
        "rationale": rationale
    })

feature_register = pd.DataFrame(register)


# ---------------------------------------------------------------------
# 3. IDENTIFY PRIMARY FEATURES ACTUALLY AVAILABLE
# ---------------------------------------------------------------------

primary_register = feature_register[
    (feature_register["proposed_status"] == "PRIMARY")
    & (feature_register["available"])
    & (feature_register["variance_status"] == "VARIABLE")
].copy()

missing_primary = feature_register[
    (feature_register["proposed_status"] == "PRIMARY")
    & (~feature_register["available"])
]

if not missing_primary.empty:
    print("\nWARNING: PRIMARY FEATURES NOT FOUND")
    display(missing_primary[["feature", "feature_family"]])


# ---------------------------------------------------------------------
# 4. SAVE FEATURE REGISTER
# ---------------------------------------------------------------------

register_path = (
    AUDIT_DIR / "step8h_combine_feature_governance.csv"
)

feature_register.to_csv(
    register_path,
    index=False
)


# ---------------------------------------------------------------------
# 5. BUILD APPROVED PRIMARY FEATURE DATASET
# ---------------------------------------------------------------------

approved_features = primary_register["feature"].tolist()

if not approved_features:
    raise RuntimeError(
        "No valid primary predictors were identified."
    )

approved = df[
    ["draft_year", "nfl_id"] + approved_features
].copy()

approved.to_parquet(
    FEATURE_DIR / "wr_combine_approved_primary_features.parquet",
    index=False
)

approved.to_csv(
    FEATURE_DIR / "wr_combine_approved_primary_features.csv",
    index=False
)


# ---------------------------------------------------------------------
# 6. PRIMARY FEATURE MISSINGNESS
# ---------------------------------------------------------------------

missingness = (
    approved[approved_features]
    .isna()
    .mean()
    .mul(100)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

missingness.to_csv(
    AUDIT_DIR / "step8h_primary_feature_missingness.csv",
    index=False
)


# ---------------------------------------------------------------------
# 7. COHORT COVERAGE
# ---------------------------------------------------------------------

cohort_coverage = []

for year, group in approved.groupby("draft_year"):

    for feature in approved_features:

        cohort_coverage.append({
            "draft_year": year,
            "feature": feature,
            "players": len(group),
            "non_missing": group[feature].notna().sum(),
            "missing": group[feature].isna().sum(),
            "coverage_pct": (
                group[feature].notna().mean() * 100
            )
        })

cohort_coverage = pd.DataFrame(cohort_coverage)

cohort_coverage.to_csv(
    AUDIT_DIR / "step8h_primary_feature_cohort_coverage.csv",
    index=False
)


# ---------------------------------------------------------------------
# 8. FINAL REPORT
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE GOVERNANCE SUMMARY")
print("=" * 70)

print("\nFEATURE REGISTER")
display(
    feature_register[
        [
            "feature",
            "feature_family",
            "proposed_status",
            "available",
            "missing_pct",
            "variance_status"
        ]
    ].round(2)
)

print("\nAPPROVED PRIMARY FEATURES")
for feature in approved_features:
    print("-", feature)

print("\nPRIMARY FEATURE DATASET")
print("Rows:", len(approved))
print("Players:", approved["nfl_id"].nunique())
print("Primary features:", len(approved_features))

print("\nPRIMARY FEATURE MISSINGNESS")
display(missingness.round(2))

print("\nCOHORT FEATURE COVERAGE")
display(cohort_coverage.round(2))

print("\nSAVED FILES")

outputs = [
    register_path,
    FEATURE_DIR / "wr_combine_approved_primary_features.parquet",
    FEATURE_DIR / "wr_combine_approved_primary_features.csv",
    AUDIT_DIR / "step8h_primary_feature_missingness.csv",
    AUDIT_DIR / "step8h_primary_feature_cohort_coverage.csv"
]

for path in outputs:
    print("VERIFIED" if path.exists() else "MISSING", path)

assert all(path.exists() for path in outputs)

print("\nSTEP 8H COMPLETED SUCCESSFULLY.")


STEP 8H — FEATURE GOVERNANCE
Source: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_final_player_features.parquet
Shape: (107, 46)
Unique players: 107



,feature,feature_family
4,combine_mean_speed_median_across_drills,Movement speed
5,combine_peak_speed_median_across_drills,Movement speed
6,combine_mean_acceleration_median_across_drills,Movement acceleration
8,combine_path_efficiency_median_across_drills,Movement efficiency



FEATURE GOVERNANCE SUMMARY

FEATURE REGISTER


,feature,feature_family,proposed_status,available,missing_pct,variance_status
0,forty,Traditional athletic testing,PRIMARY,True,9.35,VARIABLE
1,vertical,Traditional athletic testing,PRIMARY,True,10.28,VARIABLE
2,broad_jump,Traditional athletic testing,PRIMARY,True,11.21,VARIABLE
3,ten_yd_split,Traditional athletic testing,SECONDARY,True,9.35,VARIABLE
4,combine_mean_speed_median_across_drills,Movement speed,PRIMARY,False,NaN,COLUMN_NOT_FOUND
5,combine_peak_speed_median_across_drills,Movement speed,PRIMARY,False,NaN,COLUMN_NOT_FOUND
6,combine_mean_acceleration_median_across_drills,Movement acceleration,PRIMARY,False,NaN,COLUMN_NOT_FOUND
7,combine_peak_acceleration_median_across_drills,Movement acceleration,SECONDARY,False,NaN,COLUMN_NOT_FOUND
8,combine_path_efficiency_median_across_drills,Movement efficiency,PRIMARY,False,NaN,COLUMN_NOT_FOUND
9,combine_coordinate_path_length_median_across_d...,Movement distance,SECONDARY,False,NaN,COLUMN_NOT_FOUND



APPROVED PRIMARY FEATURES
- forty
- vertical
- broad_jump

PRIMARY FEATURE DATASET
Rows: 107
Players: 107
Primary features: 3

PRIMARY FEATURE MISSINGNESS


,feature,missing_pct
0,forty,9.35
1,vertical,10.28
2,broad_jump,11.21



COHORT FEATURE COVERAGE


,draft_year,feature,players,non_missing,missing,coverage_pct
0,2023,forty,39,35,4,89.74
1,2023,vertical,39,34,5,87.18
2,2023,broad_jump,39,34,5,87.18
3,2024,forty,31,29,2,93.55
4,2024,vertical,31,30,1,96.77
5,2024,broad_jump,31,30,1,96.77
6,2025,forty,37,33,4,89.19
7,2025,vertical,37,32,5,86.49
8,2025,broad_jump,37,31,6,83.78



SAVED FILES
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8h_combine_feature_governance.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_approved_primary_features.parquet
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_approved_primary_features.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8h_primary_feature_missingness.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8h_primary_feature_cohort_coverage.csv

STEP 8H COMPLETED SUCCESSFULLY.


In [20]:

# =====================================================================
# NOTEBOOK 06 — STEP 8I
# INSPECT ACTUAL COMBINE FEATURE SCHEMA
# =====================================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

FEATURE_DIR = ROOT / "data/features"

files = {
    "final_player_features":
        FEATURE_DIR / "wr_combine_final_player_features.parquet",

    "player_drill_profiles":
        FEATURE_DIR / "wr_combine_final_player_drill_profiles.parquet",

    "original_player_profiles":
        FEATURE_DIR / "wr_combine_player_profiles.parquet"
}

for label, path in files.items():

    print("\n" + "=" * 75)
    print(label.upper())
    print("=" * 75)

    if not path.exists():
        print("FILE NOT FOUND:", path)
        continue

    data = pd.read_parquet(path)

    print("Shape:", data.shape)
    print("Unique players:",
          data["nfl_id"].nunique()
          if "nfl_id" in data.columns else "n/a")

    print("\nALL COLUMNS:")

    for i, col in enumerate(data.columns):
        print(f"{i:03d}. {col}")

    print("\nMOVEMENT-RELATED COLUMNS:")

    movement_terms = [
        "speed", "acceleration", "path", "displacement",
        "turn", "efficiency", "coverage", "drill",
        "direction", "coordinate"
    ]

    movement_columns = [
        col for col in data.columns
        if any(term in col.lower() for term in movement_terms)
    ]

    for col in movement_columns:
        print("-", col)

    if label == "final_player_features":

        print("\nMOVEMENT FEATURE SUMMARY:")

        available = []

        for col in movement_columns:

            if pd.api.types.is_numeric_dtype(data[col]):
                available.append({
                    "feature": col,
                    "non_missing": data[col].notna().sum(),
                    "missing_pct": data[col].isna().mean() * 100,
                    "unique_values": data[col].nunique()
                })

        if available:
            display(
                pd.DataFrame(available)
                .sort_values("feature")
                .round(2)
            )

print("\nSTEP 8I SCHEMA INSPECTION COMPLETED.")



FINAL_PLAYER_FEATURES
Shape: (107, 46)
Unique players: 107

ALL COLUMNS:
000. draft_year
001. nfl_id
002. mean_speed_median_across_drills
003. peak_speed_median_across_drills
004. p95_speed_median_across_drills
005. mean_acceleration_median_across_drills
006. peak_acceleration_median_across_drills
007. coordinate_path_length_median_across_drills
008. net_displacement_median_across_drills
009. path_efficiency_median_across_drills
010. coordinate_mean_turn_median_across_drills
011. coordinate_p95_turn_median_across_drills
012. turning_rate_median_across_drills
013. recorded_total_turn_median_across_drills
014. coordinate_total_turn_median_across_drills
015. turning_coverage_median_across_drills
016. mean_speed_iqr_across_drills
017. peak_speed_iqr_across_drills
018. p95_speed_iqr_across_drills
019. mean_acceleration_iqr_across_drills
020. peak_acceleration_iqr_across_drills
021. coordinate_path_length_iqr_across_drills
022. net_displacement_iqr_across_drills
023. path_efficiency_iqr_acr

,feature,non_missing,missing_pct,unique_values
22,coordinate_mean_turn_iqr_across_drills,107,0.0,107
8,coordinate_mean_turn_median_across_drills,107,0.0,107
23,coordinate_p95_turn_iqr_across_drills,107,0.0,107
9,coordinate_p95_turn_median_across_drills,107,0.0,107
19,coordinate_path_length_iqr_across_drills,107,0.0,107
5,coordinate_path_length_median_across_drills,107,0.0,107
26,coordinate_total_turn_iqr_across_drills,107,0.0,107
12,coordinate_total_turn_median_across_drills,107,0.0,107
30,drills_with_turning_coverage_below_70_pct,107,0.0,2
31,drills_with_turning_coverage_below_80_pct,107,0.0,3



PLAYER_DRILL_PROFILES
Shape: (1113, 18)
Unique players: 107

ALL COLUMNS:
000. draft_year
001. nfl_id
002. drill_name
003. mean_speed
004. peak_speed
005. p95_speed
006. mean_acceleration
007. peak_acceleration
008. coordinate_path_length
009. net_displacement
010. path_efficiency
011. coordinate_mean_turn
012. coordinate_p95_turn
013. turning_rate
014. recorded_total_turn
015. coordinate_total_turn
016. turning_coverage
017. attempts_in_drill

MOVEMENT-RELATED COLUMNS:
- drill_name
- mean_speed
- peak_speed
- p95_speed
- mean_acceleration
- peak_acceleration
- coordinate_path_length
- net_displacement
- path_efficiency
- coordinate_mean_turn
- coordinate_p95_turn
- turning_rate
- recorded_total_turn
- coordinate_total_turn
- turning_coverage
- attempts_in_drill

ORIGINAL_PLAYER_PROFILES
Shape: (107, 30)
Unique players: 107

ALL COLUMNS:
000. draft_year
001. nfl_id
002. primary_drills_observed
003. primary_attempts_observed
004. combine_mean_speed_median_across_drills
005. combine_mea

In [21]:

# =====================================================================
# NOTEBOOK 06 — STEP 8J
# CORRECTED COMBINE FEATURE GOVERNANCE
# =====================================================================

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

FEATURE_DIR = ROOT / "data/features"
AUDIT_DIR = ROOT / "reports/data_audit"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = FEATURE_DIR / "wr_combine_final_player_features.parquet"

df = pd.read_parquet(SOURCE)

print("=" * 75)
print("STEP 8J — CORRECTED FEATURE GOVERNANCE")
print("=" * 75)

print("Source:", SOURCE)
print("Rows:", len(df))
print("Unique players:", df["nfl_id"].nunique())

assert df["nfl_id"].nunique() == len(df)
assert set(df["draft_year"].unique()) == {2023, 2024, 2025}


# ---------------------------------------------------------------------
# 1. FEATURE REGISTER USING VERIFIED COLUMN NAMES
# ---------------------------------------------------------------------

specs = [

    # Traditional athletic tests
    ("forty", "Traditional athleticism", "PRIMARY"),
    ("vertical", "Traditional athleticism", "PRIMARY"),
    ("broad_jump", "Traditional athleticism", "PRIMARY"),
    ("ten_yd_split", "Traditional athleticism", "SECONDARY"),

    # Movement speed
    ("mean_speed_median_across_drills",
     "Movement speed", "PRIMARY"),

    ("peak_speed_median_across_drills",
     "Movement speed", "PRIMARY"),

    ("p95_speed_median_across_drills",
     "Movement speed", "SECONDARY"),

    # Acceleration
    ("mean_acceleration_median_across_drills",
     "Movement acceleration", "PRIMARY"),

    ("peak_acceleration_median_across_drills",
     "Movement acceleration", "SECONDARY"),

    # Movement efficiency
    ("path_efficiency_median_across_drills",
     "Movement efficiency", "PRIMARY"),

    # Distance
    ("coordinate_path_length_median_across_drills",
     "Movement distance", "SECONDARY"),

    ("net_displacement_median_across_drills",
     "Movement distance", "SECONDARY"),

    # NGS comparator variables
    ("ngs_athleticism_score", "NGS comparator", "COMPARATOR"),
    ("ngs_college_production_score", "NGS comparator", "COMPARATOR"),
    ("ngs_final_score", "NGS comparator", "COMPARATOR"),

    # Low-coverage Combine tests
    ("three_cone", "Low-coverage tests", "SENSITIVITY"),
    ("short_shuttle", "Low-coverage tests", "SENSITIVITY"),
    ("bench_reps", "Low-coverage tests", "SENSITIVITY"),

    # Exploratory turning variables
    ("recorded_total_turn_median_across_drills",
     "Turning mechanics", "EXPLORATORY"),

    ("coordinate_total_turn_median_across_drills",
     "Turning mechanics", "EXPLORATORY"),

    ("coordinate_mean_turn_median_across_drills",
     "Turning mechanics", "EXPLORATORY"),

    ("coordinate_p95_turn_median_across_drills",
     "Turning mechanics", "EXPLORATORY"),

    ("turning_rate_median_across_drills",
     "Turning mechanics", "EXPLORATORY"),

    # Feature quality / coverage descriptors
    ("primary_drills_observed",
     "Feature quality", "QUALITY_CONTROL"),

    ("median_turning_coverage",
     "Feature quality", "QUALITY_CONTROL")
]


# ---------------------------------------------------------------------
# 2. BUILD REGISTER
# ---------------------------------------------------------------------

register_rows = []

for feature, family, status in specs:

    exists = feature in df.columns

    if exists:

        values = pd.to_numeric(
            df[feature],
            errors="coerce"
        )

        unique_count = values.nunique(dropna=True)
        missing_pct = values.isna().mean() * 100
        finite_count = np.isfinite(
            values.dropna()
        ).sum()

        variable = unique_count > 1
        finite = finite_count == values.notna().sum()

    else:

        unique_count = 0
        missing_pct = np.nan
        variable = False
        finite = False

    register_rows.append({
        "feature": feature,
        "feature_family": family,
        "proposed_status": status,
        "available": exists,
        "missing_pct": missing_pct,
        "unique_values": unique_count,
        "variable": variable,
        "finite_values": finite
    })

register = pd.DataFrame(register_rows)


# ---------------------------------------------------------------------
# 3. APPROVE PRIMARY PREDICTORS
# ---------------------------------------------------------------------

primary = register[
    (register["proposed_status"] == "PRIMARY")
    & (register["available"])
    & (register["variable"])
    & (register["finite_values"])
].copy()

approved_features = primary["feature"].tolist()

expected_primary = [
    "forty",
    "vertical",
    "broad_jump",
    "mean_speed_median_across_drills",
    "peak_speed_median_across_drills",
    "mean_acceleration_median_across_drills",
    "path_efficiency_median_across_drills"
]

missing_expected = [
    col for col in expected_primary
    if col not in approved_features
]

if missing_expected:
    raise ValueError(
        f"Expected primary features unavailable: {missing_expected}"
    )

print("\nAPPROVED PRIMARY FEATURES:")
for feature in approved_features:
    print("-", feature)


# ---------------------------------------------------------------------
# 4. CREATE APPROVED PRIMARY DATASET
# ---------------------------------------------------------------------

approved = df[
    ["draft_year", "nfl_id"] + approved_features
].copy()

# Ensure numeric values
for col in approved_features:
    approved[col] = pd.to_numeric(
        approved[col],
        errors="coerce"
    )

# Preserve missing traditional measurements.
# Imputation will be performed inside model training folds later.

approved.to_parquet(
    FEATURE_DIR / "wr_combine_approved_primary_features.parquet",
    index=False
)

approved.to_csv(
    FEATURE_DIR / "wr_combine_approved_primary_features.csv",
    index=False
)


# ---------------------------------------------------------------------
# 5. CREATE SECONDARY AND COMPARATOR DATASET
# ---------------------------------------------------------------------

secondary_features = register.loc[
    register["proposed_status"].isin(
        ["SECONDARY", "COMPARATOR", "SENSITIVITY"]
    )
    & register["available"]
    & register["variable"],
    "feature"
].tolist()

secondary = df[
    ["draft_year", "nfl_id"] + secondary_features
].copy()

secondary.to_parquet(
    FEATURE_DIR / "wr_combine_secondary_comparator_features.parquet",
    index=False
)


# ---------------------------------------------------------------------
# 6. CREATE EXPLORATORY TURNING DATASET
# ---------------------------------------------------------------------

turning_features = register.loc[
    (register["proposed_status"] == "EXPLORATORY")
    & register["available"],
    "feature"
].tolist()

turning = df[
    ["draft_year", "nfl_id"] + turning_features
].copy()

turning.to_parquet(
    FEATURE_DIR / "wr_combine_exploratory_turning_features.parquet",
    index=False
)


# ---------------------------------------------------------------------
# 7. MISSINGNESS AND COHORT COVERAGE
# ---------------------------------------------------------------------

missingness = pd.DataFrame({
    "feature": approved_features,
    "missing_count": [
        approved[col].isna().sum()
        for col in approved_features
    ],
    "missing_pct": [
        approved[col].isna().mean() * 100
        for col in approved_features
    ]
})

cohort_rows = []

for year, group in approved.groupby("draft_year"):

    for feature in approved_features:

        cohort_rows.append({
            "draft_year": year,
            "feature": feature,
            "players": len(group),
            "non_missing": group[feature].notna().sum(),
            "missing": group[feature].isna().sum(),
            "coverage_pct": group[feature].notna().mean() * 100
        })

cohort_coverage = pd.DataFrame(cohort_rows)


# ---------------------------------------------------------------------
# 8. SAVE AUDIT FILES
# ---------------------------------------------------------------------

register.to_csv(
    AUDIT_DIR / "step8j_corrected_feature_register.csv",
    index=False
)

missingness.to_csv(
    AUDIT_DIR / "step8j_primary_feature_missingness.csv",
    index=False
)

cohort_coverage.to_csv(
    AUDIT_DIR / "step8j_primary_feature_cohort_coverage.csv",
    index=False
)


# ---------------------------------------------------------------------
# 9. VALIDATION
# ---------------------------------------------------------------------

assert len(approved) == 107
assert approved["nfl_id"].nunique() == 107
assert len(approved_features) == 7

movement_primary = [
    "mean_speed_median_across_drills",
    "peak_speed_median_across_drills",
    "mean_acceleration_median_across_drills",
    "path_efficiency_median_across_drills"
]

assert all(
    col in approved.columns
    for col in movement_primary
)

assert approved[movement_primary].notna().all().all()

print("\n" + "=" * 75)
print("STEP 8J — VALIDATION RESULTS")
print("=" * 75)

print("Approved dataset shape:", approved.shape)
print("Unique players:", approved["nfl_id"].nunique())
print("Approved primary features:", len(approved_features))

print("\nFEATURE REGISTER")
display(register.round(2))

print("\nPRIMARY FEATURE MISSINGNESS")
display(missingness.round(2))

print("\nCOHORT COVERAGE")
display(cohort_coverage.round(2))

print("\nDRAFT COHORT DISTRIBUTION")
display(
    approved.groupby("draft_year")["nfl_id"]
    .nunique()
    .rename("players")
    .reset_index()
)

print("\nOUTPUT VERIFICATION")

output_files = [
    FEATURE_DIR / "wr_combine_approved_primary_features.parquet",
    FEATURE_DIR / "wr_combine_approved_primary_features.csv",
    FEATURE_DIR / "wr_combine_secondary_comparator_features.parquet",
    FEATURE_DIR / "wr_combine_exploratory_turning_features.parquet",
    AUDIT_DIR / "step8j_corrected_feature_register.csv",
    AUDIT_DIR / "step8j_primary_feature_missingness.csv",
    AUDIT_DIR / "step8j_primary_feature_cohort_coverage.csv"
]

for path in output_files:
    print(
        "VERIFIED" if path.exists() else "MISSING",
        path
    )

assert all(path.exists() for path in output_files)

print("\nSTEP 8J COMPLETED SUCCESSFULLY.")


STEP 8J — CORRECTED FEATURE GOVERNANCE
Source: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_final_player_features.parquet
Rows: 107
Unique players: 107

APPROVED PRIMARY FEATURES:
- forty
- vertical
- broad_jump
- mean_speed_median_across_drills
- peak_speed_median_across_drills
- mean_acceleration_median_across_drills
- path_efficiency_median_across_drills

STEP 8J — VALIDATION RESULTS
Approved dataset shape: (107, 9)
Unique players: 107
Approved primary features: 7

FEATURE REGISTER


,feature,feature_family,proposed_status,available,missing_pct,unique_values,variable,finite_values
0,forty,Traditional athleticism,PRIMARY,True,9.35,32,True,True
1,vertical,Traditional athleticism,PRIMARY,True,10.28,26,True,True
2,broad_jump,Traditional athleticism,PRIMARY,True,11.21,21,True,True
3,ten_yd_split,Traditional athleticism,SECONDARY,True,9.35,14,True,True
4,mean_speed_median_across_drills,Movement speed,PRIMARY,True,0.00,107,True,True
5,peak_speed_median_across_drills,Movement speed,PRIMARY,True,0.00,101,True,True
6,p95_speed_median_across_drills,Movement speed,SECONDARY,True,0.00,107,True,True
7,mean_acceleration_median_across_drills,Movement acceleration,PRIMARY,True,0.00,107,True,True
8,peak_acceleration_median_across_drills,Movement acceleration,SECONDARY,True,0.00,107,True,True
9,path_efficiency_median_across_drills,Movement efficiency,PRIMARY,True,0.00,107,True,True



PRIMARY FEATURE MISSINGNESS


,feature,missing_count,missing_pct
0,forty,10,9.35
1,vertical,11,10.28
2,broad_jump,12,11.21
3,mean_speed_median_across_drills,0,0.00
4,peak_speed_median_across_drills,0,0.00
5,mean_acceleration_median_across_drills,0,0.00
6,path_efficiency_median_across_drills,0,0.00



COHORT COVERAGE


,draft_year,feature,players,non_missing,missing,coverage_pct
0,2023,forty,39,35,4,89.74
1,2023,vertical,39,34,5,87.18
2,2023,broad_jump,39,34,5,87.18
3,2023,mean_speed_median_across_drills,39,39,0,100.00
4,2023,peak_speed_median_across_drills,39,39,0,100.00
5,2023,mean_acceleration_median_across_drills,39,39,0,100.00
6,2023,path_efficiency_median_across_drills,39,39,0,100.00
7,2024,forty,31,29,2,93.55
8,2024,vertical,31,30,1,96.77
9,2024,broad_jump,31,30,1,96.77



DRAFT COHORT DISTRIBUTION


,draft_year,players
0,2023,39
1,2024,31
2,2025,37



OUTPUT VERIFICATION
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_approved_primary_features.parquet
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_approved_primary_features.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_secondary_comparator_features.parquet
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_exploratory_turning_features.parquet
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8j_corrected_feature_register.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8j_primary_feature_missingness.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step8j_primary_feature_cohort_coverage.csv

STEP 8J COMPLETED SUCCESSFULLY.


In [22]:

# =====================================================================
# NOTEBOOK 06 — STEP 9
# FINAL PLAYER-SEASON MODELLING DATASET
# =====================================================================

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

FEATURE_DIR = ROOT / "data/features"
AUDIT_DIR = ROOT / "reports/data_audit"
RESULT_DIR = ROOT / "reports/model_results"

for folder in [FEATURE_DIR, AUDIT_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

COMBINE_FILE = (
    FEATURE_DIR /
    "wr_combine_approved_primary_features.parquet"
)

INTEGRATED_FILE = (
    FEATURE_DIR /
    "wr_combine_game_integrated.parquet"
)

# ---------------------------------------------------------------------
# 1. LOAD INPUT DATA
# ---------------------------------------------------------------------

combine = pd.read_parquet(COMBINE_FILE)
integrated = pd.read_parquet(INTEGRATED_FILE)

print("=" * 75)
print("STEP 9 — FINAL MODELLING DATASET")
print("=" * 75)

print("\nCOMBINE DATA")
print("Shape:", combine.shape)
print("Unique players:", combine["nfl_id"].nunique())

print("\nINTEGRATED PLAYER-SEASON DATA")
print("Shape:", integrated.shape)
print("Unique players:", integrated["nfl_id"].nunique())

print("\nINTEGRATED COLUMNS")
for col in integrated.columns:
    print("-", col)


# ---------------------------------------------------------------------
# 2. DEFINE REQUIRED IDENTIFIERS, EXPOSURE AND OUTCOMES
# ---------------------------------------------------------------------

required_identifiers = [
    "draft_year",
    "nfl_id",
    "season",
    "receiver_plays"
]

outcomes = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_total_direction_change_raw_median",
    "game_path_efficiency_median",
    "game_separation_at_pass_forward_median"
]

missing_identifiers = [
    col for col in required_identifiers
    if col not in integrated.columns
]

missing_outcomes = [
    col for col in outcomes
    if col not in integrated.columns
]

if missing_identifiers:
    raise KeyError(
        f"Missing required integrated identifiers: {missing_identifiers}"
    )

if missing_outcomes:
    raise KeyError(
        f"Missing expected game outcomes: {missing_outcomes}"
    )

# Verify Combine feature columns
predictors = [
    col for col in combine.columns
    if col not in ["draft_year", "nfl_id"]
]

expected_predictors = [
    "forty",
    "vertical",
    "broad_jump",
    "mean_speed_median_across_drills",
    "peak_speed_median_across_drills",
    "mean_acceleration_median_across_drills",
    "path_efficiency_median_across_drills"
]

assert set(predictors) == set(expected_predictors), (
    "Unexpected primary predictor schema."
)


# ---------------------------------------------------------------------
# 3. VALIDATE PLAYER IDENTIFIERS
# ---------------------------------------------------------------------

assert not combine.duplicated(
    ["draft_year", "nfl_id"]
).any(), "Duplicate Combine player keys."

assert not integrated.duplicated(
    ["season", "nfl_id"]
).any(), "Duplicate integrated player-season keys."

combine["draft_year"] = pd.to_numeric(
    combine["draft_year"], errors="raise"
)

integrated["draft_year"] = pd.to_numeric(
    integrated["draft_year"], errors="raise"
)

integrated["season"] = pd.to_numeric(
    integrated["season"], errors="raise"
)

integrated["receiver_plays"] = pd.to_numeric(
    integrated["receiver_plays"], errors="coerce"
)

# No pre-draft game seasons should be included.
pre_draft = integrated[
    integrated["season"] < integrated["draft_year"]
]

print("\nPRE-DRAFT RECORDS:", len(pre_draft))

assert len(pre_draft) == 0, (
    "Pre-draft player-season records found."
)


# ---------------------------------------------------------------------
# 4. JOIN COMBINE PREDICTORS TO GAME OUTCOMES
# ---------------------------------------------------------------------

# Keep game-side metadata and outcomes.
optional_metadata = [
    "position",
    "games",
    "games_played",
    "game_count"
]

metadata = [
    col for col in optional_metadata
    if col in integrated.columns
]

game_columns = (
    required_identifiers
    + metadata
    + outcomes
)

game_data = integrated[game_columns].copy()

model_data = game_data.merge(
    combine,
    on=["draft_year", "nfl_id"],
    how="inner",
    validate="many_to_one",
    indicator=True
)

print("\nLINKAGE AUDIT")
print(model_data["_merge"].value_counts())

assert (model_data["_merge"] == "both").all()

model_data = model_data.drop(columns="_merge")

print("\nJOINED DATASET")
print("Rows:", len(model_data))
print("Unique players:", model_data["nfl_id"].nunique())
print("Draft cohorts:", sorted(model_data["draft_year"].unique()))
print("Game seasons:", sorted(model_data["season"].unique()))


# ---------------------------------------------------------------------
# 5. APPLY PRIMARY EXPOSURE RULE
# ---------------------------------------------------------------------

MIN_RECEIVER_PLAYS = 10

model_data["receiver_plays"] = pd.to_numeric(
    model_data["receiver_plays"],
    errors="coerce"
)

all_linked = model_data.copy()

primary = model_data[
    model_data["receiver_plays"] >= MIN_RECEIVER_PLAYS
].copy()

sensitivity_20 = model_data[
    model_data["receiver_plays"] >= 20
].copy()

print("\nEXPOSURE FILTER")
print("All linked player-seasons:", len(all_linked))
print("Primary sample, >=10 plays:", len(primary))
print("Sensitivity sample, >=20 plays:", len(sensitivity_20))

print("\nPRIMARY SAMPLE PLAYERS:",
      primary["nfl_id"].nunique())

print("SENSITIVITY SAMPLE PLAYERS:",
      sensitivity_20["nfl_id"].nunique())


# ---------------------------------------------------------------------
# 6. ADD RESEARCH DESIGN VARIABLES
# ---------------------------------------------------------------------

for frame in [all_linked, primary, sensitivity_20]:

    frame["years_since_draft"] = (
        frame["season"] - frame["draft_year"]
    )

    frame["exposure_band"] = pd.cut(
        frame["receiver_plays"],
        bins=[0, 9, 19, 49, 99, np.inf],
        labels=[
            "1-9",
            "10-19",
            "20-49",
            "50-99",
            "100+"
        ]
    )


# ---------------------------------------------------------------------
# 7. VALIDATE PLAYER-LEVEL GROUPING
# ---------------------------------------------------------------------

player_seasons = (
    primary.groupby("nfl_id")["season"]
    .nunique()
    .rename("seasons_in_sample")
    .reset_index()
)

print("\nPLAYER SEASON DISTRIBUTION")
display(
    player_seasons["seasons_in_sample"]
    .value_counts()
    .sort_index()
    .rename_axis("seasons_per_player")
    .reset_index(name="players")
)

print("\nPLAYER-SEASON SAMPLE BY DRAFT COHORT")
display(
    primary.groupby("draft_year")
    .agg(
        player_seasons=("nfl_id", "size"),
        unique_players=("nfl_id", "nunique"),
        median_receiver_plays=("receiver_plays", "median"),
        mean_receiver_plays=("receiver_plays", "mean")
    )
    .round(2)
    .reset_index()
)

# Repeated player records are expected.
# They must be grouped by nfl_id during cross-validation.
assert primary["nfl_id"].nunique() < len(primary)


# ---------------------------------------------------------------------
# 8. OUTCOME COMPLETENESS AUDIT
# ---------------------------------------------------------------------

outcome_audit = pd.DataFrame({
    "outcome": outcomes,
    "observations": [
        primary[col].notna().sum()
        for col in outcomes
    ],
    "missing": [
        primary[col].isna().sum()
        for col in outcomes
    ],
    "missing_pct": [
        primary[col].isna().mean() * 100
        for col in outcomes
    ],
    "unique_values": [
        primary[col].nunique(dropna=True)
        for col in outcomes
    ]
})

print("\nOUTCOME COMPLETENESS")
display(outcome_audit.round(2))

outcome_audit.to_csv(
    AUDIT_DIR / "step9_primary_outcome_completeness.csv",
    index=False
)


# ---------------------------------------------------------------------
# 9. SAVE MODELLING DATASETS
# ---------------------------------------------------------------------

outputs = {
    "all_linked":
        FEATURE_DIR / "wr_modeling_dataset_all_linked.parquet",

    "primary_10plus":
        FEATURE_DIR / "wr_modeling_dataset_primary_10plus.parquet",

    "sensitivity_20plus":
        FEATURE_DIR / "wr_modeling_dataset_sensitivity_20plus.parquet"
}

all_linked.to_parquet(outputs["all_linked"], index=False)
primary.to_parquet(outputs["primary_10plus"], index=False)
sensitivity_20.to_parquet(outputs["sensitivity_20plus"], index=False)

# CSV versions for inspection
all_linked.to_csv(
    FEATURE_DIR / "wr_modeling_dataset_all_linked.csv",
    index=False
)

primary.to_csv(
    FEATURE_DIR / "wr_modeling_dataset_primary_10plus.csv",
    index=False
)

sensitivity_20.to_csv(
    FEATURE_DIR / "wr_modeling_dataset_sensitivity_20plus.csv",
    index=False
)


# ---------------------------------------------------------------------
# 10. SAVE PLAYER-LEVEL VALIDATION GROUPS
# ---------------------------------------------------------------------

group_manifest = (
    primary[["nfl_id", "draft_year"]]
    .drop_duplicates()
    .sort_values(["draft_year", "nfl_id"])
)

group_manifest.to_csv(
    AUDIT_DIR / "step9_player_group_manifest.csv",
    index=False
)


# ---------------------------------------------------------------------
# 11. FINAL VALIDATION
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 9 FINAL VALIDATION")
print("=" * 75)

print("Primary rows:", len(primary))
print("Primary unique players:", primary["nfl_id"].nunique())
print("Predictors:", len(predictors))
print("Outcomes:", len(outcomes))
print("Exposure threshold:", MIN_RECEIVER_PLAYS)

print("\nPREDICTOR MISSINGNESS")
display(
    primary[predictors]
    .isna()
    .mean()
    .mul(100)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "predictor"})
    .round(2)
)

print("\nSEASON / DRAFT COHORT CROSS-TAB")
display(
    pd.crosstab(
        primary["draft_year"],
        primary["season"]
    )
)

print("\nSAVED OUTPUTS")
for name, path in outputs.items():
    print(
        name,
        "VERIFIED" if path.exists() else "MISSING",
        path
    )

manifest_path = AUDIT_DIR / "step9_player_group_manifest.csv"
print(
    "group_manifest",
    "VERIFIED" if manifest_path.exists() else "MISSING",
    manifest_path
)

assert all(path.exists() for path in outputs.values())
assert manifest_path.exists()

assert primary["nfl_id"].nunique() == 66
assert len(primary) == 116

print("\nSTEP 9 COMPLETED SUCCESSFULLY.")


STEP 9 — FINAL MODELLING DATASET

COMBINE DATA
Shape: (107, 9)
Unique players: 107

INTEGRATED PLAYER-SEASON DATA
Shape: (146, 70)
Unique players: 79

INTEGRATED COLUMNS
- draft_year
- nfl_id
- primary_drills_observed
- primary_attempts_observed
- combine_mean_speed_median_across_drills
- combine_mean_speed_iqr_across_drills
- combine_peak_speed_median_across_drills
- combine_peak_speed_iqr_across_drills
- combine_p95_speed_median_across_drills
- combine_p95_speed_iqr_across_drills
- combine_mean_acceleration_median_across_drills
- combine_mean_acceleration_iqr_across_drills
- combine_peak_acceleration_median_across_drills
- combine_peak_acceleration_iqr_across_drills
- combine_coordinate_path_length_median_across_drills
- combine_coordinate_path_length_iqr_across_drills
- combine_net_displacement_median_across_drills
- combine_net_displacement_iqr_across_drills
- combine_path_efficiency_median_across_drills
- combine_path_efficiency_iqr_across_drills
- combine_coordinate_total_turn_me

,seasons_per_player,players
0,1,30
1,2,22
2,3,14



PLAYER-SEASON SAMPLE BY DRAFT COHORT


,draft_year,player_seasons,unique_players,median_receiver_plays,mean_receiver_plays
0,2023,64,28,46.0,56.00
1,2024,35,21,52.0,52.29
2,2025,17,17,31.0,38.06



OUTCOME COMPLETENESS


,outcome,observations,missing,missing_pct,unique_values
0,game_mean_speed_raw_median,116,0,0.0,116
1,game_peak_speed_raw_median,116,0,0.0,99
2,game_mean_acceleration_raw_median,116,0,0.0,116
3,game_total_direction_change_raw_median,116,0,0.0,116
4,game_path_efficiency_median,116,0,0.0,116
5,game_separation_at_pass_forward_median,116,0,0.0,116



STEP 9 FINAL VALIDATION
Primary rows: 116
Primary unique players: 66
Predictors: 7
Outcomes: 6
Exposure threshold: 10

PREDICTOR MISSINGNESS


,predictor,missing_pct
0,forty,10.34
1,vertical,12.07
2,broad_jump,10.34
3,mean_speed_median_across_drills,0.00
4,peak_speed_median_across_drills,0.00
5,mean_acceleration_median_across_drills,0.00
6,path_efficiency_median_across_drills,0.00



SEASON / DRAFT COHORT CROSS-TAB


season,2023,2024,2025
draft_year,,,
2023,24,23,17
2024,0,18,17
2025,0,0,17



SAVED OUTPUTS
all_linked VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_modeling_dataset_all_linked.parquet
primary_10plus VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_modeling_dataset_primary_10plus.parquet
sensitivity_20plus VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_modeling_dataset_sensitivity_20plus.parquet
group_manifest VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step9_player_group_manifest.csv

STEP 9 COMPLETED SUCCESSFULLY.


In [23]:

# =====================================================================
# NOTEBOOK 06 — STEP 10A
# NESTED PLAYER-GROUPED RIDGE MODELLING
# =====================================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from scipy.stats import spearmanr

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

FEATURE_DIR = ROOT / "data/features"
RESULT_DIR = ROOT / "reports/model_results"
AUDIT_DIR = ROOT / "reports/data_audit"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# 1. LOAD PRIMARY MODELLING DATA
# ---------------------------------------------------------------------

DATA_FILE = (
    FEATURE_DIR /
    "wr_modeling_dataset_primary_10plus.parquet"
)

df = pd.read_parquet(DATA_FILE).reset_index(drop=True)

print("=" * 75)
print("STEP 10A — NESTED GROUPED RIDGE MODELLING")
print("=" * 75)

print("Rows:", len(df))
print("Unique players:", df["nfl_id"].nunique())

assert len(df) == 116
assert df["nfl_id"].nunique() == 66


# ---------------------------------------------------------------------
# 2. DEFINE PREDICTOR GROUPS
# ---------------------------------------------------------------------

MODEL_A = [
    "forty",
    "vertical",
    "broad_jump"
]

MODEL_B = [
    "mean_speed_median_across_drills",
    "peak_speed_median_across_drills",
    "mean_acceleration_median_across_drills",
    "path_efficiency_median_across_drills"
]

MODEL_C = MODEL_A + MODEL_B

MODELS = {
    "A_TRADITIONAL": MODEL_A,
    "B_TRACKING": MODEL_B,
    "C_COMBINED": MODEL_C
}

for name, features in MODELS.items():

    missing = [
        col for col in features
        if col not in df.columns
    ]

    if missing:
        raise KeyError(f"{name} missing predictors: {missing}")


# ---------------------------------------------------------------------
# 3. DEFINE OUTCOMES
# ---------------------------------------------------------------------

OUTCOMES = {
    "game_mean_speed_raw_median": "PRIMARY",
    "game_peak_speed_raw_median": "PRIMARY",
    "game_mean_acceleration_raw_median": "PRIMARY",
    "game_path_efficiency_median": "PRIMARY",
    "game_separation_at_pass_forward_median": "SECONDARY"
}

for outcome in OUTCOMES:

    if outcome not in df.columns:
        raise KeyError(f"Missing outcome: {outcome}")

    df[outcome] = pd.to_numeric(
        df[outcome],
        errors="coerce"
    )

    if df[outcome].isna().any():
        raise ValueError(
            f"Missing target values in {outcome}. "
            "Investigate before modelling."
        )


# ---------------------------------------------------------------------
# 4. DEFINE VALIDATION
# ---------------------------------------------------------------------

OUTER_SPLITS = 5
INNER_SPLITS = 4

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)

ALPHAS = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0
]

groups = df["nfl_id"].to_numpy()

assert len(np.unique(groups)) == 66

print("\nVALIDATION DESIGN")
print("Outer folds:", OUTER_SPLITS)
print("Inner folds:", INNER_SPLITS)
print("Grouping variable: nfl_id")
print("Ridge alpha candidates:", ALPHAS)


# ---------------------------------------------------------------------
# 5. BUILD MODEL PIPELINE
# ---------------------------------------------------------------------

def make_pipeline():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge()
        )
    ])


# ---------------------------------------------------------------------
# 6. METRIC FUNCTION
# ---------------------------------------------------------------------

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) < 2:
        return {
            "n": len(y_true),
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "spearman": np.nan
        }

    rho = spearmanr(y_true, y_pred).statistic

    return {
        "n": len(y_true),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "r2": r2_score(y_true, y_pred),
        "spearman": rho
    }


# ---------------------------------------------------------------------
# 7. OUTER FOLD AUDIT
# ---------------------------------------------------------------------

fold_audit = []

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(df, groups=groups),
    start=1
):

    train_players = set(
        df.iloc[train_idx]["nfl_id"]
    )

    test_players = set(
        df.iloc[test_idx]["nfl_id"]
    )

    overlap = train_players.intersection(test_players)

    assert len(overlap) == 0

    fold_audit.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_players": len(train_players),
        "test_players": len(test_players),
        "player_overlap": len(overlap)
    })

fold_audit = pd.DataFrame(fold_audit)

print("\nOUTER FOLD AUDIT")
display(fold_audit)

assert fold_audit["player_overlap"].sum() == 0


# ---------------------------------------------------------------------
# 8. RUN NESTED MODELLING
# ---------------------------------------------------------------------

prediction_records = []
alpha_records = []

for outcome, outcome_role in OUTCOMES.items():

    print("\n" + "=" * 75)
    print("OUTCOME:", outcome)
    print("ROLE:", outcome_role)
    print("=" * 75)

    y = df[outcome].to_numpy(dtype=float)

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(df, y, groups),
        start=1
    ):

        train = df.iloc[train_idx]
        test = df.iloc[test_idx]

        train_groups = train["nfl_id"].to_numpy()
        test_groups = test["nfl_id"].to_numpy()

        assert set(train_groups).isdisjoint(
            set(test_groups)
        )

        y_train = train[outcome].to_numpy(dtype=float)
        y_test = test[outcome].to_numpy(dtype=float)

        # Fold-specific training-mean baseline
        baseline_prediction = np.repeat(
            np.mean(y_train),
            len(test)
        )

        for i, row_index in enumerate(test_idx):

            prediction_records.append({
                "row_id": int(row_index),
                "outcome": outcome,
                "outcome_role": outcome_role,
                "model": "BASELINE",
                "fold": fold,
                "nfl_id": df.iloc[row_index]["nfl_id"],
                "draft_year": df.iloc[row_index]["draft_year"],
                "season": df.iloc[row_index]["season"],
                "actual": y_test[i],
                "predicted": baseline_prediction[i],
                "best_alpha": np.nan
            })

        # Model A, B and C
        for model_name, features in MODELS.items():

            X_train = train[features].copy()
            X_test = test[features].copy()

            pipeline = make_pipeline()

            inner_cv = GroupKFold(
                n_splits=INNER_SPLITS
            )

            search = GridSearchCV(
                estimator=pipeline,
                param_grid={
                    "ridge__alpha": ALPHAS
                },
                scoring="neg_mean_absolute_error",
                cv=inner_cv,
                n_jobs=-1,
                refit=True
            )

            # Groups are passed into inner CV.
            search.fit(
                X_train,
                y_train,
                groups=train_groups
            )

            predictions = search.predict(X_test)

            best_alpha = search.best_params_[
                "ridge__alpha"
            ]

            alpha_records.append({
                "outcome": outcome,
                "model": model_name,
                "fold": fold,
                "best_alpha": best_alpha,
                "inner_best_mae": -search.best_score_,
                "train_players": len(np.unique(train_groups)),
                "test_players": len(np.unique(test_groups))
            })

            for i, row_index in enumerate(test_idx):

                prediction_records.append({
                    "row_id": int(row_index),
                    "outcome": outcome,
                    "outcome_role": outcome_role,
                    "model": model_name,
                    "fold": fold,
                    "nfl_id": df.iloc[row_index]["nfl_id"],
                    "draft_year": df.iloc[row_index]["draft_year"],
                    "season": df.iloc[row_index]["season"],
                    "actual": y_test[i],
                    "predicted": predictions[i],
                    "best_alpha": best_alpha
                })

            print(
                f"Fold {fold} | {model_name} | "
                f"alpha={best_alpha} | "
                f"test players={len(np.unique(test_groups))}"
            )


# ---------------------------------------------------------------------
# 9. BUILD OUT-OF-FOLD PREDICTION DATASET
# ---------------------------------------------------------------------

predictions_df = pd.DataFrame(prediction_records)

alphas_df = pd.DataFrame(alpha_records)

assert not predictions_df.duplicated(
    ["row_id", "outcome", "model"]
).any()

expected_predictions = (
    len(df)
    * len(OUTCOMES)
    * (len(MODELS) + 1)
)

assert len(predictions_df) == expected_predictions

# Every player must be assigned to only one outer fold.
player_fold_counts = (
    predictions_df.groupby(
        ["outcome", "model", "nfl_id"]
    )["fold"]
    .nunique()
)

assert player_fold_counts.max() == 1


# ---------------------------------------------------------------------
# 10. PLAYER-SEASON METRICS
# ---------------------------------------------------------------------

season_metrics = []

for (outcome, model), group in predictions_df.groupby(
    ["outcome", "model"]
):

    metrics = calculate_metrics(
        group["actual"],
        group["predicted"]
    )

    season_metrics.append({
        "outcome": outcome,
        "outcome_role": group["outcome_role"].iloc[0],
        "model": model,
        "evaluation_level": "PLAYER_SEASON",
        "players": group["nfl_id"].nunique(),
        **metrics
    })

season_metrics = pd.DataFrame(season_metrics)


# ---------------------------------------------------------------------
# 11. PLAYER-AGGREGATED METRICS
# ---------------------------------------------------------------------

# Each player receives one average actual outcome and
# one average out-of-fold prediction across eligible seasons.

player_predictions = (
    predictions_df.groupby(
        ["outcome", "outcome_role", "model", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        seasons=("season", "nunique")
    )
)

player_metrics = []

for (outcome, model), group in player_predictions.groupby(
    ["outcome", "model"]
):

    metrics = calculate_metrics(
        group["actual"],
        group["predicted"]
    )

    player_metrics.append({
        "outcome": outcome,
        "outcome_role": group["outcome_role"].iloc[0],
        "model": model,
        "evaluation_level": "PLAYER_AGGREGATED",
        "players": group["nfl_id"].nunique(),
        **metrics
    })

player_metrics = pd.DataFrame(player_metrics)


# ---------------------------------------------------------------------
# 12. COMBINE METRICS
# ---------------------------------------------------------------------

metrics_df = pd.concat(
    [season_metrics, player_metrics],
    ignore_index=True
)

metrics_df = metrics_df.sort_values(
    ["outcome", "evaluation_level", "model"]
)


# ---------------------------------------------------------------------
# 13. BASELINE IMPROVEMENT
# ---------------------------------------------------------------------

improvement_rows = []

for outcome in OUTCOMES:

    for level in [
        "PLAYER_SEASON",
        "PLAYER_AGGREGATED"
    ]:

        subset = metrics_df[
            (metrics_df["outcome"] == outcome)
            & (metrics_df["evaluation_level"] == level)
        ]

        baseline_row = subset[
            subset["model"] == "BASELINE"
        ]

        if baseline_row.empty:
            continue

        baseline_mae = baseline_row["mae"].iloc[0]

        for model_name in MODELS:

            model_row = subset[
                subset["model"] == model_name
            ]

            if model_row.empty:
                continue

            model_mae = model_row["mae"].iloc[0]

            improvement = (
                (baseline_mae - model_mae)
                / baseline_mae * 100
                if baseline_mae > 0
                else np.nan
            )

            improvement_rows.append({
                "outcome": outcome,
                "evaluation_level": level,
                "model": model_name,
                "baseline_mae": baseline_mae,
                "model_mae": model_mae,
                "mae_improvement_pct": improvement
            })

improvement_df = pd.DataFrame(improvement_rows)


# ---------------------------------------------------------------------
# 14. SAVE RESULTS
# ---------------------------------------------------------------------

prediction_path = (
    RESULT_DIR / "step10a_grouped_oof_predictions.parquet"
)

metrics_path = (
    RESULT_DIR / "step10a_grouped_model_metrics.csv"
)

player_prediction_path = (
    RESULT_DIR / "step10a_player_aggregated_predictions.csv"
)

alpha_path = (
    RESULT_DIR / "step10a_nested_cv_selected_alphas.csv"
)

fold_path = (
    AUDIT_DIR / "step10a_grouped_cv_fold_audit.csv"
)

improvement_path = (
    RESULT_DIR / "step10a_baseline_mae_improvements.csv"
)

predictions_df.to_parquet(
    prediction_path,
    index=False
)

metrics_df.to_csv(
    metrics_path,
    index=False
)

player_predictions.to_csv(
    player_prediction_path,
    index=False
)

alphas_df.to_csv(
    alpha_path,
    index=False
)

fold_audit.to_csv(
    fold_path,
    index=False
)

improvement_df.to_csv(
    improvement_path,
    index=False
)


# ---------------------------------------------------------------------
# 15. DISPLAY RESULTS
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("PLAYER-SEASON MODEL PERFORMANCE")
print("=" * 75)

display(
    season_metrics.sort_values(
        ["outcome", "mae"]
    ).round(4)
)

print("\n" + "=" * 75)
print("PLAYER-AGGREGATED MODEL PERFORMANCE")
print("=" * 75)

display(
    player_metrics.sort_values(
        ["outcome", "mae"]
    ).round(4)
)

print("\n" + "=" * 75)
print("MAE IMPROVEMENT OVER TRAINING-MEAN BASELINE")
print("=" * 75)

display(
    improvement_df.round(2)
)

print("\n" + "=" * 75)
print("SELECTED RIDGE ALPHAS")
print("=" * 75)

display(
    alphas_df.groupby(
        ["outcome", "model"]
    )["best_alpha"]
    .agg(
        folds="count",
        median_alpha="median",
        minimum_alpha="min",
        maximum_alpha="max"
    )
    .reset_index()
)


# ---------------------------------------------------------------------
# 16. FINAL INTEGRITY CHECKS
# ---------------------------------------------------------------------

assert predictions_df["predicted"].notna().all()
assert predictions_df["actual"].notna().all()
assert len(alphas_df) == (
    len(OUTCOMES) * len(MODELS) * OUTER_SPLITS
)

output_files = [
    prediction_path,
    metrics_path,
    player_prediction_path,
    alpha_path,
    fold_path,
    improvement_path
]

print("\nOUTPUT VERIFICATION")

for path in output_files:
    print(
        "VERIFIED" if path.exists() else "MISSING",
        path
    )

assert all(path.exists() for path in output_files)

print("\nSTEP 10A COMPLETED SUCCESSFULLY.")
print("Out-of-fold prediction records:", len(predictions_df))
print("Player-aggregated prediction records:", len(player_predictions))
print("Nested model fits:", len(alphas_df))
print("Player leakage detected: NO")


STEP 10A — NESTED GROUPED RIDGE MODELLING
Rows: 116
Unique players: 66

VALIDATION DESIGN
Outer folds: 5
Inner folds: 4
Grouping variable: nfl_id
Ridge alpha candidates: [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

OUTER FOLD AUDIT


,fold,train_rows,test_rows,train_players,test_players,player_overlap
0,1,92,24,52,14,0
1,2,93,23,53,13,0
2,3,93,23,53,13,0
3,4,93,23,53,13,0
4,5,93,23,53,13,0



OUTCOME: game_mean_speed_raw_median
ROLE: PRIMARY
Fold 1 | A_TRADITIONAL | alpha=100.0 | test players=14
Fold 1 | B_TRACKING | alpha=10.0 | test players=14
Fold 1 | C_COMBINED | alpha=100.0 | test players=14
Fold 2 | A_TRADITIONAL | alpha=100.0 | test players=13
Fold 2 | B_TRACKING | alpha=100.0 | test players=13
Fold 2 | C_COMBINED | alpha=100.0 | test players=13
Fold 3 | A_TRADITIONAL | alpha=100.0 | test players=13
Fold 3 | B_TRACKING | alpha=1000.0 | test players=13
Fold 3 | C_COMBINED | alpha=1000.0 | test players=13
Fold 4 | A_TRADITIONAL | alpha=1000.0 | test players=13
Fold 4 | B_TRACKING | alpha=1000.0 | test players=13
Fold 4 | C_COMBINED | alpha=1000.0 | test players=13
Fold 5 | A_TRADITIONAL | alpha=10.0 | test players=13
Fold 5 | B_TRACKING | alpha=1000.0 | test players=13
Fold 5 | C_COMBINED | alpha=100.0 | test players=13

OUTCOME: game_peak_speed_raw_median
ROLE: PRIMARY
Fold 1 | A_TRADITIONAL | alpha=100.0 | test players=14
Fold 1 | B_TRACKING | alpha=100.0 | test pla

,outcome,outcome_role,model,evaluation_level,players,n,mae,rmse,r2,spearman
1,game_mean_acceleration_raw_median,PRIMARY,BASELINE,PLAYER_SEASON,66,116,0.1577,0.2040,-0.0098,-0.1058
3,game_mean_acceleration_raw_median,PRIMARY,C_COMBINED,PLAYER_SEASON,66,116,0.1581,0.2042,-0.0109,-0.1246
2,game_mean_acceleration_raw_median,PRIMARY,B_TRACKING,PLAYER_SEASON,66,116,0.1584,0.2048,-0.0176,-0.1336
0,game_mean_acceleration_raw_median,PRIMARY,A_TRADITIONAL,PLAYER_SEASON,66,116,0.1590,0.2034,-0.0035,-0.0464
5,game_mean_speed_raw_median,PRIMARY,BASELINE,PLAYER_SEASON,66,116,0.4210,0.5852,-0.0719,-0.2940
7,game_mean_speed_raw_median,PRIMARY,C_COMBINED,PLAYER_SEASON,66,116,0.4265,0.5761,-0.0386,-0.0159
6,game_mean_speed_raw_median,PRIMARY,B_TRACKING,PLAYER_SEASON,66,116,0.4272,0.5887,-0.0846,-0.1914
4,game_mean_speed_raw_median,PRIMARY,A_TRADITIONAL,PLAYER_SEASON,66,116,0.4391,0.5762,-0.0390,-0.0008
10,game_path_efficiency_median,PRIMARY,B_TRACKING,PLAYER_SEASON,66,116,0.0079,0.0108,0.0165,0.1877
11,game_path_efficiency_median,PRIMARY,C_COMBINED,PLAYER_SEASON,66,116,0.0080,0.0107,0.0207,0.1500



PLAYER-AGGREGATED MODEL PERFORMANCE


,outcome,outcome_role,model,evaluation_level,players,n,mae,rmse,r2,spearman
1,game_mean_acceleration_raw_median,PRIMARY,BASELINE,PLAYER_AGGREGATED,66,66,0.1432,0.1809,-0.0074,-0.0366
3,game_mean_acceleration_raw_median,PRIMARY,C_COMBINED,PLAYER_AGGREGATED,66,66,0.1435,0.1810,-0.0087,-0.0505
0,game_mean_acceleration_raw_median,PRIMARY,A_TRADITIONAL,PLAYER_AGGREGATED,66,66,0.1439,0.1805,-0.0026,0.0131
2,game_mean_acceleration_raw_median,PRIMARY,B_TRACKING,PLAYER_AGGREGATED,66,66,0.1448,0.1821,-0.0203,-0.0984
5,game_mean_speed_raw_median,PRIMARY,BASELINE,PLAYER_AGGREGATED,66,66,0.3744,0.5272,-0.0787,-0.2577
6,game_mean_speed_raw_median,PRIMARY,B_TRACKING,PLAYER_AGGREGATED,66,66,0.3768,0.5354,-0.1128,-0.1858
7,game_mean_speed_raw_median,PRIMARY,C_COMBINED,PLAYER_AGGREGATED,66,66,0.3803,0.5235,-0.0637,0.0052
4,game_mean_speed_raw_median,PRIMARY,A_TRADITIONAL,PLAYER_AGGREGATED,66,66,0.3920,0.5250,-0.0701,0.0275
10,game_path_efficiency_median,PRIMARY,B_TRACKING,PLAYER_AGGREGATED,66,66,0.0078,0.0100,0.0028,0.0972
11,game_path_efficiency_median,PRIMARY,C_COMBINED,PLAYER_AGGREGATED,66,66,0.0078,0.0098,0.0270,0.1540



MAE IMPROVEMENT OVER TRAINING-MEAN BASELINE


,outcome,evaluation_level,model,baseline_mae,model_mae,mae_improvement_pct
0,game_mean_speed_raw_median,PLAYER_SEASON,A_TRADITIONAL,0.42,0.44,-4.28
1,game_mean_speed_raw_median,PLAYER_SEASON,B_TRACKING,0.42,0.43,-1.47
2,game_mean_speed_raw_median,PLAYER_SEASON,C_COMBINED,0.42,0.43,-1.30
3,game_mean_speed_raw_median,PLAYER_AGGREGATED,A_TRADITIONAL,0.37,0.39,-4.71
4,game_mean_speed_raw_median,PLAYER_AGGREGATED,B_TRACKING,0.37,0.38,-0.64
5,game_mean_speed_raw_median,PLAYER_AGGREGATED,C_COMBINED,0.37,0.38,-1.57
6,game_peak_speed_raw_median,PLAYER_SEASON,A_TRADITIONAL,0.49,0.49,0.05
7,game_peak_speed_raw_median,PLAYER_SEASON,B_TRACKING,0.49,0.48,1.85
8,game_peak_speed_raw_median,PLAYER_SEASON,C_COMBINED,0.49,0.46,4.67
9,game_peak_speed_raw_median,PLAYER_AGGREGATED,A_TRADITIONAL,0.44,0.42,3.64



SELECTED RIDGE ALPHAS


,outcome,model,folds,median_alpha,minimum_alpha,maximum_alpha
0,game_mean_acceleration_raw_median,A_TRADITIONAL,5,1000.0,100.0,1000.0
1,game_mean_acceleration_raw_median,B_TRACKING,5,1000.0,100.0,1000.0
2,game_mean_acceleration_raw_median,C_COMBINED,5,1000.0,1000.0,1000.0
3,game_mean_speed_raw_median,A_TRADITIONAL,5,100.0,10.0,1000.0
4,game_mean_speed_raw_median,B_TRACKING,5,1000.0,10.0,1000.0
5,game_mean_speed_raw_median,C_COMBINED,5,100.0,100.0,1000.0
6,game_path_efficiency_median,A_TRADITIONAL,5,1000.0,100.0,1000.0
7,game_path_efficiency_median,B_TRACKING,5,100.0,100.0,1000.0
8,game_path_efficiency_median,C_COMBINED,5,100.0,100.0,1000.0
9,game_peak_speed_raw_median,A_TRADITIONAL,5,100.0,10.0,100.0



OUTPUT VERIFICATION
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10a_grouped_oof_predictions.parquet
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10a_grouped_model_metrics.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10a_player_aggregated_predictions.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10a_nested_cv_selected_alphas.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step10a_grouped_cv_fold_audit.csv
VERIFIED /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10a_baseline_mae_improvements.csv

STEP 10A COMPLETED SUCCESSFULLY.
Out-of-fold prediction records: 2320
Player-aggregated prediction records: 1320
Nested model fits: 75
Player leakage detected: NO


In [24]:

# =====================================================================
# NOTEBOOK 06 — STEP 10B
# LEAVE-ONE-DRAFT-COHORT-OUT VALIDATION
# =====================================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from scipy.stats import spearmanr

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
FEATURE_DIR = ROOT / "data/features"
RESULT_DIR = ROOT / "reports/model_results"
AUDIT_DIR = ROOT / "reports/data_audit"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------------

df = pd.read_parquet(
    FEATURE_DIR / "wr_modeling_dataset_primary_10plus.parquet"
).reset_index(drop=True)

assert len(df) == 116
assert df["nfl_id"].nunique() == 66

print("=" * 75)
print("STEP 10B — LEAVE-ONE-DRAFT-COHORT-OUT VALIDATION")
print("=" * 75)

print("Rows:", len(df))
print("Players:", df["nfl_id"].nunique())
print("Draft cohorts:", sorted(df["draft_year"].unique()))

# ---------------------------------------------------------------------
# 2. PREDICTOR GROUPS AND OUTCOMES
# ---------------------------------------------------------------------

MODEL_A = ["forty", "vertical", "broad_jump"]

MODEL_B = [
    "mean_speed_median_across_drills",
    "peak_speed_median_across_drills",
    "mean_acceleration_median_across_drills",
    "path_efficiency_median_across_drills"
]

MODEL_C = MODEL_A + MODEL_B

MODELS = {
    "A_TRADITIONAL": MODEL_A,
    "B_TRACKING": MODEL_B,
    "C_COMBINED": MODEL_C
}

OUTCOMES = {
    "game_mean_speed_raw_median": "PRIMARY",
    "game_peak_speed_raw_median": "PRIMARY",
    "game_mean_acceleration_raw_median": "PRIMARY",
    "game_path_efficiency_median": "PRIMARY",
    "game_separation_at_pass_forward_median": "SECONDARY"
}

for outcome in OUTCOMES:
    df[outcome] = pd.to_numeric(df[outcome], errors="coerce")
    assert df[outcome].notna().all(), f"Missing outcome: {outcome}"

ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# ---------------------------------------------------------------------
# 3. PIPELINE AND METRICS
# ---------------------------------------------------------------------

def make_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge())
    ])


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) < 2:
        return {
            "n": len(y_true),
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "spearman": np.nan
        }

    rho = spearmanr(y_true, y_pred).statistic

    return {
        "n": len(y_true),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
        "spearman": rho
    }


# ---------------------------------------------------------------------
# 4. VERIFY COHORT AND PLAYER STRUCTURE
# ---------------------------------------------------------------------

cohort_player_counts = (
    df.groupby("draft_year")["nfl_id"]
    .nunique()
    .reset_index(name="unique_players")
)

print("\nDRAFT COHORT PLAYER COUNTS")
display(cohort_player_counts)

# A player must belong to one draft cohort.
player_cohort_counts = (
    df.groupby("nfl_id")["draft_year"].nunique()
)

assert player_cohort_counts.max() == 1

# ---------------------------------------------------------------------
# 5. LEAVE-ONE-COHORT-OUT MODELLING
# ---------------------------------------------------------------------

prediction_records = []
alpha_records = []
fold_records = []

cohorts = sorted(df["draft_year"].unique())

for held_out_cohort in cohorts:

    print("\n" + "=" * 75)
    print("HELD-OUT DRAFT COHORT:", held_out_cohort)
    print("=" * 75)

    train = df[
        df["draft_year"] != held_out_cohort
    ].copy()

    test = df[
        df["draft_year"] == held_out_cohort
    ].copy()

    train_players = set(train["nfl_id"])
    test_players = set(test["nfl_id"])

    overlap = train_players.intersection(test_players)

    assert len(overlap) == 0

    n_train_players = train["nfl_id"].nunique()
    n_test_players = test["nfl_id"].nunique()

    assert n_train_players >= 4

    fold_records.append({
        "held_out_cohort": held_out_cohort,
        "train_rows": len(train),
        "test_rows": len(test),
        "train_players": n_train_players,
        "test_players": n_test_players,
        "player_overlap": len(overlap)
    })

    print("Training rows:", len(train))
    print("Testing rows:", len(test))
    print("Training players:", n_train_players)
    print("Testing players:", n_test_players)

    train_groups = train["nfl_id"].to_numpy()

    inner_cv = GroupKFold(n_splits=4)

    for outcome, outcome_role in OUTCOMES.items():

        y_train = train[outcome].to_numpy(dtype=float)
        y_test = test[outcome].to_numpy(dtype=float)

        # Training-only mean baseline
        baseline_prediction = np.repeat(
            np.mean(y_train),
            len(test)
        )

        for i, (_, row) in enumerate(test.iterrows()):

            prediction_records.append({
                "held_out_cohort": held_out_cohort,
                "outcome": outcome,
                "outcome_role": outcome_role,
                "model": "BASELINE",
                "nfl_id": row["nfl_id"],
                "season": row["season"],
                "actual": y_test[i],
                "predicted": baseline_prediction[i],
                "best_alpha": np.nan
            })

        # Fit A, B and C
        for model_name, features in MODELS.items():

            search = GridSearchCV(
                estimator=make_pipeline(),
                param_grid={
                    "ridge__alpha": ALPHAS
                },
                scoring="neg_mean_absolute_error",
                cv=inner_cv,
                n_jobs=-1,
                refit=True
            )

            search.fit(
                train[features],
                y_train,
                groups=train_groups
            )

            predictions = search.predict(test[features])

            best_alpha = search.best_params_["ridge__alpha"]

            alpha_records.append({
                "held_out_cohort": held_out_cohort,
                "outcome": outcome,
                "model": model_name,
                "best_alpha": best_alpha,
                "inner_best_mae": -search.best_score_,
                "train_players": n_train_players,
                "test_players": n_test_players
            })

            for i, (_, row) in enumerate(test.iterrows()):

                prediction_records.append({
                    "held_out_cohort": held_out_cohort,
                    "outcome": outcome,
                    "outcome_role": outcome_role,
                    "model": model_name,
                    "nfl_id": row["nfl_id"],
                    "season": row["season"],
                    "actual": y_test[i],
                    "predicted": predictions[i],
                    "best_alpha": best_alpha
                })

            print(
                outcome,
                "|",
                model_name,
                "| alpha:",
                best_alpha
            )


# ---------------------------------------------------------------------
# 6. PREDICTION DATASET
# ---------------------------------------------------------------------

predictions_df = pd.DataFrame(prediction_records)
alphas_df = pd.DataFrame(alpha_records)
fold_audit = pd.DataFrame(fold_records)

assert not predictions_df.duplicated(
    ["held_out_cohort", "outcome", "model", "nfl_id", "season"]
).any()

assert fold_audit["player_overlap"].sum() == 0

# ---------------------------------------------------------------------
# 7. PLAYER-SEASON METRICS BY HELD-OUT COHORT
# ---------------------------------------------------------------------

cohort_metrics = []

for (cohort, outcome, model), group in predictions_df.groupby(
    ["held_out_cohort", "outcome", "model"]
):

    metrics = calculate_metrics(
        group["actual"],
        group["predicted"]
    )

    cohort_metrics.append({
        "held_out_cohort": cohort,
        "outcome": outcome,
        "outcome_role": group["outcome_role"].iloc[0],
        "model": model,
        "evaluation_level": "PLAYER_SEASON",
        "players": group["nfl_id"].nunique(),
        **metrics
    })

cohort_metrics = pd.DataFrame(cohort_metrics)


# ---------------------------------------------------------------------
# 8. PLAYER-AGGREGATED METRICS BY COHORT
# ---------------------------------------------------------------------

player_predictions = (
    predictions_df.groupby(
        ["held_out_cohort", "outcome", "outcome_role", "model", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        seasons=("season", "nunique")
    )
)

player_metrics = []

for (cohort, outcome, model), group in player_predictions.groupby(
    ["held_out_cohort", "outcome", "model"]
):

    metrics = calculate_metrics(
        group["actual"],
        group["predicted"]
    )

    player_metrics.append({
        "held_out_cohort": cohort,
        "outcome": outcome,
        "outcome_role": group["outcome_role"].iloc[0],
        "model": model,
        "evaluation_level": "PLAYER_AGGREGATED",
        "players": group["nfl_id"].nunique(),
        **metrics
    })

player_metrics = pd.DataFrame(player_metrics)


# ---------------------------------------------------------------------
# 9. MAE IMPROVEMENT OVER BASELINE
# ---------------------------------------------------------------------

improvement_records = []

for (cohort, outcome, level), subset in pd.concat(
    [cohort_metrics, player_metrics],
    ignore_index=True
).groupby(["held_out_cohort", "outcome", "evaluation_level"]):

    baseline = subset[
        subset["model"] == "BASELINE"
    ]

    if baseline.empty:
        continue

    baseline_mae = baseline["mae"].iloc[0]

    for model_name in MODELS:

        model_row = subset[
            subset["model"] == model_name
        ]

        if model_row.empty:
            continue

        model_mae = model_row["mae"].iloc[0]

        improvement = (
            100 * (baseline_mae - model_mae) / baseline_mae
            if baseline_mae > 0 else np.nan
        )

        improvement_records.append({
            "held_out_cohort": cohort,
            "outcome": outcome,
            "evaluation_level": level,
            "model": model_name,
            "baseline_mae": baseline_mae,
            "model_mae": model_mae,
            "mae_improvement_pct": improvement
        })

improvement_df = pd.DataFrame(improvement_records)


# ---------------------------------------------------------------------
# 10. SAVE OUTPUTS
# ---------------------------------------------------------------------

outputs = {
    "predictions": (
        RESULT_DIR / "step10b_cohort_holdout_predictions.parquet"
    ),
    "cohort_metrics": (
        RESULT_DIR / "step10b_cohort_holdout_metrics.csv"
    ),
    "player_predictions": (
        RESULT_DIR / "step10b_cohort_player_predictions.csv"
    ),
    "player_metrics": (
        RESULT_DIR / "step10b_cohort_player_metrics.csv"
    ),
    "alphas": (
        RESULT_DIR / "step10b_cohort_selected_alphas.csv"
    ),
    "improvements": (
        RESULT_DIR / "step10b_cohort_baseline_improvements.csv"
    ),
    "fold_audit": (
        AUDIT_DIR / "step10b_cohort_holdout_fold_audit.csv"
    )
}

predictions_df.to_parquet(outputs["predictions"], index=False)
cohort_metrics.to_csv(outputs["cohort_metrics"], index=False)
player_predictions.to_csv(outputs["player_predictions"], index=False)
player_metrics.to_csv(outputs["player_metrics"], index=False)
alphas_df.to_csv(outputs["alphas"], index=False)
improvement_df.to_csv(outputs["improvements"], index=False)
fold_audit.to_csv(outputs["fold_audit"], index=False)


# ---------------------------------------------------------------------
# 11. DISPLAY
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("COHORT HOLDOUT FOLD AUDIT")
print("=" * 75)
display(fold_audit)

print("\n" + "=" * 75)
print("PLAYER-SEASON COHORT PERFORMANCE")
print("=" * 75)
display(
    cohort_metrics.sort_values(
        ["outcome", "held_out_cohort", "mae"]
    ).round(4)
)

print("\n" + "=" * 75)
print("PLAYER-AGGREGATED COHORT PERFORMANCE")
print("=" * 75)
display(
    player_metrics.sort_values(
        ["outcome", "held_out_cohort", "mae"]
    ).round(4)
)

print("\n" + "=" * 75)
print("MAE IMPROVEMENT OVER BASELINE")
print("=" * 75)
display(improvement_df.round(2))

print("\n" + "=" * 75)
print("SELECTED ALPHAS")
print("=" * 75)
display(
    alphas_df.groupby(
        ["held_out_cohort", "outcome", "model"]
    )["best_alpha"]
    .agg(
        median_alpha="median",
        minimum_alpha="min",
        maximum_alpha="max"
    )
    .reset_index()
)

# ---------------------------------------------------------------------
# 12. FINAL VALIDATION
# ---------------------------------------------------------------------

assert len(fold_audit) == 3
assert fold_audit["player_overlap"].sum() == 0
assert predictions_df["actual"].notna().all()
assert predictions_df["predicted"].notna().all()
assert all(path.exists() for path in outputs.values())

print("\nOUTPUT VERIFICATION")

for name, path in outputs.items():
    print("VERIFIED", name, path)

print("\nSTEP 10B COMPLETED SUCCESSFULLY.")


STEP 10B — LEAVE-ONE-DRAFT-COHORT-OUT VALIDATION
Rows: 116
Players: 66
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]

DRAFT COHORT PLAYER COUNTS


,draft_year,unique_players
0,2023,28
1,2024,21
2,2025,17



HELD-OUT DRAFT COHORT: 2023
Training rows: 52
Testing rows: 64
Training players: 38
Testing players: 28
game_mean_speed_raw_median | A_TRADITIONAL | alpha: 1000.0
game_mean_speed_raw_median | B_TRACKING | alpha: 1000.0
game_mean_speed_raw_median | C_COMBINED | alpha: 1000.0
game_peak_speed_raw_median | A_TRADITIONAL | alpha: 100.0
game_peak_speed_raw_median | B_TRACKING | alpha: 1000.0
game_peak_speed_raw_median | C_COMBINED | alpha: 1000.0
game_mean_acceleration_raw_median | A_TRADITIONAL | alpha: 100.0
game_mean_acceleration_raw_median | B_TRACKING | alpha: 1000.0
game_mean_acceleration_raw_median | C_COMBINED | alpha: 1000.0
game_path_efficiency_median | A_TRADITIONAL | alpha: 10.0
game_path_efficiency_median | B_TRACKING | alpha: 1000.0
game_path_efficiency_median | C_COMBINED | alpha: 10.0
game_separation_at_pass_forward_median | A_TRADITIONAL | alpha: 1000.0
game_separation_at_pass_forward_median | B_TRACKING | alpha: 1000.0
game_separation_at_pass_forward_median | C_COMBINED | 

,held_out_cohort,train_rows,test_rows,train_players,test_players,player_overlap
0,2023,52,64,38,28,0
1,2024,81,35,45,21,0
2,2025,99,17,49,17,0



PLAYER-SEASON COHORT PERFORMANCE


,held_out_cohort,outcome,outcome_role,model,evaluation_level,players,n,mae,rmse,r2,spearman
1,2023,game_mean_acceleration_raw_median,PRIMARY,BASELINE,PLAYER_SEASON,28,64,0.1861,0.2373,-0.0350,NaN
2,2023,game_mean_acceleration_raw_median,PRIMARY,B_TRACKING,PLAYER_SEASON,28,64,0.1862,0.2373,-0.0350,-0.0332
3,2023,game_mean_acceleration_raw_median,PRIMARY,C_COMBINED,PLAYER_SEASON,28,64,0.1866,0.2376,-0.0372,-0.0995
0,2023,game_mean_acceleration_raw_median,PRIMARY,A_TRADITIONAL,PLAYER_SEASON,28,64,0.1894,0.2390,-0.0500,-0.0488
21,2024,game_mean_acceleration_raw_median,PRIMARY,BASELINE,PLAYER_SEASON,21,35,0.1137,0.1442,-0.0038,NaN
22,2024,game_mean_acceleration_raw_median,PRIMARY,B_TRACKING,PLAYER_SEASON,21,35,0.1138,0.1440,-0.0016,0.1166
20,2024,game_mean_acceleration_raw_median,PRIMARY,A_TRADITIONAL,PLAYER_SEASON,21,35,0.1143,0.1444,-0.0074,0.0569
23,2024,game_mean_acceleration_raw_median,PRIMARY,C_COMBINED,PLAYER_SEASON,21,35,0.1144,0.1443,-0.0052,0.0210
40,2025,game_mean_acceleration_raw_median,PRIMARY,A_TRADITIONAL,PLAYER_SEASON,17,17,0.1605,0.1865,-0.3977,0.0711
43,2025,game_mean_acceleration_raw_median,PRIMARY,C_COMBINED,PLAYER_SEASON,17,17,0.1609,0.1868,-0.4030,0.1250


... (3 outputs left out)

,held_out_cohort,outcome,evaluation_level,model,baseline_mae,model_mae,mae_improvement_pct
0,2023,game_mean_acceleration_raw_median,PLAYER_AGGREGATED,A_TRADITIONAL,0.17,0.17,-2.08
1,2023,game_mean_acceleration_raw_median,PLAYER_AGGREGATED,B_TRACKING,0.17,0.17,-0.24
2,2023,game_mean_acceleration_raw_median,PLAYER_AGGREGATED,C_COMBINED,0.17,0.17,-0.61
3,2023,game_mean_acceleration_raw_median,PLAYER_SEASON,A_TRADITIONAL,0.19,0.19,-1.74
4,2023,game_mean_acceleration_raw_median,PLAYER_SEASON,B_TRACKING,0.19,0.19,-0.04
...,...,...,...,...,...,...,...
85,2025,game_separation_at_pass_forward_median,PLAYER_AGGREGATED,B_TRACKING,0.67,0.68,-2.30
86,2025,game_separation_at_pass_forward_median,PLAYER_AGGREGATED,C_COMBINED,0.67,0.67,-0.20
87,2025,game_separation_at_pass_forward_median,PLAYER_SEASON,A_TRADITIONAL,0.67,0.67,0.34
88,2025,game_separation_at_pass_forward_median,PLAYER_SEASON,B_TRACKING,0.67,0.68,-2.30



SELECTED ALPHAS


,held_out_cohort,outcome,model,median_alpha,minimum_alpha,maximum_alpha
0,2023,game_mean_acceleration_raw_median,A_TRADITIONAL,100.0,100.0,100.0
1,2023,game_mean_acceleration_raw_median,B_TRACKING,1000.0,1000.0,1000.0
2,2023,game_mean_acceleration_raw_median,C_COMBINED,1000.0,1000.0,1000.0
3,2023,game_mean_speed_raw_median,A_TRADITIONAL,1000.0,1000.0,1000.0
4,2023,game_mean_speed_raw_median,B_TRACKING,1000.0,1000.0,1000.0
5,2023,game_mean_speed_raw_median,C_COMBINED,1000.0,1000.0,1000.0
6,2023,game_path_efficiency_median,A_TRADITIONAL,10.0,10.0,10.0
7,2023,game_path_efficiency_median,B_TRACKING,1000.0,1000.0,1000.0
8,2023,game_path_efficiency_median,C_COMBINED,10.0,10.0,10.0
9,2023,game_peak_speed_raw_median,A_TRADITIONAL,100.0,100.0,100.0



OUTPUT VERIFICATION
VERIFIED predictions /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10b_cohort_holdout_predictions.parquet
VERIFIED cohort_metrics /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10b_cohort_holdout_metrics.csv
VERIFIED player_predictions /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10b_cohort_player_predictions.csv
VERIFIED player_metrics /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10b_cohort_player_metrics.csv
VERIFIED alphas /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10b_cohort_selected_alphas.csv
VERIFIED improvements /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10b_cohort_baseline_improvements.csv
VERIFIED fold_audit /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/data_audit/step10b_cohort_holdout_fold_audit.csv

STEP 10B COMPLETED SUCCESSFULLY.


In [25]:

# ============================================================================
# STEP 10C — MODEL STABILITY AND UNCERTAINTY AUDIT: INPUT INSPECTION
# ============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
MODEL_DIR = ROOT / "reports" / "model_results"

FILES = {
    "10A_predictions": MODEL_DIR / "step10a_grouped_oof_predictions.parquet",
    "10A_metrics": MODEL_DIR / "step10a_grouped_model_metrics.csv",
    "10A_alphas": MODEL_DIR / "step10a_nested_cv_selected_alphas.csv",
    "10B_predictions": MODEL_DIR / "step10b_cohort_holdout_predictions.parquet",
    "10B_metrics": MODEL_DIR / "step10b_cohort_holdout_metrics.csv",
    "10B_player_predictions": MODEL_DIR / "step10b_cohort_player_predictions.csv",
    "10B_player_metrics": MODEL_DIR / "step10b_cohort_player_metrics.csv",
    "10B_alphas": MODEL_DIR / "step10b_cohort_selected_alphas.csv",
}

loaded = {}

print("=" * 85)
print("STEP 10C — INPUT SCHEMA AND MODEL STABILITY AUDIT")
print("=" * 85)

for name, path in FILES.items():

    if not path.exists():
        print(f"\nMISSING: {name} -> {path}")
        continue

    if path.suffix == ".parquet":
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)

    loaded[name] = df

    print("\n" + "-" * 85)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nSample:")
    print(df.head(3).to_string(index=False))

# --------------------------------------------------------------------------
# RIDGE ALPHA DISTRIBUTION
# --------------------------------------------------------------------------

for name in ["10A_alphas", "10B_alphas"]:

    if name not in loaded:
        continue

    df = loaded[name]

    print("\n" + "=" * 85)
    print(f"ALPHA DISTRIBUTION — {name}")
    print("=" * 85)

    alpha_col = next(
        (c for c in df.columns if "median_alpha" in c),
        None
    )

    if alpha_col is None:
        alpha_col = next(
            (c for c in df.columns if c.lower() == "alpha"),
            None
        )

    if alpha_col is not None:
        print(df[alpha_col].value_counts(dropna=False).sort_index())
        print("\nMaximum-alpha selections:",
              (df[alpha_col] == df[alpha_col].max()).sum())
        print("Maximum alpha:", df[alpha_col].max())

# --------------------------------------------------------------------------
# PLAYER-LEVEL PREDICTION COVERAGE
# --------------------------------------------------------------------------

for name in ["10A_predictions", "10B_player_predictions"]:

    if name not in loaded:
        continue

    df = loaded[name]

    print("\n" + "=" * 85)
    print(f"PREDICTION COVERAGE — {name}")
    print("=" * 85)

    for col in ["nfl_id", "model", "outcome", "held_out_cohort"]:
        if col in df.columns:
            print(f"{col}: {df[col].nunique(dropna=False)} unique values")

    print("Rows:", len(df))
    print("Missing values by column:")
    print(df.isna().sum().to_string())

print("\n" + "=" * 85)
print("STEP 10C INPUT INSPECTION COMPLETED")
print("No modelling outputs were modified.")
print("=" * 85)


STEP 10C — INPUT SCHEMA AND MODEL STABILITY AUDIT

-------------------------------------------------------------------------------------
10A_predictions
Shape: (2320, 11)
Columns: ['row_id', 'outcome', 'outcome_role', 'model', 'fold', 'nfl_id', 'draft_year', 'season', 'actual', 'predicted', 'best_alpha']

Sample:
 row_id                    outcome outcome_role    model  fold  nfl_id  draft_year  season   actual  predicted  best_alpha
      3 game_mean_speed_raw_median      PRIMARY BASELINE     1   55885        2023    2023 4.124947   4.199494         NaN
      4 game_mean_speed_raw_median      PRIMARY BASELINE     1   55885        2023    2024 3.940435   4.199494         NaN
      5 game_mean_speed_raw_median      PRIMARY BASELINE     1   55885        2023    2025 4.123055   4.199494         NaN

-------------------------------------------------------------------------------------
10A_metrics
Shape: (40, 10)
Columns: ['outcome', 'outcome_role', 'model', 'evaluation_level', 'players', '

In [26]:

# ============================================================================
# STEP 10C — RIDGE ALPHA AUDIT + PAIRED PLAYER-LEVEL BOOTSTRAP
# NFL Big Data Bowl 2027 | WR-MovementIQ
# ============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
MODEL_DIR = ROOT / "reports" / "model_results"
AUDIT_DIR = ROOT / "reports" / "data_audit"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAPS = 5000

rng = np.random.default_rng(RANDOM_SEED)

print("=" * 85)
print("STEP 10C — RIDGE ALPHA AND PAIRED PLAYER BOOTSTRAP")
print("=" * 85)

# ============================================================================
# 1. CORRECTED ALPHA AUDIT
# ============================================================================

alpha_files = {
    "10A": MODEL_DIR / "step10a_nested_cv_selected_alphas.csv",
    "10B": MODEL_DIR / "step10b_cohort_selected_alphas.csv"
}

alpha_audits = []

for stage, path in alpha_files.items():

    print(f"\n{'=' * 75}")
    print(f"RIDGE ALPHA AUDIT — {stage}")
    print("=" * 75)

    if not path.exists():
        print("FILE NOT FOUND:", path)
        continue

    alpha_df = pd.read_csv(path)

    print("Shape:", alpha_df.shape)
    print("Columns:", alpha_df.columns.tolist())

    alpha_col = next(
        (
            c for c in [
                "best_alpha",
                "median_alpha",
                "alpha"
            ]
            if c in alpha_df.columns
        ),
        None
    )

    if alpha_col is None:
        print("No recognized alpha column found.")
        continue

    print("\nAlpha frequency:")
    print(alpha_df[alpha_col].value_counts(dropna=False).sort_index())

    print("\nAlpha by model:")
    if "model" in alpha_df.columns:

        alpha_summary = (
            alpha_df.groupby("model")[alpha_col]
            .agg(
                records="count",
                median_alpha="median",
                minimum_alpha="min",
                maximum_alpha="max"
            )
            .reset_index()
        )

        print(alpha_summary.to_string(index=False))

        alpha_summary["stage"] = stage
        alpha_audits.append(alpha_summary)

    max_alpha = alpha_df[alpha_col].max()

    print("\nMaximum selected alpha:", max_alpha)
    print(
        "Selections at maximum:",
        int((alpha_df[alpha_col] == max_alpha).sum())
    )

if alpha_audits:
    alpha_audit_all = pd.concat(alpha_audits, ignore_index=True)

    alpha_audit_all.to_csv(
        AUDIT_DIR / "step10c_ridge_alpha_audit.csv",
        index=False
    )

# ============================================================================
# 2. VERIFY STEP 10A MISSING ALPHA VALUES BY MODEL
# ============================================================================

path_10a = MODEL_DIR / "step10a_grouped_oof_predictions.parquet"

print("\n" + "=" * 85)
print("STEP 10A — ALPHA MISSINGNESS BY MODEL")
print("=" * 85)

if path_10a.exists():

    pred_10a_oof = pd.read_parquet(path_10a)

    if "best_alpha" in pred_10a_oof.columns:

        missing_alpha = (
            pred_10a_oof.groupby("model")
            .agg(
                rows=("best_alpha", "size"),
                missing_alpha=("best_alpha", lambda x: x.isna().sum()),
                available_alpha=("best_alpha", lambda x: x.notna().sum())
            )
            .reset_index()
        )

        print(missing_alpha.to_string(index=False))

        missing_alpha.to_csv(
            AUDIT_DIR / "step10c_alpha_missingness_by_model.csv",
            index=False
        )

# ============================================================================
# 3. LOAD PLAYER-AGGREGATED PREDICTIONS
# ============================================================================

prediction_files = {
    "10A_GROUPED": MODEL_DIR / "step10a_player_aggregated_predictions.csv",
    "10B_COHORT": MODEL_DIR / "step10b_cohort_player_predictions.csv"
}

prediction_data = {}

for stage, path in prediction_files.items():

    print("\n" + "=" * 85)
    print("LOADING:", stage)
    print("=" * 85)

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    required = [
        "nfl_id",
        "outcome",
        "model",
        "actual",
        "predicted"
    ]

    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"{stage} missing required columns: {missing}")

    print("Models:", sorted(df["model"].dropna().unique()))
    print("Outcomes:", sorted(df["outcome"].dropna().unique()))
    print("Unique players:", df["nfl_id"].nunique())

    prediction_data[stage] = df

# ============================================================================
# 4. PAIRED PLAYER-LEVEL BOOTSTRAP FUNCTION
# ============================================================================

def paired_player_bootstrap(
    df,
    stage,
    group_name,
    group_value,
    n_bootstraps=5000
):

    """
    Resample independent players with replacement.

    Improvement (%) =
        (Baseline MAE - Model MAE) / Baseline MAE * 100

    Positive improvement means the model beats the baseline.
    """

    results = []

    for outcome in sorted(df["outcome"].dropna().unique()):

        subset = df[df["outcome"] == outcome].copy()

        # Keep only models needed for paired comparison.
        subset = subset[
            subset["model"].isin(
                ["BASELINE", "A_TRADITIONAL", "B_TRACKING", "C_COMBINED"]
            )
        ].copy()

        # Ensure one prediction per player/model/outcome.
        duplicate_check = subset.duplicated(
            ["nfl_id", "model"],
            keep=False
        )

        if duplicate_check.any():
            raise ValueError(
                f"Duplicate player/model rows in {stage}, "
                f"{outcome}. Resolve duplicates before bootstrapping."
            )

        # Pivot errors and actual values to a player-wide structure.
        subset["absolute_error"] = (
            subset["actual"] - subset["predicted"]
        ).abs()

        error_wide = subset.pivot(
            index="nfl_id",
            columns="model",
            values="absolute_error"
        )

        actual_wide = subset.pivot(
            index="nfl_id",
            columns="model",
            values="actual"
        )

        if "BASELINE" not in error_wide.columns:
            continue

        available_models = [
            m for m in [
                "A_TRADITIONAL",
                "B_TRACKING",
                "C_COMBINED"
            ]
            if m in error_wide.columns
        ]

        for model in available_models:

            paired = pd.DataFrame({
                "baseline_error": error_wide["BASELINE"],
                "model_error": error_wide[model]
            }).dropna()

            if len(paired) < 5:
                continue

            # Confirm both models are evaluated against the same actual
            # player outcome values.
            shared_ids = paired.index

            model_actual = actual_wide.loc[shared_ids, model]
            baseline_actual = actual_wide.loc[shared_ids, "BASELINE"]

            if not np.allclose(
                model_actual.to_numpy(dtype=float),
                baseline_actual.to_numpy(dtype=float),
                equal_nan=True
            ):
                raise ValueError(
                    f"Actual outcomes differ between baseline and {model}: "
                    f"{stage}, {outcome}"
                )

            baseline_errors = paired["baseline_error"].to_numpy()
            model_errors = paired["model_error"].to_numpy()

            n_players = len(paired)

            baseline_mae = baseline_errors.mean()
            model_mae = model_errors.mean()

            if baseline_mae == 0:
                continue

            improvement = (
                (baseline_mae - model_mae) / baseline_mae
            ) * 100

            # Paired bootstrap: sample player indices once and use the
            # same resampled players for both models.
            boot_improvements = []

            for _ in range(n_bootstraps):

                sample_idx = rng.integers(
                    0,
                    n_players,
                    size=n_players
                )

                b_mae = baseline_errors[sample_idx].mean()
                m_mae = model_errors[sample_idx].mean()

                if b_mae > 0:
                    boot_improvements.append(
                        ((b_mae - m_mae) / b_mae) * 100
                    )

            boot_improvements = np.asarray(boot_improvements)

            ci_lower, ci_upper = np.percentile(
                boot_improvements,
                [2.5, 97.5]
            )

            # Paired MAE difference: positive means model has lower error.
            paired_difference = (
                baseline_errors - model_errors
            )

            results.append({
                "stage": stage,
                "group": group_name,
                "group_value": group_value,
                "outcome": outcome,
                "model": model,
                "players": n_players,
                "baseline_mae": baseline_mae,
                "model_mae": model_mae,
                "mae_improvement_pct": improvement,
                "improvement_ci_lower_pct": ci_lower,
                "improvement_ci_upper_pct": ci_upper,
                "mean_paired_mae_difference": paired_difference.mean(),
                "bootstrap_probability_improvement": (
                    boot_improvements > 0
                ).mean()
            })

    return results

# ============================================================================
# 5. STEP 10A — OVERALL PLAYER-LEVEL UNCERTAINTY
# ============================================================================

all_results = []

df_10a = prediction_data["10A_GROUPED"]

results_10a = paired_player_bootstrap(
    df=df_10a,
    stage="10A_GROUPED",
    group_name="overall",
    group_value="all_players",
    n_bootstraps=N_BOOTSTRAPS
)

all_results.extend(results_10a)

# ============================================================================
# 6. STEP 10B — OVERALL AND COHORT-SPECIFIC UNCERTAINTY
# ============================================================================

df_10b = prediction_data["10B_COHORT"]

# Overall pooled evaluation.
results_10b_all = paired_player_bootstrap(
    df=df_10b,
    stage="10B_COHORT",
    group_name="overall",
    group_value="all_cohorts",
    n_bootstraps=N_BOOTSTRAPS
)

all_results.extend(results_10b_all)

# Individual held-out draft cohorts.
if "held_out_cohort" in df_10b.columns:

    for cohort in sorted(df_10b["held_out_cohort"].dropna().unique()):

        cohort_df = df_10b[
            df_10b["held_out_cohort"] == cohort
        ].copy()

        cohort_results = paired_player_bootstrap(
            df=cohort_df,
            stage="10B_COHORT",
            group_name="held_out_cohort",
            group_value=int(cohort),
            n_bootstraps=N_BOOTSTRAPS
        )

        all_results.extend(cohort_results)

# ============================================================================
# 7. RESULTS TABLE
# ============================================================================

bootstrap_results = pd.DataFrame(all_results)

if bootstrap_results.empty:
    raise ValueError("No valid paired bootstrap results were produced.")

bootstrap_results = bootstrap_results.sort_values(
    ["stage", "group", "group_value", "outcome", "model"]
).reset_index(drop=True)

print("\n" + "=" * 100)
print("PAIRED PLAYER-LEVEL BOOTSTRAP RESULTS")
print("=" * 100)

print(
    bootstrap_results.round(4).to_string(index=False)
)

# ============================================================================
# 8. INTERPRETATION FLAGS
# ============================================================================

bootstrap_results["ci_excludes_zero"] = (
    (bootstrap_results["improvement_ci_lower_pct"] > 0) |
    (bootstrap_results["improvement_ci_upper_pct"] < 0)
)

bootstrap_results["direction"] = np.select(
    [
        bootstrap_results["improvement_ci_lower_pct"] > 0,
        bootstrap_results["improvement_ci_upper_pct"] < 0
    ],
    [
        "ROBUST_IMPROVEMENT_SIGNAL",
        "ROBUST_DETERIORATION_SIGNAL"
    ],
    default="UNCERTAIN"
)

print("\n" + "=" * 100)
print("INTERPRETATION FLAGS")
print("=" * 100)

print(
    bootstrap_results[
        [
            "stage",
            "group",
            "group_value",
            "outcome",
            "model",
            "players",
            "mae_improvement_pct",
            "improvement_ci_lower_pct",
            "improvement_ci_upper_pct",
            "bootstrap_probability_improvement",
            "direction"
        ]
    ].round(4).to_string(index=False)
)

# ============================================================================
# 9. SAVE OUTPUTS
# ============================================================================

output_csv = (
    MODEL_DIR / "step10c_paired_player_bootstrap_results.csv"
)

output_parquet = (
    MODEL_DIR / "step10c_paired_player_bootstrap_results.parquet"
)

bootstrap_results.to_csv(output_csv, index=False)
bootstrap_results.to_parquet(output_parquet, index=False)

print("\n" + "=" * 85)
print("OUTPUT VERIFICATION")
print("=" * 85)

for path in [output_csv, output_parquet]:
    print(
        "VERIFIED" if path.exists() else "MISSING",
        path
    )

print("\nSTEP 10C COMPLETED SUCCESSFULLY.")
print("Bootstrap repetitions:", N_BOOTSTRAPS)
print("Independent resampling unit: nfl_id")
print("Random seed:", RANDOM_SEED)


STEP 10C — RIDGE ALPHA AND PAIRED PLAYER BOOTSTRAP

RIDGE ALPHA AUDIT — 10A
Shape: (75, 7)
Columns: ['outcome', 'model', 'fold', 'best_alpha', 'inner_best_mae', 'train_players', 'test_players']

Alpha frequency:
best_alpha
10.0       4
100.0     34
1000.0    37
Name: count, dtype: int64

Alpha by model:
        model  records  median_alpha  minimum_alpha  maximum_alpha
A_TRADITIONAL       25         100.0           10.0         1000.0
   B_TRACKING       25         100.0           10.0         1000.0
   C_COMBINED       25        1000.0          100.0         1000.0

Maximum selected alpha: 1000.0
Selections at maximum: 37

RIDGE ALPHA AUDIT — 10B
Shape: (45, 7)
Columns: ['held_out_cohort', 'outcome', 'model', 'best_alpha', 'inner_best_mae', 'train_players', 'test_players']

Alpha frequency:
best_alpha
10.0       5
100.0     14
1000.0    26
Name: count, dtype: int64

Alpha by model:
        model  records  median_alpha  minimum_alpha  maximum_alpha
A_TRADITIONAL       15        1000.0 

ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column group_value with type object')

In [27]:

# ============================================================================
# STEP 10C — SAFE OUTPUT SAVE
# Fix mixed types in group_value before Parquet serialization
# ============================================================================

from pathlib import Path

MODEL_DIR = Path(
    "/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results"
)

output_csv = MODEL_DIR / "step10c_paired_player_bootstrap_results.csv"
output_parquet = MODEL_DIR / "step10c_paired_player_bootstrap_results.parquet"

# Preserve the original results and normalize the mixed-type column.
bootstrap_results = bootstrap_results.copy()
bootstrap_results["group_value"] = (
    bootstrap_results["group_value"].astype(str)
)

# Save both formats.
bootstrap_results.to_csv(output_csv, index=False)
bootstrap_results.to_parquet(output_parquet, index=False)

# Verify files.
for path in [output_csv, output_parquet]:
    print(
        f"{'VERIFIED' if path.exists() else 'MISSING'}: {path}"
    )
    if path.exists():
        print(f"Size: {path.stat().st_size:,} bytes")

# Read-back verification.
check = pd.read_parquet(output_parquet)

print("\nPARQUET READ-BACK")
print("Rows:", len(check))
print("Columns:", len(check.columns))
print("Unique outcomes:", check["outcome"].nunique())
print("Unique models:", check["model"].nunique())
print("Stages:", check["stage"].unique().tolist())

print("\nSTEP 10C OUTPUTS SAVED SUCCESSFULLY.")


VERIFIED: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10c_paired_player_bootstrap_results.csv
Size: 16,848 bytes
VERIFIED: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/reports/model_results/step10c_paired_player_bootstrap_results.parquet
Size: 14,445 bytes

PARQUET READ-BACK
Rows: 75
Columns: 15
Unique outcomes: 5
Unique models: 3
Stages: ['10A_GROUPED', '10B_COHORT']

STEP 10C OUTPUTS SAVED SUCCESSFULLY.


In [28]:
ALPHAS = [
    0.001,
    0.01,
    0.1,
    1,
    10,
    100,
    1000,
    10000,
    100000
]

In [29]:

# ============================================================================
# STEP 10B-R — RIDGE ALPHA SENSITIVITY RERUN
# NFL BIG DATA BOWL 2027 | WR-MovementIQ
#
# Same leave-one-draft-cohort-out design.
# Only the Ridge alpha grid is expanded.
# Original Step 10B outputs remain untouched.
# ============================================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --------------------------------------------------------------------------
# 1. PATHS AND SETTINGS
# --------------------------------------------------------------------------

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

DATA_DIR = ROOT / "data" / "features"
MODEL_DIR = ROOT / "reports" / "model_results"
AUDIT_DIR = ROOT / "reports" / "data_audit"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "wr_modeling_dataset_primary_10plus.parquet"

ALPHAS = [
    0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000
]

INNER_FOLDS = 4

RANDOM_SEED = 42

# --------------------------------------------------------------------------
# 2. APPROVED FEATURES AND OUTCOMES
# --------------------------------------------------------------------------

MODEL_A = [
    "forty",
    "vertical",
    "broad_jump"
]

MODEL_B = [
    "mean_speed_median_across_drills",
    "peak_speed_median_across_drills",
    "mean_acceleration_median_across_drills",
    "path_efficiency_median_across_drills"
]

MODEL_C = MODEL_A + MODEL_B

OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_path_efficiency_median",
    "game_separation_at_pass_forward_median"
]

MODEL_FEATURES = {
    "A_TRADITIONAL": MODEL_A,
    "B_TRACKING": MODEL_B,
    "C_COMBINED": MODEL_C
}

# --------------------------------------------------------------------------
# 3. LOAD AND VALIDATE DATA
# --------------------------------------------------------------------------

df = pd.read_parquet(DATA_PATH)

required = [
    "draft_year",
    "nfl_id",
    "season",
    "receiver_plays"
] + MODEL_C + OUTCOMES

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.copy()

df["draft_year"] = pd.to_numeric(
    df["draft_year"], errors="coerce"
)

df["nfl_id"] = df["nfl_id"].astype(str)

print("=" * 80)
print("STEP 10B-R — RIDGE ALPHA SENSITIVITY")
print("=" * 80)

print("Dataset:", df.shape)
print("Players:", df["nfl_id"].nunique())
print("Draft cohorts:", sorted(df["draft_year"].dropna().unique()))
print("Alpha grid:", ALPHAS)

# --------------------------------------------------------------------------
# 4. RIDGE PIPELINE
# --------------------------------------------------------------------------

def make_pipeline(alpha):

    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])


# --------------------------------------------------------------------------
# 5. INNER GROUPED ALPHA SELECTION
# --------------------------------------------------------------------------

def select_alpha(X, y, groups):

    unique_groups = pd.Series(groups).nunique()

    n_splits = min(INNER_FOLDS, unique_groups)

    if n_splits < 2:
        raise ValueError("Insufficient training players for inner CV.")

    cv = GroupKFold(n_splits=n_splits)

    alpha_scores = []

    for alpha in ALPHAS:

        fold_maes = []

        for train_idx, val_idx in cv.split(X, y, groups):

            X_train = X.iloc[train_idx]
            X_val = X.iloc[val_idx]

            y_train = y.iloc[train_idx]
            y_val = y.iloc[val_idx]

            model = make_pipeline(alpha)

            model.fit(X_train, y_train)

            predictions = model.predict(X_val)

            fold_maes.append(
                mean_absolute_error(y_val, predictions)
            )

        alpha_scores.append({
            "alpha": alpha,
            "inner_best_mae": np.mean(fold_maes),
            "inner_mae_std": np.std(fold_maes),
            "inner_folds": n_splits
        })

    score_df = pd.DataFrame(alpha_scores)

    # Deterministic tie-breaking: select the smaller alpha.
    score_df = score_df.sort_values(
        ["inner_best_mae", "alpha"]
    ).reset_index(drop=True)

    return score_df.iloc[0]["alpha"], score_df


# --------------------------------------------------------------------------
# 6. METRICS
# --------------------------------------------------------------------------

def calculate_metrics(actual, predicted):

    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": np.sqrt(mean_squared_error(actual, predicted)),
        "r2": r2_score(actual, predicted) if len(actual) > 1 else np.nan,
        "spearman": pd.Series(actual).corr(
            pd.Series(predicted),
            method="spearman"
        )
    }


# --------------------------------------------------------------------------
# 7. LEAVE-ONE-DRAFT-COHORT-OUT
# --------------------------------------------------------------------------

prediction_records = []
alpha_records = []
fold_records = []
metric_records = []

cohorts = sorted(df["draft_year"].dropna().unique())

for held_out in cohorts:

    train = df[df["draft_year"] != held_out].copy()
    test = df[df["draft_year"] == held_out].copy()

    train_players = set(train["nfl_id"])
    test_players = set(test["nfl_id"])

    overlap = train_players.intersection(test_players)

    assert len(overlap) == 0, (
        f"Player leakage in cohort {held_out}"
    )

    print("\n" + "=" * 80)
    print("HELD-OUT DRAFT COHORT:", held_out)
    print("=" * 80)

    print("Training rows:", len(train))
    print("Test rows:", len(test))
    print("Training players:", len(train_players))
    print("Test players:", len(test_players))
    print("Player overlap:", len(overlap))

    fold_records.append({
        "held_out_cohort": int(held_out),
        "train_rows": len(train),
        "test_rows": len(test),
        "train_players": len(train_players),
        "test_players": len(test_players),
        "player_overlap": len(overlap)
    })

    for outcome in OUTCOMES:

        train_y = pd.to_numeric(
            train[outcome], errors="coerce"
        )

        test_y = pd.to_numeric(
            test[outcome], errors="coerce"
        )

        train_valid = train_y.notna()
        test_valid = test_y.notna()

        train_outcome = train.loc[train_valid].copy()
        test_outcome = test.loc[test_valid].copy()

        y_train = train_y.loc[train_valid]
        y_test = test_y.loc[test_valid]

        # Baseline: training-set mean only.
        baseline_prediction = np.repeat(
            y_train.mean(),
            len(test_outcome)
        )

        for i, (_, row) in enumerate(test_outcome.iterrows()):

            prediction_records.append({
                "held_out_cohort": int(held_out),
                "outcome": outcome,
                "model": "BASELINE",
                "nfl_id": row["nfl_id"],
                "season": row["season"],
                "actual": float(y_test.iloc[i]),
                "predicted": float(baseline_prediction[i]),
                "best_alpha": np.nan
            })

        baseline_metrics = calculate_metrics(
            y_test.to_numpy(),
            baseline_prediction
        )

        metric_records.append({
            "held_out_cohort": int(held_out),
            "outcome": outcome,
            "model": "BASELINE",
            "best_alpha": np.nan,
            "test_players": test_outcome["nfl_id"].nunique(),
            **baseline_metrics
        })

        for model_name, features in MODEL_FEATURES.items():

            X_train = train_outcome[features].apply(
                pd.to_numeric, errors="coerce"
            )

            X_test = test_outcome[features].apply(
                pd.to_numeric, errors="coerce"
            )

            selected_alpha, alpha_scores = select_alpha(
                X_train.reset_index(drop=True),
                y_train.reset_index(drop=True),
                train_outcome["nfl_id"].reset_index(drop=True)
            )

            alpha_records.append({
                "held_out_cohort": int(held_out),
                "outcome": outcome,
                "model": model_name,
                "best_alpha": float(selected_alpha),
                "inner_best_mae": float(
                    alpha_scores.iloc[0]["inner_best_mae"]
                ),
                "train_players": train_outcome["nfl_id"].nunique(),
                "test_players": test_outcome["nfl_id"].nunique(),
                "selected_at_lower_boundary": (
                    selected_alpha == min(ALPHAS)
                ),
                "selected_at_upper_boundary": (
                    selected_alpha == max(ALPHAS)
                )
            })

            final_model = make_pipeline(selected_alpha)

            final_model.fit(X_train, y_train)

            predictions = final_model.predict(X_test)

            for i, (_, row) in enumerate(test_outcome.iterrows()):

                prediction_records.append({
                    "held_out_cohort": int(held_out),
                    "outcome": outcome,
                    "model": model_name,
                    "nfl_id": row["nfl_id"],
                    "season": row["season"],
                    "actual": float(y_test.iloc[i]),
                    "predicted": float(predictions[i]),
                    "best_alpha": float(selected_alpha)
                })

            model_metrics = calculate_metrics(
                y_test.to_numpy(),
                predictions
            )

            metric_records.append({
                "held_out_cohort": int(held_out),
                "outcome": outcome,
                "model": model_name,
                "best_alpha": float(selected_alpha),
                "test_players": test_outcome["nfl_id"].nunique(),
                **model_metrics
            })

            print(
                outcome,
                model_name,
                "alpha =", selected_alpha,
                "MAE =", round(model_metrics["mae"], 5)
            )

# --------------------------------------------------------------------------
# 8. SAVE RESULTS
# --------------------------------------------------------------------------

predictions_df = pd.DataFrame(prediction_records)
alphas_df = pd.DataFrame(alpha_records)
folds_df = pd.DataFrame(fold_records)
metrics_df = pd.DataFrame(metric_records)

prefix = "step10b_ridge_expanded"

outputs = {
    MODEL_DIR / f"{prefix}_predictions.parquet": predictions_df,
    MODEL_DIR / f"{prefix}_selected_alphas.csv": alphas_df,
    MODEL_DIR / f"{prefix}_metrics.csv": metrics_df,
    AUDIT_DIR / f"{prefix}_fold_audit.csv": folds_df
}

for path, data in outputs.items():

    if path.suffix == ".parquet":
        data.to_parquet(path, index=False)
    else:
        data.to_csv(path, index=False)

    print(
        "VERIFIED" if path.exists() else "MISSING",
        path,
        "Bytes:", path.stat().st_size if path.exists() else None
    )

# --------------------------------------------------------------------------
# 9. ALPHA SELECTION SUMMARY
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("EXPANDED RIDGE ALPHA FREQUENCY")
print("=" * 80)

print(
    alphas_df["best_alpha"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nALPHA SELECTION BY MODEL")

print(
    alphas_df.groupby("model")["best_alpha"]
    .agg(
        records="count",
        median_alpha="median",
        minimum_alpha="min",
        maximum_alpha="max"
    )
)

print("\nUpper boundary selections:")
print(
    alphas_df.groupby("model")["selected_at_upper_boundary"]
    .sum()
)

print("\nLower boundary selections:")
print(
    alphas_df.groupby("model")["selected_at_lower_boundary"]
    .sum()
)

print("\nSTEP 10B-R COMPLETED.")


STEP 10B-R — RIDGE ALPHA SENSITIVITY
Dataset: (116, 21)
Players: 66
Draft cohorts: [np.int64(2023), np.int64(2024), np.int64(2025)]
Alpha grid: [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000]

HELD-OUT DRAFT COHORT: 2023
Training rows: 52
Test rows: 64
Training players: 38
Test players: 28
Player overlap: 0
game_mean_speed_raw_median A_TRADITIONAL alpha = 100000.0 MAE = 0.47245
game_mean_speed_raw_median B_TRACKING alpha = 100000.0 MAE = 0.47243
game_mean_speed_raw_median C_COMBINED alpha = 100000.0 MAE = 0.47245
game_peak_speed_raw_median A_TRADITIONAL alpha = 100.0 MAE = 0.49488
game_peak_speed_raw_median B_TRACKING alpha = 100000.0 MAE = 0.50667
game_peak_speed_raw_median C_COMBINED alpha = 100000.0 MAE = 0.50664
game_mean_acceleration_raw_median A_TRADITIONAL alpha = 100.0 MAE = 0.18937
game_mean_acceleration_raw_median B_TRACKING alpha = 100000.0 MAE = 0.18613
game_mean_acceleration_raw_median C_COMBINED alpha = 100000.0 MAE = 0.18614
game_path_efficiency_median A_TRADITIONAL 

In [30]:

# ============================================================================
# STEP 10B-R2 — RIDGE REGULARISATION DIAGNOSTIC
# Purpose:
#   1. Reconstruct inner-CV alpha curves.
#   2. Measure coefficient shrinkage.
#   3. Compare model predictions with training-mean predictions.
#   4. Identify flat CV curves and upper-boundary selections.
# ============================================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

DATA_PATH = ROOT / "data/features/wr_modeling_dataset_primary_10plus.parquet"

MODEL_DIR = ROOT / "reports/model_results"
AUDIT_DIR = ROOT / "reports/data_audit"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

ALPHAS = [
    0.001, 0.01, 0.1, 1, 10,
    100, 1000, 10000, 100000
]

MODEL_FEATURES = {
    "A_TRADITIONAL": [
        "forty", "vertical", "broad_jump"
    ],
    "B_TRACKING": [
        "mean_speed_median_across_drills",
        "peak_speed_median_across_drills",
        "mean_acceleration_median_across_drills",
        "path_efficiency_median_across_drills"
    ],
    "C_COMBINED": [
        "forty", "vertical", "broad_jump",
        "mean_speed_median_across_drills",
        "peak_speed_median_across_drills",
        "mean_acceleration_median_across_drills",
        "path_efficiency_median_across_drills"
    ]
}

OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_path_efficiency_median",
    "game_separation_at_pass_forward_median"
]

df = pd.read_parquet(DATA_PATH).copy()

df["draft_year"] = pd.to_numeric(
    df["draft_year"], errors="coerce"
)
df["nfl_id"] = df["nfl_id"].astype(str)

def make_model(alpha):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

curve_records = []
diagnostic_records = []

# --------------------------------------------------------------------------
# RECONSTRUCT THE SAME COHORT HOLDOUTS AND INNER CV
# --------------------------------------------------------------------------

for held_out in sorted(df["draft_year"].dropna().unique()):

    train = df[df["draft_year"] != held_out].copy()
    test = df[df["draft_year"] == held_out].copy()

    for outcome in OUTCOMES:

        train_y = pd.to_numeric(
            train[outcome], errors="coerce"
        )

        test_y = pd.to_numeric(
            test[outcome], errors="coerce"
        )

        train_valid = train_y.notna()
        test_valid = test_y.notna()

        tr = train.loc[train_valid].copy()
        te = test.loc[test_valid].copy()

        y_train = train_y.loc[train_valid].reset_index(drop=True)
        y_test = test_y.loc[test_valid].reset_index(drop=True)

        groups = tr["nfl_id"].reset_index(drop=True)

        for model_name, features in MODEL_FEATURES.items():

            X_train = tr[features].apply(
                pd.to_numeric, errors="coerce"
            ).reset_index(drop=True)

            X_test = te[features].apply(
                pd.to_numeric, errors="coerce"
            ).reset_index(drop=True)

            cv = GroupKFold(
                n_splits=min(4, groups.nunique())
            )

            alpha_scores = {}

            # Inner grouped CV
            for alpha in ALPHAS:

                fold_scores = []

                for train_idx, val_idx in cv.split(
                    X_train, y_train, groups
                ):

                    model = make_model(alpha)

                    model.fit(
                        X_train.iloc[train_idx],
                        y_train.iloc[train_idx]
                    )

                    pred = model.predict(
                        X_train.iloc[val_idx]
                    )

                    fold_scores.append(
                        mean_absolute_error(
                            y_train.iloc[val_idx], pred
                        )
                    )

                alpha_scores[alpha] = np.mean(fold_scores)

                curve_records.append({
                    "held_out_cohort": int(held_out),
                    "outcome": outcome,
                    "model": model_name,
                    "alpha": alpha,
                    "inner_cv_mae": alpha_scores[alpha]
                })

            selected_alpha = min(
                alpha_scores,
                key=lambda a: (alpha_scores[a], a)
            )

            # Fit final model on all outer training data
            final_model = make_model(selected_alpha)

            final_model.fit(X_train, y_train)

            predictions = final_model.predict(X_test)

            ridge = final_model.named_steps["ridge"]

            coefficient_norm = float(
                np.linalg.norm(ridge.coef_)
            )

            coefficient_max_abs = float(
                np.max(np.abs(ridge.coef_))
            )

            training_mean = float(y_train.mean())

            baseline_predictions = np.repeat(
                training_mean, len(y_test)
            )

            model_mae = mean_absolute_error(
                y_test, predictions
            )

            baseline_mae = mean_absolute_error(
                y_test, baseline_predictions
            )

            prediction_sd = float(np.std(predictions))

            prediction_mean = float(np.mean(predictions))

            baseline_prediction_sd = float(
                np.std(baseline_predictions)
            )

            # Distance of predictions from the training mean
            prediction_deviation = float(
                np.mean(np.abs(predictions - training_mean))
            )

            # CV curve diagnostics
            best_cv_mae = alpha_scores[selected_alpha]

            second_best = sorted(
                alpha_scores.values()
            )[1]

            cv_gap = float(second_best - best_cv_mae)

            min_alpha = min(ALPHAS)
            max_alpha = max(ALPHAS)

            diagnostic_records.append({
                "held_out_cohort": int(held_out),
                "outcome": outcome,
                "model": model_name,
                "selected_alpha": selected_alpha,
                "selected_at_upper_boundary": (
                    selected_alpha == max_alpha
                ),
                "inner_cv_best_mae": best_cv_mae,
                "second_best_cv_mae": second_best,
                "cv_gap_second_best": cv_gap,
                "training_mean": training_mean,
                "test_actual_mean": float(y_test.mean()),
                "test_prediction_mean": prediction_mean,
                "test_prediction_sd": prediction_sd,
                "mean_abs_prediction_distance_from_train_mean":
                    prediction_deviation,
                "baseline_mae": baseline_mae,
                "model_mae": model_mae,
                "mae_improvement": baseline_mae - model_mae,
                "coefficient_l2_norm": coefficient_norm,
                "coefficient_max_abs": coefficient_max_abs,
                "training_players": groups.nunique(),
                "test_players": te["nfl_id"].nunique()
            })

# --------------------------------------------------------------------------
# SAVE
# --------------------------------------------------------------------------

curves = pd.DataFrame(curve_records)
diagnostics = pd.DataFrame(diagnostic_records)

curve_path = MODEL_DIR / "step10b_ridge_alpha_cv_curves.csv"
diagnostic_path = MODEL_DIR / "step10b_ridge_regularisation_diagnostics.csv"

curves.to_csv(curve_path, index=False)
diagnostics.to_csv(diagnostic_path, index=False)

# --------------------------------------------------------------------------
# DISPLAY
# --------------------------------------------------------------------------

print("=" * 80)
print("RIDGE REGULARISATION DIAGNOSTIC")
print("=" * 80)

print("\n1. ALPHA CURVE SUMMARY")

print(
    curves.groupby(["model", "alpha"])["inner_cv_mae"]
    .mean()
    .unstack()
    .round(5)
)

print("\n2. SELECTED ALPHA AND SHRINKAGE")

display_columns = [
    "held_out_cohort",
    "outcome",
    "model",
    "selected_alpha",
    "selected_at_upper_boundary",
    "baseline_mae",
    "model_mae",
    "mae_improvement",
    "test_prediction_sd",
    "mean_abs_prediction_distance_from_train_mean",
    "coefficient_l2_norm",
    "coefficient_max_abs",
    "cv_gap_second_best"
]

print(
    diagnostics[display_columns]
    .round(6)
    .to_string(index=False)
)

print("\n3. UPPER-BOUNDARY SELECTIONS BY MODEL")

print(
    diagnostics.groupby("model")[
        "selected_at_upper_boundary"
    ].agg(["sum", "count"])
)

print("\n4. OUTPUT FILES")

print(curve_path)
print(diagnostic_path)

print("\nSTEP 10B-R2 COMPLETED.")


RIDGE REGULARISATION DIAGNOSTIC

1. ALPHA CURVE SUMMARY
alpha          0.001       0.010       0.100       1.000       10.000      \
model                                                                       
A_TRADITIONAL     0.37849     0.37847     0.37836     0.37728     0.36998   
B_TRACKING        0.38422     0.38420     0.38407     0.38281     0.37423   
C_COMBINED        0.41173     0.41169     0.41135     0.40818     0.38918   

alpha          100.000     1000.000    10000.000   100000.000  
model                                                          
A_TRADITIONAL     0.35798     0.35697     0.35724     0.35727  
B_TRACKING        0.36026     0.35739     0.35729     0.35728  
C_COMBINED        0.36131     0.35706     0.35725     0.35728  

2. SELECTED ALPHA AND SHRINKAGE
 held_out_cohort                                outcome         model  selected_alpha  selected_at_upper_boundary  baseline_mae  model_mae  mae_improvement  test_prediction_sd  mean_abs_prediction_distance

In [31]:

# STEP 10B-R3 — OUTCOME-SPECIFIC ALPHA STABILITY

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

MODEL_DIR = ROOT / "reports/model_results"

curves = pd.read_csv(
    MODEL_DIR / "step10b_ridge_alpha_cv_curves.csv"
)

diagnostics = pd.read_csv(
    MODEL_DIR / "step10b_ridge_regularisation_diagnostics.csv"
)

# Calculate each alpha's percentage excess MAE relative to
# the best alpha within the same cohort, outcome and model.
group_cols = [
    "held_out_cohort", "outcome", "model"
]

curves["best_group_mae"] = (
    curves.groupby(group_cols)["inner_cv_mae"]
    .transform("min")
)

curves["excess_mae_pct"] = (
    (curves["inner_cv_mae"] - curves["best_group_mae"])
    / curves["best_group_mae"].abs().clip(lower=1e-12)
) * 100

# Select alpha independently for each cohort/outcome/model.
selected = (
    curves.sort_values(
        group_cols + ["inner_cv_mae", "alpha"]
    )
    .groupby(group_cols, as_index=False)
    .first()
)

print("=" * 80)
print("1. SELECTED ALPHA BY OUTCOME AND MODEL")
print("=" * 80)

print(
    selected.pivot_table(
        index="outcome",
        columns="model",
        values="alpha",
        aggfunc=lambda x: ", ".join(map(str, x))
    ).to_string()
)

print("\n" + "=" * 80)
print("2. ALPHA FREQUENCY BY OUTCOME")
print("=" * 80)

print(
    selected.groupby(["outcome", "model"])["alpha"]
    .value_counts()
    .unstack(fill_value=0)
    .to_string()
)

print("\n" + "=" * 80)
print("3. NORMALISED CV CURVE — MEAN EXCESS MAE (%)")
print("=" * 80)

normalised = (
    curves.groupby(["outcome", "model", "alpha"])
    ["excess_mae_pct"]
    .mean()
    .reset_index()
)

print(
    normalised.pivot_table(
        index=["outcome", "model"],
        columns="alpha",
        values="excess_mae_pct"
    ).round(3).to_string()
)

print("\n" + "=" * 80)
print("4. ALPHA SELECTION BOUNDARY AUDIT")
print("=" * 80)

print(
    selected.groupby(["outcome", "model"])
    .agg(
        folds=("held_out_cohort", "count"),
        upper_boundary=("alpha", lambda x: (x == 100000).sum()),
        median_alpha=("alpha", "median")
    ).to_string()
)

# Save
selected.to_csv(
    MODEL_DIR / "step10b_ridge_alpha_by_outcome.csv",
    index=False
)

normalised.to_csv(
    MODEL_DIR / "step10b_ridge_normalised_cv_curves.csv",
    index=False
)

print("\nSTEP 10B-R3 COMPLETED.")


1. SELECTED ALPHA BY OUTCOME AND MODEL
model                                                A_TRADITIONAL                  B_TRACKING                    C_COMBINED
outcome                                                                                                                     
game_mean_acceleration_raw_median          100.0, 1000.0, 100000.0   100000.0, 100000.0, 100.0  100000.0, 100000.0, 100000.0
game_mean_speed_raw_median                  100000.0, 100.0, 100.0  100000.0, 100000.0, 1000.0       100000.0, 100.0, 1000.0
game_path_efficiency_median               10.0, 100000.0, 100000.0        100000.0, 10.0, 10.0             10.0, 100.0, 10.0
game_peak_speed_raw_median                     100.0, 100.0, 100.0      100000.0, 100.0, 100.0        100000.0, 100.0, 100.0
game_separation_at_pass_forward_median  100000.0, 1000.0, 100000.0   100000.0, 100000.0, 100.0    100000.0, 100000.0, 1000.0

2. ALPHA FREQUENCY BY OUTCOME
alpha                                                 1

In [32]:

# ============================================================================
# STEP 10B-R4 — PAIRED PLAYER BOOTSTRAP
# Expanded-alpha Step 10B predictions
# ============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

MODEL_DIR = ROOT / "reports/model_results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT = MODEL_DIR / "step10b_ridge_expanded_predictions.parquet"

OUTPUT_CSV = MODEL_DIR / "step10b_ridge_expanded_paired_bootstrap.csv"
OUTPUT_PARQUET = MODEL_DIR / "step10b_ridge_expanded_paired_bootstrap.parquet"

N_BOOT = 5000
SEED = 42

pred = pd.read_parquet(INPUT)

required = [
    "held_out_cohort", "outcome", "model",
    "nfl_id", "actual", "predicted"
]

missing = [c for c in required if c not in pred.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

pred["nfl_id"] = pred["nfl_id"].astype(str)

# --------------------------------------------------------------------------
# 1. AGGREGATE TO ONE RECORD PER PLAYER
# --------------------------------------------------------------------------

# Each player contributes one actual and one prediction per
# held-out cohort, outcome and model.
player_pred = (
    pred.groupby(
        ["held_out_cohort", "outcome", "model", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        player_seasons=("actual", "size")
    )
)

# Verify actual outcomes agree across models for each player.
actual_check = (
    player_pred.groupby(
        ["held_out_cohort", "outcome", "nfl_id"]
    )["actual"].nunique()
)

if (actual_check > 1).any():
    raise ValueError("Actual outcomes differ across models.")

# --------------------------------------------------------------------------
# 2. PAIRED PLAYER BOOTSTRAP
# --------------------------------------------------------------------------

rng = np.random.default_rng(SEED)

results = []

groups = [
    ("ALL_COHORTS", player_pred),
]

for cohort, sub in player_pred.groupby("held_out_cohort"):
    groups.append((int(cohort), sub))

for group_name, group_data in groups:

    for outcome, outcome_data in group_data.groupby("outcome"):

        pivot = outcome_data.pivot(
            index="nfl_id",
            columns="model",
            values=["actual", "predicted"]
        )

        if "BASELINE" not in pivot["predicted"].columns:
            continue

        actual = pivot["actual"]["BASELINE"].to_numpy()
        baseline = pivot["predicted"]["BASELINE"].to_numpy()

        n_players = len(actual)

        if n_players < 5:
            continue

        baseline_mae = mean_absolute_error(actual, baseline)

        for model_name in [
            "A_TRADITIONAL",
            "B_TRACKING",
            "C_COMBINED"
        ]:

            if model_name not in pivot["predicted"].columns:
                continue

            model_pred = pivot["predicted"][model_name].to_numpy()

            model_mae = mean_absolute_error(actual, model_pred)

            observed_improvement = (
                (baseline_mae - model_mae)
                / max(baseline_mae, 1e-12)
            ) * 100

            bootstrap_improvements = []

            for _ in range(N_BOOT):

                idx = rng.integers(
                    0, n_players, size=n_players
                )

                sampled_actual = actual[idx]
                sampled_baseline = baseline[idx]
                sampled_model = model_pred[idx]

                b_mae = mean_absolute_error(
                    sampled_actual, sampled_baseline
                )

                m_mae = mean_absolute_error(
                    sampled_actual, sampled_model
                )

                improvement = (
                    (b_mae - m_mae)
                    / max(b_mae, 1e-12)
                ) * 100

                bootstrap_improvements.append(improvement)

            bootstrap_improvements = np.asarray(
                bootstrap_improvements
            )

            ci_low, ci_high = np.percentile(
                bootstrap_improvements, [2.5, 97.5]
            )

            results.append({
                "stage": "10B_R_EXPANDED_ALPHA",
                "cohort": group_name,
                "outcome": outcome,
                "model": model_name,
                "players": n_players,
                "baseline_mae": baseline_mae,
                "model_mae": model_mae,
                "mae_improvement_pct": observed_improvement,
                "bootstrap_ci_lower": ci_low,
                "bootstrap_ci_upper": ci_high,
                "probability_improvement": np.mean(
                    bootstrap_improvements > 0
                ),
                "bootstrap_draws": N_BOOT
            })

results_df = pd.DataFrame(results)

# --------------------------------------------------------------------------
# 3. SAVE
# --------------------------------------------------------------------------

results_df.to_csv(OUTPUT_CSV, index=False)
results_df.to_parquet(OUTPUT_PARQUET, index=False)

print("=" * 80)
print("STEP 10B-R4 — PAIRED PLAYER BOOTSTRAP")
print("=" * 80)

print("Records:", len(results_df))
print("Outcomes:", results_df["outcome"].nunique())
print("Models:", results_df["model"].nunique())

print("\nPEAK SPEED — ALL COHORTS")

peak = results_df[
    (results_df["outcome"] == "game_peak_speed_raw_median") &
    (results_df["cohort"] == "ALL_COHORTS")
]

print(
    peak[
        [
            "model", "players", "baseline_mae", "model_mae",
            "mae_improvement_pct", "bootstrap_ci_lower",
            "bootstrap_ci_upper", "probability_improvement"
        ]
    ].round(4).to_string(index=False)
)

print("\nPEAK SPEED — BY COHORT")

peak_cohort = results_df[
    results_df["outcome"] == "game_peak_speed_raw_median"
]

print(
    peak_cohort[
        [
            "cohort", "model", "players",
            "mae_improvement_pct",
            "bootstrap_ci_lower", "bootstrap_ci_upper",
            "probability_improvement"
        ]
    ].round(4).to_string(index=False)
)

print("\nOUTPUT FILES")
for path in [OUTPUT_CSV, OUTPUT_PARQUET]:
    print(path, path.exists(), path.stat().st_size)

print("\nSTEP 10B-R4 COMPLETED.")


ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column cohort with type object')

In [33]:
# ============================================================
# STEP 10B-R4 — CORRECTED OUTPUT SAVE
# Fix mixed cohort data types before Parquet export
# ============================================================

import pandas as pd
from pathlib import Path

# Ensure cohort has one consistent data type
results_df["cohort"] = results_df["cohort"].astype("string")

# Normalize other object columns to avoid mixed-type Arrow errors
for col in results_df.select_dtypes(include=["object"]).columns:
    results_df[col] = results_df[col].apply(
        lambda x: str(x) if isinstance(x, (int, float)) else x
    )

# Output paths
OUTPUT_DIR = ROOT / "reports" / "model_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "step10b_ridge_expanded_paired_bootstrap.csv"
OUTPUT_PARQUET = OUTPUT_DIR / "step10b_ridge_expanded_paired_bootstrap.parquet"

# Save both formats
results_df.to_csv(OUTPUT_CSV, index=False)
results_df.to_parquet(OUTPUT_PARQUET, index=False)

# Read-back validation
check_df = pd.read_parquet(OUTPUT_PARQUET)

print("=" * 75)
print("STEP 10B-R4 — OUTPUT VALIDATION")
print("=" * 75)

print("CSV exists:", OUTPUT_CSV.exists())
print("Parquet exists:", OUTPUT_PARQUET.exists())

print("CSV size:", OUTPUT_CSV.stat().st_size, "bytes")
print("Parquet size:", OUTPUT_PARQUET.stat().st_size, "bytes")

print("\nOriginal shape:", results_df.shape)
print("Parquet read-back shape:", check_df.shape)

print("\nCohort values:")
print(check_df["cohort"].value_counts(dropna=False))

print("\nUnique outcomes:", check_df["outcome"].nunique())
print("Unique models:", check_df["model"].nunique())

print("\nColumns:")
print(check_df.columns.tolist())

assert check_df.shape == results_df.shape
assert check_df["cohort"].notna().all()

print("\nSUCCESS: Parquet saved and read back successfully.")

STEP 10B-R4 — OUTPUT VALIDATION
CSV exists: True
Parquet exists: True
CSV size: 11274 bytes
Parquet size: 10903 bytes

Original shape: (60, 12)
Parquet read-back shape: (60, 12)

Cohort values:
cohort
ALL_COHORTS    15
2023           15
2024           15
2025           15
Name: count, dtype: Int64

Unique outcomes: 5
Unique models: 3

Columns:
['stage', 'cohort', 'outcome', 'model', 'players', 'baseline_mae', 'model_mae', 'mae_improvement_pct', 'bootstrap_ci_lower', 'bootstrap_ci_upper', 'probability_improvement', 'bootstrap_draws']

SUCCESS: Parquet saved and read back successfully.


In [34]:

# ============================================================
# STEP 10B-R5 — EXPANDED RIDGE BOOTSTRAP RESULTS AUDIT
# ============================================================

import pandas as pd
from pathlib import Path

RESULTS = (
    ROOT / "reports" / "model_results" /
    "step10b_ridge_expanded_paired_bootstrap.parquet"
)

df = pd.read_parquet(RESULTS)

# 1. Overall results
print("=" * 90)
print("1. POOLED OUT-OF-COHORT BOOTSTRAP RESULTS")
print("=" * 90)

overall = df[df["cohort"] == "ALL_COHORTS"].copy()

overall = overall.sort_values(
    ["outcome", "model"]
)

print(overall[
    [
        "outcome",
        "model",
        "players",
        "baseline_mae",
        "model_mae",
        "mae_improvement_pct",
        "bootstrap_ci_lower",
        "bootstrap_ci_upper",
        "probability_improvement"
    ]
].to_string(index=False))

# 2. Peak speed by cohort
print("\n" + "=" * 90)
print("2. PEAK SPEED — COHORT-SPECIFIC RESULTS")
print("=" * 90)

peak = df[df["outcome"] == "game_peak_speed_raw_median"].copy()

print(peak[
    [
        "cohort",
        "model",
        "players",
        "mae_improvement_pct",
        "bootstrap_ci_lower",
        "bootstrap_ci_upper",
        "probability_improvement"
    ]
].sort_values(["cohort", "model"]).to_string(index=False))

# 3. Statistical evidence classification
print("\n" + "=" * 90)
print("3. CONFIDENCE INTERVAL CLASSIFICATION")
print("=" * 90)

def classify(row):
    if row["bootstrap_ci_lower"] > 0:
        return "ROBUST POSITIVE IMPROVEMENT"
    elif row["bootstrap_ci_upper"] < 0:
        return "ROBUST PERFORMANCE DETERIORATION"
    return "UNCERTAIN — CI CROSSES ZERO"

df["evidence_class"] = df.apply(classify, axis=1)

print(
    df[df["cohort"] == "ALL_COHORTS"][
        ["outcome", "model", "mae_improvement_pct",
         "bootstrap_ci_lower", "bootstrap_ci_upper",
         "evidence_class"]
    ].to_string(index=False)
)

# 4. Save interpretation table
OUT = (
    ROOT / "reports" / "model_results" /
    "step10b_ridge_expanded_bootstrap_interpretation.csv"
)

df.to_csv(OUT, index=False)

print("\nSaved:", OUT)


1. POOLED OUT-OF-COHORT BOOTSTRAP RESULTS
                               outcome         model  players  baseline_mae  model_mae  mae_improvement_pct  bootstrap_ci_lower  bootstrap_ci_upper  probability_improvement
     game_mean_acceleration_raw_median A_TRADITIONAL       66      0.149432   0.150969            -1.028092           -2.987249            0.806584                   0.1356
     game_mean_acceleration_raw_median    B_TRACKING       66      0.149432   0.150183            -0.502593           -2.232467            1.077454                   0.2762
     game_mean_acceleration_raw_median    C_COMBINED       66      0.149432   0.149437            -0.003374           -0.011997            0.004733                   0.2232
            game_mean_speed_raw_median A_TRADITIONAL       66      0.373679   0.376053            -0.635304           -6.020772            3.883351                   0.3992
            game_mean_speed_raw_median    B_TRACKING       66      0.373679   0.374711       

In [35]:

# ============================================================
# STEP 10B-R5 — PREDICTION INTEGRITY AND COHORT AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

PRED_PATH = (
    ROOT / "reports" / "model_results" /
    "step10b_ridge_expanded_predictions.parquet"
)

BOOT_PATH = (
    ROOT / "reports" / "model_results" /
    "step10b_ridge_expanded_paired_bootstrap.parquet"
)

pred = pd.read_parquet(PRED_PATH)
boot = pd.read_parquet(BOOT_PATH)

print("=" * 85)
print("1. PREDICTION DATA STRUCTURE")
print("=" * 85)

print("Shape:", pred.shape)
print("Columns:", pred.columns.tolist())
print(pred.head())

# Resolve expected column names
def find_col(candidates, columns):
    for c in candidates:
        if c in columns:
            return c
    return None

player_col = find_col(["nfl_id", "player_id"], pred.columns)
cohort_col = find_col(
    ["held_out_cohort", "cohort", "draft_year"], pred.columns
)
outcome_col = find_col(["outcome"], pred.columns)
model_col = find_col(["model"], pred.columns)
actual_col = find_col(["actual", "y_true"], pred.columns)
pred_col = find_col(["predicted", "prediction", "y_pred"], pred.columns)

required = {
    "player": player_col,
    "cohort": cohort_col,
    "outcome": outcome_col,
    "model": model_col,
    "actual": actual_col,
    "prediction": pred_col
}

print("\nResolved columns:", required)

if any(v is None for v in required.values()):
    raise ValueError(
        "Could not resolve all prediction columns. "
        "Inspect the printed column list before proceeding."
    )

# 2. Player-level prediction integrity
print("\n" + "=" * 85)
print("2. PREDICTION DISTRIBUTION BY COHORT, OUTCOME AND MODEL")
print("=" * 85)

summary = (
    pred.groupby([cohort_col, outcome_col, model_col])
    .agg(
        players=(player_col, "nunique"),
        actual_mean=(actual_col, "mean"),
        prediction_mean=(pred_col, "mean"),
        prediction_sd=(pred_col, "std"),
        prediction_min=(pred_col, "min"),
        prediction_max=(pred_col, "max"),
        actual_sd=(actual_col, "std"),
        mae=(pred_col, lambda x: np.nan)
    )
    .reset_index()
)

# Calculate MAE explicitly
mae_table = (
    pred.assign(
        absolute_error=lambda d: (
            d[actual_col] - d[pred_col]
        ).abs()
    )
    .groupby([cohort_col, outcome_col, model_col])
    .agg(mae=("absolute_error", "mean"))
    .reset_index()
)

summary = summary.drop(columns=["mae"]).merge(
    mae_table,
    on=[cohort_col, outcome_col, model_col],
    how="left"
)

print(
    summary[
        summary[outcome_col] == "game_peak_speed_raw_median"
    ].sort_values([cohort_col, model_col]).to_string(index=False)
)

# 3. Compare model errors against the reported bootstrap table
print("\n" + "=" * 85)
print("3. PEAK-SPEED COHORT MAE RECONCILIATION")
print("=" * 85)

peak_mae = mae_table[
    mae_table[outcome_col] == "game_peak_speed_raw_median"
].copy()

baseline_rows = peak_mae[
    peak_mae[model_col].astype(str).str.upper().str.contains("BASELINE")
]

model_rows = peak_mae[
    ~peak_mae[model_col].astype(str).str.upper().str.contains("BASELINE")
]

print("Baseline rows found:", len(baseline_rows))
print("Model rows found:", len(model_rows))

print("\nPeak-speed MAE:")
print(peak_mae.sort_values([cohort_col, model_col]).to_string(index=False))

# 4. Bootstrap cohort results
print("\n" + "=" * 85)
print("4. BOOTSTRAP COHORT RESULTS")
print("=" * 85)

print(
    boot[
        (boot["outcome"] == "game_peak_speed_raw_median") &
        (boot["cohort"].astype(str) != "ALL_COHORTS")
    ][
        [
            "cohort", "model", "players",
            "baseline_mae", "model_mae",
            "mae_improvement_pct",
            "bootstrap_ci_lower",
            "bootstrap_ci_upper",
            "probability_improvement"
        ]
    ].sort_values(["cohort", "model"]).to_string(index=False)
)

# 5. Identify near-constant model predictions
print("\n" + "=" * 85)
print("5. NEAR-CONSTANT PREDICTION WARNING")
print("=" * 85)

peak_summary = summary[
    summary[outcome_col] == "game_peak_speed_raw_median"
].copy()

for _, row in peak_summary.iterrows():
    ratio = (
        row["prediction_sd"] / row["actual_sd"]
        if pd.notna(row["actual_sd"]) and row["actual_sd"] > 0
        else np.nan
    )

    warning = (
        "POSSIBLE COLLAPSE"
        if pd.notna(ratio) and ratio < 0.05
        else "CHECK"
        if pd.notna(ratio) and ratio < 0.20
        else "VARIATION PRESENT"
    )

    print(
        row[cohort_col],
        row[model_col],
        "| prediction SD:", round(row["prediction_sd"], 6),
        "| actual SD:", round(row["actual_sd"], 6),
        "| SD ratio:", round(ratio, 4) if pd.notna(ratio) else None,
        "|", warning
    )

print("\nAUDIT COMPLETE.")


1. PREDICTION DATA STRUCTURE
Shape: (2320, 8)
Columns: ['held_out_cohort', 'outcome', 'model', 'nfl_id', 'season', 'actual', 'predicted', 'best_alpha']
   held_out_cohort                     outcome     model nfl_id  season  \
0             2023  game_mean_speed_raw_median  BASELINE  55884    2023   
1             2023  game_mean_speed_raw_median  BASELINE  55884    2024   
2             2023  game_mean_speed_raw_median  BASELINE  55884    2025   
3             2023  game_mean_speed_raw_median  BASELINE  55885    2023   
4             2023  game_mean_speed_raw_median  BASELINE  55885    2024   

     actual  predicted  best_alpha  
0  3.665263   4.104065         NaN  
1  3.711991   4.104065         NaN  
2  4.159833   4.104065         NaN  
3  4.124947   4.104065         NaN  
4  3.940435   4.104065         NaN  

Resolved columns: {'player': 'nfl_id', 'cohort': 'held_out_cohort', 'outcome': 'outcome', 'model': 'model', 'actual': 'actual', 'prediction': 'predicted'}

2. PREDICTION DIST

In [36]:

# ============================================================
# STEP 10B-R6 — PLAYER-LEVEL MAE RECONCILIATION
# ============================================================

import pandas as pd
import numpy as np

PRED_PATH = (
    ROOT / "reports" / "model_results" /
    "step10b_ridge_expanded_predictions.parquet"
)

BOOT_PATH = (
    ROOT / "reports" / "model_results" /
    "step10b_ridge_expanded_paired_bootstrap.parquet"
)

pred = pd.read_parquet(PRED_PATH)
boot = pd.read_parquet(BOOT_PATH)

# ------------------------------------------------------------
# 1. Aggregate player-season predictions to one row per player
# ------------------------------------------------------------

player_pred = (
    pred.groupby(
        ["held_out_cohort", "outcome", "nfl_id", "model"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        seasons=("season", "nunique"),
        player_season_rows=("season", "size")
    )
)

player_pred["absolute_error"] = (
    player_pred["actual"] - player_pred["predicted"]
).abs()

# ------------------------------------------------------------
# 2. Recalculate player-level MAE
# ------------------------------------------------------------

player_mae = (
    player_pred.groupby(
        ["held_out_cohort", "outcome", "model"],
        as_index=False
    )
    .agg(
        players=("nfl_id", "nunique"),
        player_level_mae=("absolute_error", "mean"),
        prediction_sd=("predicted", "std"),
        actual_sd=("actual", "std")
    )
)

# ------------------------------------------------------------
# 3. Reconcile against bootstrap MAE
# ------------------------------------------------------------

cohort_boot = boot[
    boot["cohort"].astype(str) != "ALL_COHORTS"
].copy()

cohort_boot["held_out_cohort"] = (
    cohort_boot["cohort"].astype(str)
)

player_mae["held_out_cohort"] = (
    player_mae["held_out_cohort"].astype(str)
)

reconciliation = player_mae.merge(
    cohort_boot[
        [
            "held_out_cohort",
            "outcome",
            "model",
            "players",
            "model_mae",
            "baseline_mae"
        ]
    ],
    on=["held_out_cohort", "outcome", "model"],
    how="left",
    suffixes=("_recalculated", "_bootstrap")
)

reconciliation["mae_difference"] = (
    reconciliation["player_level_mae"] -
    reconciliation["model_mae"]
)

# ------------------------------------------------------------
# 4. Peak-speed reconciliation
# ------------------------------------------------------------

print("=" * 90)
print("1. PEAK-SPEED PLAYER-LEVEL MAE RECONCILIATION")
print("=" * 90)

peak = reconciliation[
    reconciliation["outcome"] == "game_peak_speed_raw_median"
].copy()

print(
    peak[
        [
            "held_out_cohort",
            "model",
            "players_recalculated",
            "player_level_mae",
            "model_mae",
            "baseline_mae",
            "mae_difference"
        ]
    ].sort_values(
        ["held_out_cohort", "model"]
    ).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Recalculate player-level improvements
# ------------------------------------------------------------

baseline = player_mae[
    player_mae["model"] == "BASELINE"
][
    ["held_out_cohort", "outcome", "player_level_mae"]
].rename(
    columns={"player_level_mae": "baseline_player_mae"}
)

comparison = player_mae[
    player_mae["model"] != "BASELINE"
].merge(
    baseline,
    on=["held_out_cohort", "outcome"],
    how="left"
)

comparison["improvement_pct"] = (
    (
        comparison["baseline_player_mae"] -
        comparison["player_level_mae"]
    ) /
    comparison["baseline_player_mae"]
) * 100

print("\n" + "=" * 90)
print("2. PLAYER-LEVEL PEAK-SPEED IMPROVEMENTS")
print("=" * 90)

print(
    comparison[
        comparison["outcome"] == "game_peak_speed_raw_median"
    ][
        [
            "held_out_cohort",
            "model",
            "players",
            "baseline_player_mae",
            "player_level_mae",
            "improvement_pct"
        ]
    ].sort_values(
        ["held_out_cohort", "model"]
    ).to_string(index=False)
)

# ------------------------------------------------------------
# 6. Check player-season exposure and aggregation
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("3. PLAYER-SEASON EXPOSURE")
print("=" * 90)

exposure = (
    pred[
        pred["outcome"] == "game_peak_speed_raw_median"
    ]
    .groupby(["held_out_cohort", "nfl_id"])
    .agg(
        seasons=("season", "nunique"),
        rows=("season", "size")
    )
    .reset_index()
)

print(
    exposure.groupby("held_out_cohort")["seasons"]
    .value_counts()
    .sort_index()
)

print("\nSeason count summary:")
print(
    exposure.groupby("held_out_cohort")["seasons"]
    .describe()
)

# ------------------------------------------------------------
# 7. Save audit outputs
# ------------------------------------------------------------

OUT_DIR = ROOT / "reports" / "model_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

player_pred.to_parquet(
    OUT_DIR / "step10b_r6_player_level_predictions.parquet",
    index=False
)

reconciliation.to_csv(
    OUT_DIR / "step10b_r6_mae_reconciliation.csv",
    index=False
)

comparison.to_csv(
    OUT_DIR / "step10b_r6_player_level_improvements.csv",
    index=False
)

print("\nR6 outputs saved successfully.")
print("Player-level predictions:", len(player_pred))
print("Reconciliation rows:", len(reconciliation))


1. PEAK-SPEED PLAYER-LEVEL MAE RECONCILIATION
held_out_cohort         model  players_recalculated  player_level_mae  model_mae  baseline_mae  mae_difference
           2023 A_TRADITIONAL                    28          0.443830   0.443830      0.460082    0.000000e+00
           2023      BASELINE                    28          0.460082        NaN           NaN             NaN
           2023    B_TRACKING                    28          0.460064   0.460064      0.460082    0.000000e+00
           2023    C_COMBINED                    28          0.460030   0.460030      0.460082    0.000000e+00
           2024 A_TRADITIONAL                    21          0.434642   0.434642      0.435974    1.110223e-16
           2024      BASELINE                    21          0.435974        NaN           NaN             NaN
           2024    B_TRACKING                    21          0.415661   0.415661      0.435974    0.000000e+00
           2024    C_COMBINED                    21          0.416

In [37]:

# ============================================================
# STEP 10B-R7 — PLAYER INFLUENCE AND ROBUSTNESS AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

PRED_PATH = (
    ROOT / "reports" / "model_results" /
    "step10b_ridge_expanded_predictions.parquet"
)

OUT_DIR = ROOT / "reports" / "model_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pred = pd.read_parquet(PRED_PATH)

# ------------------------------------------------------------
# 1. Aggregate player-season predictions to player level
# ------------------------------------------------------------

player_pred = (
    pred.groupby(
        ["held_out_cohort", "outcome", "model", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        seasons_observed=("season", "nunique")
    )
)

# Validate actual outcome is consistent across models
actual_check = (
    player_pred.groupby(
        ["held_out_cohort", "outcome", "nfl_id"]
    )["actual"].nunique()
)

assert actual_check.max() == 1, (
    "Actual outcomes differ across models for the same player."
)

print("Player-level prediction rows:", len(player_pred))
print("Unique players:", player_pred["nfl_id"].nunique())

# ------------------------------------------------------------
# 2. Function to calculate player-level model improvement
# ------------------------------------------------------------

def calculate_improvement(data, model):
    wide = data.pivot(
        index="nfl_id",
        columns="model",
        values=["actual", "predicted"]
    )

    required_models = ["BASELINE", model]

    for m in required_models:
        if ("predicted", m) not in wide.columns:
            return None

    actual = wide[("actual", "BASELINE")]
    baseline_prediction = wide[("predicted", "BASELINE")]
    model_prediction = wide[("predicted", model)]

    valid = (
        actual.notna() &
        baseline_prediction.notna() &
        model_prediction.notna()
    )

    actual = actual[valid]
    baseline_prediction = baseline_prediction[valid]
    model_prediction = model_prediction[valid]

    if len(actual) < 3:
        return None

    baseline_mae = np.mean(
        np.abs(actual - baseline_prediction)
    )

    model_mae = np.mean(
        np.abs(actual - model_prediction)
    )

    improvement = (
        (baseline_mae - model_mae) / baseline_mae * 100
        if baseline_mae > 0 else np.nan
    )

    return {
        "players": len(actual),
        "baseline_mae": baseline_mae,
        "model_mae": model_mae,
        "improvement_pct": improvement
    }


# ------------------------------------------------------------
# 3. Leave-one-player-out analysis
# ------------------------------------------------------------

models = [
    "A_TRADITIONAL",
    "B_TRACKING",
    "C_COMBINED"
]

outcomes = player_pred["outcome"].unique()
cohorts = sorted(
    player_pred["held_out_cohort"].unique()
)

result_rows = []

for cohort in cohorts:
    for outcome in outcomes:

        subset = player_pred[
            (player_pred["held_out_cohort"] == cohort) &
            (player_pred["outcome"] == outcome)
        ].copy()

        for model in models:

            full_result = calculate_improvement(
                subset, model
            )

            if full_result is None:
                continue

            loo_values = []

            for player in subset["nfl_id"].unique():

                reduced = subset[
                    subset["nfl_id"] != player
                ]

                loo_result = calculate_improvement(
                    reduced, model
                )

                if loo_result is not None:
                    loo_values.append({
                        "removed_player": player,
                        "improvement_pct":
                            loo_result["improvement_pct"]
                    })

            loo_df = pd.DataFrame(loo_values)

            if len(loo_df) == 0:
                continue

            result_rows.append({
                "cohort": str(cohort),
                "outcome": outcome,
                "model": model,
                "players": full_result["players"],
                "full_improvement_pct":
                    full_result["improvement_pct"],
                "loo_min_improvement_pct":
                    loo_df["improvement_pct"].min(),
                "loo_median_improvement_pct":
                    loo_df["improvement_pct"].median(),
                "loo_max_improvement_pct":
                    loo_df["improvement_pct"].max(),
                "loo_positive_fraction":
                    (loo_df["improvement_pct"] > 0).mean(),
                "loo_draws": len(loo_df)
            })

# ------------------------------------------------------------
# 4. Pooled player-level influence analysis
# ------------------------------------------------------------

for outcome in outcomes:

    subset = player_pred[
        player_pred["outcome"] == outcome
    ].copy()

    for model in models:

        full_result = calculate_improvement(
            subset, model
        )

        if full_result is None:
            continue

        loo_values = []

        for player in subset["nfl_id"].unique():

            reduced = subset[
                subset["nfl_id"] != player
            ]

            loo_result = calculate_improvement(
                reduced, model
            )

            if loo_result is not None:
                loo_values.append(
                    loo_result["improvement_pct"]
                )

        if len(loo_values) == 0:
            continue

        result_rows.append({
            "cohort": "ALL_COHORTS",
            "outcome": outcome,
            "model": model,
            "players": full_result["players"],
            "full_improvement_pct":
                full_result["improvement_pct"],
            "loo_min_improvement_pct":
                np.min(loo_values),
            "loo_median_improvement_pct":
                np.median(loo_values),
            "loo_max_improvement_pct":
                np.max(loo_values),
            "loo_positive_fraction":
                np.mean(np.array(loo_values) > 0),
            "loo_draws": len(loo_values)
        })

robustness = pd.DataFrame(result_rows)

# ------------------------------------------------------------
# 5. Display peak-speed findings
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("STEP 10B-R7 — PEAK-SPEED LEAVE-ONE-PLAYER-OUT RESULTS")
print("=" * 90)

peak_results = robustness[
    robustness["outcome"] ==
    "game_peak_speed_raw_median"
].copy()

print(
    peak_results.sort_values(
        ["cohort", "model"]
    ).to_string(index=False)
)

# ------------------------------------------------------------
# 6. Flag fragile results
# ------------------------------------------------------------

def robustness_class(row):

    if row["loo_min_improvement_pct"] > 0:
        return "POSITIVE UNDER EVERY PLAYER REMOVAL"

    if row["loo_max_improvement_pct"] < 0:
        return "NEGATIVE UNDER EVERY PLAYER REMOVAL"

    if row["loo_positive_fraction"] >= 0.80:
        return "MOSTLY POSITIVE — SOME PLAYER SENSITIVITY"

    if row["loo_positive_fraction"] <= 0.20:
        return "MOSTLY NEGATIVE"

    return "SENSITIVE / MIXED"

robustness["robustness_class"] = robustness.apply(
    robustness_class,
    axis=1
)

print("\n" + "=" * 90)
print("PEAK-SPEED ROBUSTNESS CLASSIFICATION")
print("=" * 90)

print(
    robustness[
        robustness["outcome"] ==
        "game_peak_speed_raw_median"
    ][
        [
            "cohort",
            "model",
            "full_improvement_pct",
            "loo_min_improvement_pct",
            "loo_max_improvement_pct",
            "loo_positive_fraction",
            "robustness_class"
        ]
    ].sort_values(
        ["cohort", "model"]
    ).to_string(index=False)
)

# ------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------

CSV_PATH = (
    OUT_DIR / "step10b_r7_player_influence_robustness.csv"
)

PARQUET_PATH = (
    OUT_DIR / "step10b_r7_player_influence_robustness.parquet"
)

robustness.to_csv(CSV_PATH, index=False)
robustness.to_parquet(PARQUET_PATH, index=False)

# Read-back validation
check = pd.read_parquet(PARQUET_PATH)

assert check.shape == robustness.shape

print("\n" + "=" * 90)
print("R7 OUTPUT VALIDATION")
print("=" * 90)

print("Rows:", len(check))
print("Columns:", len(check.columns))
print("CSV:", CSV_PATH)
print("Parquet:", PARQUET_PATH)
print("SUCCESS: R7 outputs saved and verified.")


Player-level prediction rows: 1320
Unique players: 66

STEP 10B-R7 — PEAK-SPEED LEAVE-ONE-PLAYER-OUT RESULTS
     cohort                    outcome         model  players  full_improvement_pct  loo_min_improvement_pct  loo_median_improvement_pct  loo_max_improvement_pct  loo_positive_fraction  loo_draws
       2023 game_peak_speed_raw_median A_TRADITIONAL       28              3.532608                 2.433370                    3.474913                 4.850494               1.000000         28
       2023 game_peak_speed_raw_median    B_TRACKING       28              0.003910                 0.003356                    0.003990                 0.004463               1.000000         28
       2023 game_peak_speed_raw_median    C_COMBINED       28              0.011472                 0.009046                    0.011422                 0.013715               1.000000         28
       2024 game_peak_speed_raw_median A_TRADITIONAL       21              0.305631                -3.16884

In [38]:

# ============================================================
# STEP 10B-R8 — PAIRED MODEL COMPARISON
# Does tracking add predictive value beyond traditional metrics?
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Paths and configuration
# ------------------------------------------------------------

PRED_PATH = (
    ROOT / "reports" / "model_results" /
    "step10b_ridge_expanded_predictions.parquet"
)

OUT_DIR = ROOT / "reports" / "model_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_BOOTSTRAP = 5000
RANDOM_SEED = 2027

MODELS = [
    "A_TRADITIONAL",
    "B_TRACKING",
    "C_COMBINED"
]

COMPARISONS = [
    ("A_TRADITIONAL", "B_TRACKING"),
    ("A_TRADITIONAL", "C_COMBINED"),
    ("B_TRACKING", "C_COMBINED")
]

# ------------------------------------------------------------
# 2. Load and aggregate to one record per player
# ------------------------------------------------------------

pred = pd.read_parquet(PRED_PATH)

player_pred = (
    pred.groupby(
        ["held_out_cohort", "outcome", "model", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        seasons=("season", "nunique")
    )
)

player_pred["absolute_error"] = (
    player_pred["actual"] - player_pred["predicted"]
).abs()

print("=" * 85)
print("STEP 10B-R8 — INPUT VALIDATION")
print("=" * 85)

print("Player-level rows:", len(player_pred))
print("Unique players:", player_pred["nfl_id"].nunique())
print("Outcomes:", player_pred["outcome"].nunique())

# Each player must have one draft-cohort assignment per outcome
assert (
    player_pred.groupby(["outcome", "nfl_id"])
    ["held_out_cohort"].nunique().max() == 1
)

# ------------------------------------------------------------
# 3. Paired bootstrap function
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

def paired_bootstrap(data, reference, challenger):

    wide = data.pivot(
        index="nfl_id",
        columns="model",
        values="absolute_error"
    )

    required = [reference, challenger]

    if not all(m in wide.columns for m in required):
        return None

    wide = wide[required].dropna()

    n = len(wide)

    if n < 5:
        return None

    ref_error = wide[reference].to_numpy()
    challenger_error = wide[challenger].to_numpy()

    ref_mae = ref_error.mean()
    challenger_mae = challenger_error.mean()

    # Positive means challenger has lower MAE
    delta = ref_mae - challenger_mae

    improvement_pct = (
        delta / ref_mae * 100
        if ref_mae > 0 else np.nan
    )

    bootstrap_delta = np.empty(N_BOOTSTRAP)
    bootstrap_pct = np.empty(N_BOOTSTRAP)

    for b in range(N_BOOTSTRAP):

        idx = rng.integers(0, n, size=n)

        sampled_ref = ref_error[idx]
        sampled_challenger = challenger_error[idx]

        sampled_ref_mae = sampled_ref.mean()
        sampled_challenger_mae = sampled_challenger.mean()

        bootstrap_delta[b] = (
            sampled_ref_mae - sampled_challenger_mae
        )

        bootstrap_pct[b] = (
            bootstrap_delta[b] / sampled_ref_mae * 100
            if sampled_ref_mae > 0 else np.nan
        )

    return {
        "players": n,
        "reference_mae": ref_mae,
        "challenger_mae": challenger_mae,
        "mae_difference": delta,
        "relative_improvement_pct": improvement_pct,
        "ci_difference_lower": np.percentile(
            bootstrap_delta, 2.5
        ),
        "ci_difference_upper": np.percentile(
            bootstrap_delta, 97.5
        ),
        "ci_improvement_lower": np.nanpercentile(
            bootstrap_pct, 2.5
        ),
        "ci_improvement_upper": np.nanpercentile(
            bootstrap_pct, 97.5
        ),
        "probability_challenger_better": np.mean(
            bootstrap_delta > 0
        ),
        "bootstrap_draws": N_BOOTSTRAP
    }

# ------------------------------------------------------------
# 4. Run comparisons by cohort and pooled
# ------------------------------------------------------------

cohorts = sorted(
    player_pred["held_out_cohort"].unique()
)

group_specs = [
    (str(cohort), player_pred[
        player_pred["held_out_cohort"] == cohort
    ])
    for cohort in cohorts
]

group_specs.append(("ALL_COHORTS", player_pred))

rows = []

for group_name, group_data in group_specs:

    for outcome in sorted(group_data["outcome"].unique()):

        subset = group_data[
            group_data["outcome"] == outcome
        ]

        for reference, challenger in COMPARISONS:

            result = paired_bootstrap(
                subset,
                reference,
                challenger
            )

            if result is None:
                continue

            rows.append({
                "cohort": group_name,
                "outcome": outcome,
                "reference_model": reference,
                "challenger_model": challenger,
                **result
            })

results = pd.DataFrame(rows)

# ------------------------------------------------------------
# 5. Evidence classification
# ------------------------------------------------------------

def classify_comparison(row):

    if row["ci_difference_lower"] > 0:
        return "CHALLENGER SIGNIFICANTLY BETTER"

    if row["ci_difference_upper"] < 0:
        return "REFERENCE SIGNIFICANTLY BETTER"

    return "NO CLEAR DIFFERENCE"

results["evidence"] = results.apply(
    classify_comparison,
    axis=1
)

# ------------------------------------------------------------
# 6. Display peak-speed comparisons
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PEAK-SPEED — DIRECT PAIRED MODEL COMPARISONS")
print("=" * 100)

peak = results[
    results["outcome"] ==
    "game_peak_speed_raw_median"
].copy()

display_columns = [
    "cohort",
    "reference_model",
    "challenger_model",
    "players",
    "reference_mae",
    "challenger_mae",
    "relative_improvement_pct",
    "ci_improvement_lower",
    "ci_improvement_upper",
    "probability_challenger_better",
    "evidence"
]

print(
    peak[display_columns]
    .sort_values(
        ["cohort", "reference_model", "challenger_model"]
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. Pooled comparison across all five outcomes
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("ALL OUTCOMES — POOLED DIRECT COMPARISONS")
print("=" * 100)

pooled = results[
    results["cohort"] == "ALL_COHORTS"
]

print(
    pooled[display_columns]
    .sort_values(
        ["outcome", "reference_model", "challenger_model"]
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. Save and validate
# ------------------------------------------------------------

CSV_PATH = (
    OUT_DIR / "step10b_r8_paired_model_comparisons.csv"
)

PARQUET_PATH = (
    OUT_DIR / "step10b_r8_paired_model_comparisons.parquet"
)

results.to_csv(CSV_PATH, index=False)
results.to_parquet(PARQUET_PATH, index=False)

check = pd.read_parquet(PARQUET_PATH)

assert check.shape == results.shape
assert check["cohort"].notna().all()

print("\n" + "=" * 85)
print("STEP 10B-R8 — SAVE VALIDATION")
print("=" * 85)

print("Rows:", len(check))
print("Columns:", len(check.columns))
print("CSV size:", CSV_PATH.stat().st_size)
print("Parquet size:", PARQUET_PATH.stat().st_size)
print("SUCCESS: Paired comparisons saved and verified.")


STEP 10B-R8 — INPUT VALIDATION
Player-level rows: 1320
Unique players: 66
Outcomes: 5

PEAK-SPEED — DIRECT PAIRED MODEL COMPARISONS
     cohort reference_model challenger_model  players  reference_mae  challenger_mae  relative_improvement_pct  ci_improvement_lower  ci_improvement_upper  probability_challenger_better            evidence
       2023   A_TRADITIONAL       B_TRACKING       28       0.443830        0.460064                 -3.657917             -9.573280              2.413076                         0.1030 NO CLEAR DIFFERENCE
       2023   A_TRADITIONAL       C_COMBINED       28       0.443830        0.460030                 -3.650079             -9.442346              2.182496                         0.1042 NO CLEAR DIFFERENCE
       2023      B_TRACKING       C_COMBINED       28       0.460064        0.460030                  0.007562             -0.002410              0.017558                         0.9302 NO CLEAR DIFFERENCE
       2024   A_TRADITIONAL       B_TRACKING

KeyError: 'outcome'

In [39]:

# ============================================================
# STEP 10B-R8 — CORRECTED DISPLAY AND FINAL SAVE
# ============================================================

import pandas as pd
from pathlib import Path

OUT_DIR = ROOT / "reports" / "model_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Include outcome in display columns
display_columns = [
    "cohort",
    "outcome",
    "reference_model",
    "challenger_model",
    "players",
    "reference_mae",
    "challenger_mae",
    "relative_improvement_pct",
    "ci_improvement_lower",
    "ci_improvement_upper",
    "probability_challenger_better",
    "evidence"
]

# 1. Pooled comparisons across all outcomes
print("=" * 100)
print("STEP 10B-R8 — POOLED DIRECT MODEL COMPARISONS")
print("=" * 100)

pooled = results[
    results["cohort"].astype(str) == "ALL_COHORTS"
].copy()

print(
    pooled[display_columns]
    .sort_values(
        ["outcome", "reference_model", "challenger_model"]
    )
    .to_string(index=False)
)

# 2. Tracking incremental value over traditional metrics
print("\n" + "=" * 100)
print("TRACKING INCREMENTAL VALUE — C_COMBINED VS A_TRADITIONAL")
print("=" * 100)

incremental = results[
    (results["cohort"].astype(str) == "ALL_COHORTS") &
    (results["reference_model"] == "A_TRADITIONAL") &
    (results["challenger_model"] == "C_COMBINED")
].copy()

print(
    incremental[
        [
            "outcome",
            "players",
            "reference_mae",
            "challenger_mae",
            "relative_improvement_pct",
            "ci_improvement_lower",
            "ci_improvement_upper",
            "probability_challenger_better",
            "evidence"
        ]
    ].to_string(index=False)
)

# 3. Save results
CSV_PATH = OUT_DIR / "step10b_r8_paired_model_comparisons.csv"
PARQUET_PATH = OUT_DIR / "step10b_r8_paired_model_comparisons.parquet"

results["cohort"] = results["cohort"].astype("string")

results.to_csv(CSV_PATH, index=False)
results.to_parquet(PARQUET_PATH, index=False)

# 4. Read-back validation
check = pd.read_parquet(PARQUET_PATH)

assert check.shape == results.shape
assert check["outcome"].nunique() == 5
assert check["cohort"].nunique() == 4

print("\n" + "=" * 85)
print("R8 SAVE VALIDATION")
print("=" * 85)

print("Rows:", len(check))
print("Columns:", len(check.columns))
print("Cohorts:", check["cohort"].unique().tolist())
print("Outcomes:", check["outcome"].nunique())
print("CSV bytes:", CSV_PATH.stat().st_size)
print("Parquet bytes:", PARQUET_PATH.stat().st_size)
print("SUCCESS: R8 results saved and verified.")


STEP 10B-R8 — POOLED DIRECT MODEL COMPARISONS
     cohort                                outcome reference_model challenger_model  players  reference_mae  challenger_mae  relative_improvement_pct  ci_improvement_lower  ci_improvement_upper  probability_challenger_better            evidence
ALL_COHORTS      game_mean_acceleration_raw_median   A_TRADITIONAL       B_TRACKING       66       0.150969        0.150183                  0.520152             -1.948120              3.045066                         0.6660 NO CLEAR DIFFERENCE
ALL_COHORTS      game_mean_acceleration_raw_median   A_TRADITIONAL       C_COMBINED       66       0.150969        0.149437                  1.014290             -0.812505              2.929530                         0.8622 NO CLEAR DIFFERENCE
ALL_COHORTS      game_mean_acceleration_raw_median      B_TRACKING       C_COMBINED       66       0.150183        0.149437                  0.496722             -1.087034              2.133940                         0

In [40]:

# ============================================================
# STEP 11A — TRADITIONAL COMBINE FEATURE ABLATION
# Outcome: In-game receiver peak speed
# Validation: Leave-one-draft-cohort-out
# Inner CV: GroupKFold by nfl_id
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, GroupKFold

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DATA_PATH = (
    ROOT / "data" / "features" /
    "wr_modeling_dataset_primary_10plus.parquet"
)

OUT_DIR = ROOT / "reports" / "model_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME = "game_peak_speed_raw_median"
GROUP = "nfl_id"
COHORT = "draft_year"

ALPHAS = [
    0.001, 0.01, 0.1, 1, 10,
    100, 1000, 10000, 100000
]

SPECIFICATIONS = {
    "A0_FULL": ["forty", "vertical", "broad_jump"],
    "A1_NO_FORTY": ["vertical", "broad_jump"],
    "A2_NO_VERTICAL": ["forty", "broad_jump"],
    "A3_NO_BROAD": ["forty", "vertical"]
}

# ------------------------------------------------------------
# 2. Load and validate data
# ------------------------------------------------------------

df = pd.read_parquet(DATA_PATH).copy()

required = [
    OUTCOME, GROUP, COHORT,
    "forty", "vertical", "broad_jump"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in [OUTCOME, "forty", "vertical", "broad_jump"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required).copy()

assert df[GROUP].notna().all()
assert df[COHORT].nunique() == 3

print("=" * 85)
print("STEP 11A — DATA VALIDATION")
print("=" * 85)

print("Rows:", len(df))
print("Unique players:", df[GROUP].nunique())
print("Cohorts:", sorted(df[COHORT].unique().tolist()))

print("\nCohort distribution:")
print(
    df.groupby(COHORT)
    .agg(
        rows=(GROUP, "size"),
        players=(GROUP, "nunique")
    )
)

# ------------------------------------------------------------
# 3. Nested grouped Ridge modelling
# ------------------------------------------------------------

prediction_rows = []
alpha_rows = []
fold_rows = []

cohorts = sorted(df[COHORT].unique())

for heldout in cohorts:

    train = df[df[COHORT] != heldout].copy()
    test = df[df[COHORT] == heldout].copy()

    train_players = train[GROUP].nunique()
    test_players = test[GROUP].nunique()

    overlap = set(train[GROUP]) & set(test[GROUP])

    assert len(overlap) == 0

    inner_splits = min(4, train_players)

    if inner_splits < 2:
        raise ValueError("Insufficient players for grouped inner CV.")

    inner_cv = GroupKFold(n_splits=inner_splits)

    for spec_name, features in SPECIFICATIONS.items():

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge())
        ])

        search = GridSearchCV(
            estimator=pipeline,
            param_grid={"ridge__alpha": ALPHAS},
            scoring="neg_mean_absolute_error",
            cv=inner_cv,
            refit=True,
            n_jobs=-1
        )

        search.fit(
            train[features],
            train[OUTCOME],
            groups=train[GROUP]
        )

        predictions = search.predict(test[features])

        for i, (_, row) in enumerate(test.iterrows()):

            prediction_rows.append({
                "held_out_cohort": heldout,
                "specification": spec_name,
                "nfl_id": row[GROUP],
                "season": row["season"],
                "actual": row[OUTCOME],
                "predicted": predictions[i],
                "best_alpha": search.best_params_[
                    "ridge__alpha"
                ]
            })

        alpha_rows.append({
            "held_out_cohort": heldout,
            "specification": spec_name,
            "best_alpha": search.best_params_[
                "ridge__alpha"
            ],
            "inner_best_mae": -search.best_score_,
            "train_players": train_players,
            "test_players": test_players
        })

        fold_rows.append({
            "held_out_cohort": heldout,
            "train_rows": len(train),
            "test_rows": len(test),
            "train_players": train_players,
            "test_players": test_players,
            "player_overlap": len(overlap),
            "specification": spec_name
        })

predictions_df = pd.DataFrame(prediction_rows)
alphas_df = pd.DataFrame(alpha_rows)
folds_df = pd.DataFrame(fold_rows)

# ------------------------------------------------------------
# 4. Aggregate to player level
# ------------------------------------------------------------

player_predictions = (
    predictions_df.groupby(
        ["held_out_cohort", "specification", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        seasons=("season", "nunique")
    )
)

player_predictions["absolute_error"] = (
    player_predictions["actual"] -
    player_predictions["predicted"]
).abs()

# ------------------------------------------------------------
# 5. Calculate cohort and pooled MAE
# ------------------------------------------------------------

metric_rows = []

for cohort in cohorts + ["ALL_COHORTS"]:

    subset = player_predictions.copy()

    if cohort != "ALL_COHORTS":
        subset = subset[
            subset["held_out_cohort"] == cohort
        ]

    for spec in SPECIFICATIONS:

        model_data = subset[
            subset["specification"] == spec
        ]

        metric_rows.append({
            "cohort": str(cohort),
            "specification": spec,
            "players": model_data["nfl_id"].nunique(),
            "mae": model_data["absolute_error"].mean()
        })

metrics = pd.DataFrame(metric_rows)

# ------------------------------------------------------------
# 6. Paired player bootstrap against full model
# ------------------------------------------------------------

rng = np.random.default_rng(202711)

bootstrap_rows = []

for cohort in cohorts + ["ALL_COHORTS"]:

    subset = player_predictions.copy()

    if cohort != "ALL_COHORTS":
        subset = subset[
            subset["held_out_cohort"] == cohort
        ]

    wide = subset.pivot(
        index="nfl_id",
        columns="specification",
        values="absolute_error"
    )

    for ablated in [
        "A1_NO_FORTY",
        "A2_NO_VERTICAL",
        "A3_NO_BROAD"
    ]:

        paired = wide[
            ["A0_FULL", ablated]
        ].dropna()

        full_errors = paired["A0_FULL"].to_numpy()
        ablated_errors = paired[ablated].to_numpy()

        n = len(paired)

        full_mae = full_errors.mean()
        ablated_mae = ablated_errors.mean()

        # Positive = full model is better
        contribution = (
            (ablated_mae - full_mae) /
            ablated_mae * 100
            if ablated_mae > 0 else np.nan
        )

        draws = []

        for _ in range(5000):
            idx = rng.integers(0, n, size=n)

            f = full_errors[idx].mean()
            a = ablated_errors[idx].mean()

            draws.append(
                (a - f) / a * 100 if a > 0 else np.nan
            )

        draws = np.asarray(draws)
        draws = draws[np.isfinite(draws)]

        bootstrap_rows.append({
            "cohort": str(cohort),
            "full_model": "A0_FULL",
            "ablated_model": ablated,
            "players": n,
            "full_mae": full_mae,
            "ablated_mae": ablated_mae,
            "feature_contribution_pct": contribution,
            "ci_lower": np.percentile(draws, 2.5),
            "ci_upper": np.percentile(draws, 97.5),
            "probability_full_model_better": np.mean(
                draws > 0
            )
        })

bootstrap_results = pd.DataFrame(bootstrap_rows)

# ------------------------------------------------------------
# 7. Display primary findings
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("STEP 11A — POOLED FEATURE ABLATION PERFORMANCE")
print("=" * 95)

print(
    metrics[
        metrics["cohort"] == "ALL_COHORTS"
    ].sort_values("mae").to_string(index=False)
)

print("\n" + "=" * 95)
print("STEP 11A — PAIRED FEATURE CONTRIBUTION")
print("=" * 95)

print(
    bootstrap_results[
        bootstrap_results["cohort"] == "ALL_COHORTS"
    ].to_string(index=False)
)

print("\n" + "=" * 95)
print("STEP 11A — COHORT-SPECIFIC PERFORMANCE")
print("=" * 95)

print(
    metrics[
        metrics["cohort"] != "ALL_COHORTS"
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 8. Save all outputs
# ------------------------------------------------------------

outputs = {
    "step11a_predictions": predictions_df,
    "step11a_player_predictions": player_predictions,
    "step11a_metrics": metrics,
    "step11a_selected_alphas": alphas_df,
    "step11a_fold_audit": folds_df,
    "step11a_feature_bootstrap": bootstrap_results
}

for name, data in outputs.items():

    csv_path = OUT_DIR / f"{name}.csv"
    parquet_path = OUT_DIR / f"{name}.parquet"

    data.to_csv(csv_path, index=False)
    data.to_parquet(parquet_path, index=False)

    check = pd.read_parquet(parquet_path)

    assert check.shape == data.shape

print("\n" + "=" * 85)
print("STEP 11A — SAVE VALIDATION")
print("=" * 85)

for name, data in outputs.items():
    print(name, data.shape)

print("SUCCESS: All Step 11A outputs saved and verified.")


STEP 11A — DATA VALIDATION
Rows: 95
Unique players: 55
Cohorts: [2023, 2024, 2025]

Cohort distribution:
            rows  players
draft_year               
2023          47       21
2024          34       20
2025          14       14

STEP 11A — POOLED FEATURE ABLATION PERFORMANCE
     cohort  specification  players      mae
ALL_COHORTS    A3_NO_BROAD       55 0.402074
ALL_COHORTS        A0_FULL       55 0.408552
ALL_COHORTS A2_NO_VERTICAL       55 0.427953
ALL_COHORTS    A1_NO_FORTY       55 0.431521

STEP 11A — PAIRED FEATURE CONTRIBUTION
     cohort full_model  ablated_model  players  full_mae  ablated_mae  feature_contribution_pct  ci_lower  ci_upper  probability_full_model_better
ALL_COHORTS    A0_FULL    A1_NO_FORTY       55  0.408552     0.431521                  5.322710 -9.065057 16.529141                         0.7846
ALL_COHORTS    A0_FULL A2_NO_VERTICAL       55  0.408552     0.427953                  4.533411 -1.985630 10.514028                         0.9118
ALL_COHORTS

In [41]:

# ============================================================
# STEP 11B — SAMPLE CONSISTENCY AND MISSINGNESS AUDIT
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = (
    ROOT / "data" / "features" /
    "wr_modeling_dataset_primary_10plus.parquet"
)

OUT_DIR = ROOT / "reports" / "data_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_audit = pd.read_parquet(DATA_PATH).copy()

FEATURES = ["forty", "vertical", "broad_jump"]
OUTCOME = "game_peak_speed_raw_median"
GROUP = "nfl_id"
COHORT = "draft_year"

required = FEATURES + [OUTCOME, GROUP, COHORT]

missing_columns = [
    col for col in required if col not in df_audit.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

for col in FEATURES + [OUTCOME]:
    df_audit[col] = pd.to_numeric(
        df_audit[col], errors="coerce"
    )

# ------------------------------------------------------------
# 1. Original dataset size
# ------------------------------------------------------------

print("=" * 90)
print("1. ORIGINAL DATASET")
print("=" * 90)

print("Rows:", len(df_audit))
print("Unique players:", df_audit[GROUP].nunique())

print(
    df_audit.groupby(COHORT)
    .agg(
        rows=(GROUP, "size"),
        players=(GROUP, "nunique")
    )
)

# ------------------------------------------------------------
# 2. Missingness by predictor and outcome
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("2. MISSINGNESS BY VARIABLE")
print("=" * 90)

missing_summary = pd.DataFrame({
    "variable": required,
    "missing_rows": [
        df_audit[c].isna().sum() for c in required
    ],
    "missing_pct": [
        df_audit[c].isna().mean() * 100 for c in required
    ]
})

print(missing_summary.to_string(index=False))

# ------------------------------------------------------------
# 3. Complete-case exclusions
# ------------------------------------------------------------

df_audit["complete_predictors"] = (
    df_audit[FEATURES].notna().all(axis=1)
)

df_audit["complete_model"] = (
    df_audit[required].notna().all(axis=1)
)

print("\n" + "=" * 90)
print("3. COMPLETE-CASE COUNTS")
print("=" * 90)

print("Original rows:", len(df_audit))
print(
    "Rows with all three predictors:",
    df_audit["complete_predictors"].sum()
)
print(
    "Rows with all predictors and outcome:",
    df_audit["complete_model"].sum()
)
print(
    "Rows excluded:",
    (~df_audit["complete_model"]).sum()
)

# ------------------------------------------------------------
# 4. Missingness combinations
# ------------------------------------------------------------

for col in FEATURES:
    df_audit[f"missing_{col}"] = df_audit[col].isna()

pattern_columns = [f"missing_{c}" for c in FEATURES]

patterns = (
    df_audit.groupby(pattern_columns, dropna=False)
    .agg(
        rows=(GROUP, "size"),
        players=(GROUP, "nunique")
    )
    .reset_index()
)

print("\n" + "=" * 90)
print("4. MISSINGNESS PATTERNS")
print("=" * 90)

print(patterns.to_string(index=False))

# ------------------------------------------------------------
# 5. Cohort-specific sample loss
# ------------------------------------------------------------

cohort_audit = (
    df_audit.groupby(COHORT)
    .agg(
        original_rows=(GROUP, "size"),
        original_players=(GROUP, "nunique"),
        complete_case_rows=("complete_model", "sum")
    )
    .reset_index()
)

complete_players = (
    df_audit[df_audit["complete_model"]]
    .groupby(COHORT)[GROUP]
    .nunique()
    .rename("complete_case_players")
)

cohort_audit = cohort_audit.merge(
    complete_players,
    on=COHORT,
    how="left"
)

cohort_audit["excluded_rows"] = (
    cohort_audit["original_rows"] -
    cohort_audit["complete_case_rows"]
)

cohort_audit["excluded_pct"] = (
    cohort_audit["excluded_rows"] /
    cohort_audit["original_rows"] * 100
)

print("\n" + "=" * 90)
print("5. COHORT SAMPLE REDUCTION")
print("=" * 90)

print(cohort_audit.to_string(index=False))

# ------------------------------------------------------------
# 6. Identify excluded players
# ------------------------------------------------------------

excluded = df_audit[
    ~df_audit["complete_model"]
].copy()

excluded_players = (
    excluded.groupby([COHORT, GROUP])
    .agg(
        excluded_player_seasons=(GROUP, "size"),
        missing_forty=("forty", lambda x: x.isna().sum()),
        missing_vertical=("vertical", lambda x: x.isna().sum()),
        missing_broad_jump=("broad_jump", lambda x: x.isna().sum()),
        missing_outcome=(OUTCOME, lambda x: x.isna().sum())
    )
    .reset_index()
)

print("\n" + "=" * 90)
print("6. EXCLUDED PLAYER SUMMARY")
print("=" * 90)

print("Excluded player-seasons:", len(excluded))
print("Excluded unique players:", excluded[GROUP].nunique())

print(excluded_players.to_string(index=False))

# ------------------------------------------------------------
# 7. Save audit outputs
# ------------------------------------------------------------

missing_summary.to_csv(
    OUT_DIR / "step11b_missingness_by_variable.csv",
    index=False
)

patterns.to_csv(
    OUT_DIR / "step11b_missingness_patterns.csv",
    index=False
)

cohort_audit.to_csv(
    OUT_DIR / "step11b_cohort_sample_reduction.csv",
    index=False
)

excluded_players.to_csv(
    OUT_DIR / "step11b_excluded_players.csv",
    index=False
)

print("\nAudit files saved to:", OUT_DIR)


1. ORIGINAL DATASET
Rows: 116
Unique players: 66
            rows  players
draft_year               
2023          64       28
2024          35       21
2025          17       17

2. MISSINGNESS BY VARIABLE
                  variable  missing_rows  missing_pct
                     forty            12    10.344828
                  vertical            14    12.068966
                broad_jump            12    10.344828
game_peak_speed_raw_median             0     0.000000
                    nfl_id             0     0.000000
                draft_year             0     0.000000

3. COMPLETE-CASE COUNTS
Original rows: 116
Rows with all three predictors: 95
Rows with all predictors and outcome: 95
Rows excluded: 21

4. MISSINGNESS PATTERNS
 missing_forty  missing_vertical  missing_broad_jump  rows  players
         False             False               False    95       55
         False              True               False     2        1
         False              True                

In [43]:

# ============================================================
# STEP 11C — CORRECTED SAMPLE RECONCILIATION
# ============================================================

import pandas as pd
import numpy as np

MODEL_PATH = (
    ROOT / "reports" / "model_results" /
    "step10b_ridge_expanded_predictions.parquet"
)

ABLATION_PATH = (
    ROOT / "reports" / "model_results" /
    "step11a_player_predictions.parquet"
)

OUT_DIR = ROOT / "reports" / "data_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load
# ------------------------------------------------------------

r8 = pd.read_parquet(MODEL_PATH)
ablation = pd.read_parquet(ABLATION_PATH)

# ------------------------------------------------------------
# 2. Normalize merge keys
# ------------------------------------------------------------

def normalize_player_id(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64")

r8["nfl_id"] = normalize_player_id(r8["nfl_id"])
ablation["nfl_id"] = normalize_player_id(ablation["nfl_id"])

r8["held_out_cohort"] = (
    pd.to_numeric(r8["held_out_cohort"], errors="coerce")
    .astype("Int64")
)

ablation["held_out_cohort"] = (
    pd.to_numeric(ablation["held_out_cohort"], errors="coerce")
    .astype("Int64")
)

# ------------------------------------------------------------
# 3. Reconstruct R8 traditional player predictions
# ------------------------------------------------------------

r8_trad = r8[
    (r8["outcome"] == "game_peak_speed_raw_median") &
    (r8["model"] == "A_TRADITIONAL")
].copy()

r8_player = (
    r8_trad.groupby(
        ["held_out_cohort", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean")
    )
)

r8_player["r8_absolute_error"] = (
    r8_player["actual"] - r8_player["predicted"]
).abs()

# ------------------------------------------------------------
# 4. Extract Step 11A full model
# ------------------------------------------------------------

a0 = ablation[
    ablation["specification"] == "A0_FULL"
].copy()

a0 = a0.rename(columns={
    "predicted": "a0_predicted",
    "actual": "a0_actual"
})

a0["a0_absolute_error"] = (
    a0["a0_actual"] - a0["a0_predicted"]
).abs()

# ------------------------------------------------------------
# 5. Validate keys and merge
# ------------------------------------------------------------

keys = ["held_out_cohort", "nfl_id"]

assert r8_player[keys].notna().all().all()
assert a0[keys].notna().all().all()

assert not r8_player.duplicated(keys).any()
assert not a0.duplicated(keys).any()

comparison = r8_player.merge(
    a0[
        keys + [
            "a0_actual",
            "a0_predicted",
            "a0_absolute_error"
        ]
    ],
    on=keys,
    how="inner",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 6. Sample overlap audit
# ------------------------------------------------------------

r8_keys = set(map(tuple, r8_player[keys].to_numpy()))
a0_keys = set(map(tuple, a0[keys].to_numpy()))

print("=" * 90)
print("STEP 11C — SAMPLE OVERLAP")
print("=" * 90)

print("R8 player records:", len(r8_player))
print("Step 11A player records:", len(a0))
print("Matched records:", len(comparison))
print("R8 only:", len(r8_keys - a0_keys))
print("Step 11A only:", len(a0_keys - r8_keys))
print("Matched unique players:", comparison["nfl_id"].nunique())

# ------------------------------------------------------------
# 7. Cohort-specific comparison
# ------------------------------------------------------------

cohort_comparison = (
    comparison.groupby("held_out_cohort")
    .agg(
        players=("nfl_id", "nunique"),
        r8_mae=("r8_absolute_error", "mean"),
        step11a_mae=("a0_absolute_error", "mean")
    )
    .reset_index()
)

cohort_comparison["mae_difference"] = (
    cohort_comparison["step11a_mae"] -
    cohort_comparison["r8_mae"]
)

print("\n" + "=" * 90)
print("COHORT-SPECIFIC MAE")
print("=" * 90)

print(cohort_comparison.to_string(index=False))

# ------------------------------------------------------------
# 8. Pooled comparison
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("POOLED MATCHED-SAMPLE COMPARISON")
print("=" * 90)

print("R8 MAE:", comparison["r8_absolute_error"].mean())
print("Step 11A MAE:", comparison["a0_absolute_error"].mean())

print(
    "MAE difference:",
    comparison["a0_absolute_error"].mean() -
    comparison["r8_absolute_error"].mean()
)

# ------------------------------------------------------------
# 9. Prediction difference diagnostics
# ------------------------------------------------------------

comparison["prediction_difference"] = (
    comparison["a0_predicted"] -
    comparison["predicted"]
)

print("\nPrediction difference:")
print(
    comparison["prediction_difference"]
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# 10. Save
# ------------------------------------------------------------

comparison.to_csv(
    OUT_DIR / "step11c_r8_ablation_reconciliation.csv",
    index=False
)

comparison.to_parquet(
    OUT_DIR / "step11c_r8_ablation_reconciliation.parquet",
    index=False
)

cohort_comparison.to_csv(
    OUT_DIR / "step11c_cohort_mae_comparison.csv",
    index=False
)

print("\nSUCCESS: Step 11C reconciliation saved.")
print("Comparison shape:", comparison.shape)


STEP 11C — SAMPLE OVERLAP
R8 player records: 66
Step 11A player records: 55
Matched records: 55
R8 only: 11
Step 11A only: 0
Matched unique players: 55

COHORT-SPECIFIC MAE
 held_out_cohort  players   r8_mae  step11a_mae  mae_difference
            2023       21 0.413873     0.414815        0.000942
            2024       20 0.447821     0.484069        0.036249
            2025       14 0.311315     0.291277       -0.020038

POOLED MATCHED-SAMPLE COMPARISON
R8 MAE: 0.40011168771490885
Step 11A MAE: 0.40855218001812466
MAE difference: 0.008440492303215807

Prediction difference:
count    55.000000
mean      0.004983
std       0.148204
min      -0.359535
25%      -0.086669
50%       0.015101
75%       0.089281
max       0.406384

SUCCESS: Step 11C reconciliation saved.
Comparison shape: (55, 9)


In [44]:
# STEP 11D — SELECTED ALPHA AUDIT

alpha_path = (
    ROOT / "reports" / "model_results" /
    "step11a_selected_alphas.csv"
)

alphas = pd.read_csv(alpha_path)

print("ALPHA SELECTION BY COHORT AND SPECIFICATION")
print(
    alphas.sort_values(
        ["held_out_cohort", "specification"]
    ).to_string(index=False)
)

print("\nALPHA FREQUENCY")
print(
    alphas["best_alpha"]
    .value_counts()
    .sort_index()
)

print("\nALPHA FREQUENCY BY SPECIFICATION")
print(
    alphas.groupby("specification")["best_alpha"]
    .value_counts()
    .unstack(fill_value=0)
)

ALPHA SELECTION BY COHORT AND SPECIFICATION
 held_out_cohort  specification  best_alpha  inner_best_mae  train_players  test_players
            2023        A0_FULL          10        0.495037             34            21
            2023    A1_NO_FORTY      100000        0.500745             34            21
            2023 A2_NO_VERTICAL          10        0.476385             34            21
            2023    A3_NO_BROAD         100        0.499388             34            21
            2024        A0_FULL          10        0.409149             35            20
            2024    A1_NO_FORTY      100000        0.476375             35            20
            2024 A2_NO_VERTICAL          10        0.403050             35            20
            2024    A3_NO_BROAD          10        0.408280             35            20
            2025        A0_FULL          10        0.474296             41            14
            2025    A1_NO_FORTY      100000        0.487604       

In [45]:

# ============================================================
# STEP 11E — RIDGE COLLAPSE AND ALPHA-CURVE DIAGNOSTICS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, GroupKFold
from pathlib import Path

DATA_PATH = (
    ROOT / "data" / "features" /
    "wr_modeling_dataset_primary_10plus.parquet"
)

OUT_DIR = ROOT / "reports" / "model_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME = "game_peak_speed_raw_median"
GROUP = "nfl_id"
COHORT = "draft_year"

ALPHAS = [
    0.001, 0.01, 0.1, 1, 10,
    100, 1000, 10000, 100000
]

SPECIFICATIONS = {
    "A0_FULL": ["forty", "vertical", "broad_jump"],
    "A1_NO_FORTY": ["vertical", "broad_jump"],
    "A2_NO_VERTICAL": ["forty", "broad_jump"],
    "A3_NO_BROAD": ["forty", "vertical"]
}

# ------------------------------------------------------------
# 1. Load the exact complete-case sample
# ------------------------------------------------------------

df = pd.read_parquet(DATA_PATH).copy()

for col in [
    OUTCOME, "forty", "vertical", "broad_jump"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=[
        OUTCOME, GROUP, COHORT,
        "forty", "vertical", "broad_jump"
    ]
).copy()

assert len(df) == 95
assert df[GROUP].nunique() == 55

# ------------------------------------------------------------
# 2. Fit every specification and cohort split
# ------------------------------------------------------------

diagnostic_rows = []
curve_rows = []

for heldout in sorted(df[COHORT].unique()):

    train = df[df[COHORT] != heldout].copy()
    test = df[df[COHORT] == heldout].copy()

    cv = GroupKFold(n_splits=4)

    for spec, features in SPECIFICATIONS.items():

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge())
        ])

        search = GridSearchCV(
            pipeline,
            param_grid={"ridge__alpha": ALPHAS},
            scoring="neg_mean_absolute_error",
            cv=cv,
            n_jobs=-1,
            refit=True
        )

        search.fit(
            train[features],
            train[OUTCOME],
            groups=train[GROUP]
        )

        predictions = search.predict(test[features])

        # Selected model
        best_alpha = search.best_params_["ridge__alpha"]
        test_mae = np.mean(
            np.abs(test[OUTCOME].to_numpy() - predictions)
        )

        # Baseline: training median
        baseline_prediction = train[OUTCOME].median()

        baseline_mae = np.mean(
            np.abs(
                test[OUTCOME].to_numpy() -
                baseline_prediction
            )
        )

        # Coefficients after standardization
        coefficients = (
            search.best_estimator_
            .named_steps["ridge"].coef_
        )

        coefficient_norm = np.linalg.norm(coefficients)

        # CV curve for every alpha
        cv_maes = -search.cv_results_["mean_test_score"]

        for alpha, cv_mae in zip(ALPHAS, cv_maes):

            curve_rows.append({
                "held_out_cohort": int(heldout),
                "specification": spec,
                "alpha": alpha,
                "inner_cv_mae": cv_mae,
                "relative_to_best_pct": (
                    (cv_mae - cv_maes.min()) /
                    cv_maes.min() * 100
                )
            })

        diagnostic_rows.append({
            "held_out_cohort": int(heldout),
            "specification": spec,
            "features": ", ".join(features),
            "best_alpha": best_alpha,
            "inner_best_mae": -search.best_score_,
            "test_mae": test_mae,
            "baseline_mae": baseline_mae,
            "improvement_vs_baseline_pct": (
                (baseline_mae - test_mae) /
                baseline_mae * 100
            ),
            "prediction_sd": np.std(predictions, ddof=1),
            "actual_sd": test[OUTCOME].std(),
            "coefficient_norm": coefficient_norm,
            "test_players": test[GROUP].nunique()
        })

diagnostics = pd.DataFrame(diagnostic_rows)
curves = pd.DataFrame(curve_rows)

# ------------------------------------------------------------
# 3. Display model collapse audit
# ------------------------------------------------------------

print("=" * 100)
print("STEP 11E — MODEL COLLAPSE DIAGNOSTICS")
print("=" * 100)

print(
    diagnostics.sort_values(
        ["held_out_cohort", "specification"]
    ).to_string(index=False)
)

# ------------------------------------------------------------
# 4. Display alpha curves
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("STEP 11E — INNER CV ALPHA CURVES")
print("=" * 100)

for (cohort, spec), group in curves.groupby(
    ["held_out_cohort", "specification"]
):

    print(f"\nCohort: {cohort} | Model: {spec}")

    print(
        group[
            ["alpha", "inner_cv_mae", "relative_to_best_pct"]
        ].sort_values("alpha").to_string(index=False)
    )

# ------------------------------------------------------------
# 5. Save outputs
# ------------------------------------------------------------

diagnostics.to_csv(
    OUT_DIR / "step11e_ridge_collapse_diagnostics.csv",
    index=False
)

diagnostics.to_parquet(
    OUT_DIR / "step11e_ridge_collapse_diagnostics.parquet",
    index=False
)

curves.to_csv(
    OUT_DIR / "step11e_ridge_alpha_curves.csv",
    index=False
)

curves.to_parquet(
    OUT_DIR / "step11e_ridge_alpha_curves.parquet",
    index=False
)

# Read-back validation
assert pd.read_parquet(
    OUT_DIR / "step11e_ridge_collapse_diagnostics.parquet"
).shape == diagnostics.shape

assert pd.read_parquet(
    OUT_DIR / "step11e_ridge_alpha_curves.parquet"
).shape == curves.shape

print("\nSUCCESS: Step 11E diagnostics saved and verified.")
print("Diagnostic rows:", len(diagnostics))
print("Alpha-curve rows:", len(curves))


STEP 11E — MODEL COLLAPSE DIAGNOSTICS
 held_out_cohort  specification                    features  best_alpha  inner_best_mae  test_mae  baseline_mae  improvement_vs_baseline_pct  prediction_sd  actual_sd  coefficient_norm  test_players
            2023        A0_FULL forty, vertical, broad_jump          10        0.495037  0.465652      0.481383                     3.267898       0.164022   0.670972          0.197889            21
            2023    A1_NO_FORTY        vertical, broad_jump      100000        0.500745  0.490913      0.481383                    -1.979719       0.000036   0.670972          0.000043            21
            2023 A2_NO_VERTICAL           forty, broad_jump          10        0.476385  0.495666      0.481383                    -2.967127       0.177508   0.670972          0.186856            21
            2023    A3_NO_BROAD             forty, vertical         100        0.499388  0.470425      0.481383                     2.276273       0.062970   0.670972

In [46]:

# ============================================================
# STEP 11F — FIXED-ALPHA FEATURE ABLATION
# Same complete-case sample, same cohort splits, alpha=10
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

DATA_PATH = (
    ROOT / "data" / "features" /
    "wr_modeling_dataset_primary_10plus.parquet"
)

OUT_DIR = ROOT / "reports" / "model_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME = "game_peak_speed_raw_median"
GROUP = "nfl_id"
COHORT = "draft_year"
FIXED_ALPHA = 10

SPECIFICATIONS = {
    "A0_FULL": ["forty", "vertical", "broad_jump"],
    "A1_NO_FORTY": ["vertical", "broad_jump"],
    "A2_NO_VERTICAL": ["forty", "broad_jump"],
    "A3_NO_BROAD": ["forty", "vertical"]
}

# ------------------------------------------------------------
# 1. Load exact complete-case sample
# ------------------------------------------------------------

df = pd.read_parquet(DATA_PATH).copy()

for col in [
    OUTCOME, "forty", "vertical", "broad_jump"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=[
        OUTCOME, GROUP, COHORT,
        "forty", "vertical", "broad_jump"
    ]
).copy()

assert len(df) == 95
assert df[GROUP].nunique() == 55

# ------------------------------------------------------------
# 2. Fit fixed-alpha models by held-out cohort
# ------------------------------------------------------------

prediction_rows = []
diagnostic_rows = []

for heldout in sorted(df[COHORT].unique()):

    train = df[df[COHORT] != heldout].copy()
    test = df[df[COHORT] == heldout].copy()

    assert set(train[GROUP]).isdisjoint(set(test[GROUP]))

    baseline_value = train[OUTCOME].median()

    for spec, features in SPECIFICATIONS.items():

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=FIXED_ALPHA))
        ])

        model.fit(train[features], train[OUTCOME])

        pred = model.predict(test[features])

        errors = np.abs(test[OUTCOME].to_numpy() - pred)

        coefficients = (
            model.named_steps["ridge"].coef_
        )

        diagnostic_rows.append({
            "held_out_cohort": int(heldout),
            "specification": spec,
            "alpha": FIXED_ALPHA,
            "players": test[GROUP].nunique(),
            "test_rows": len(test),
            "test_mae_player_season": errors.mean(),
            "baseline_mae_player_season": np.mean(
                np.abs(
                    test[OUTCOME].to_numpy() - baseline_value
                )
            ),
            "prediction_sd_player_season": np.std(
                pred, ddof=1
            ),
            "actual_sd_player_season": test[OUTCOME].std(),
            "coefficient_norm": np.linalg.norm(coefficients)
        })

        for i, (_, row) in enumerate(test.iterrows()):

            prediction_rows.append({
                "held_out_cohort": int(heldout),
                "specification": spec,
                "nfl_id": row[GROUP],
                "season": row["season"],
                "actual": row[OUTCOME],
                "predicted": pred[i],
                "absolute_error": errors[i],
                "alpha": FIXED_ALPHA
            })

predictions = pd.DataFrame(prediction_rows)
diagnostics = pd.DataFrame(diagnostic_rows)

# ------------------------------------------------------------
# 3. Aggregate predictions to player level
# ------------------------------------------------------------

player_predictions = (
    predictions.groupby(
        ["held_out_cohort", "specification", "nfl_id"],
        as_index=False
    )
    .agg(
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
        seasons=("season", "nunique")
    )
)

player_predictions["absolute_error"] = (
    player_predictions["actual"] -
    player_predictions["predicted"]
).abs()

# ------------------------------------------------------------
# 4. Calculate player-level MAE
# ------------------------------------------------------------

metric_rows = []

cohorts = sorted(df[COHORT].unique())

for cohort in cohorts + ["ALL_COHORTS"]:

    subset = player_predictions.copy()

    if cohort != "ALL_COHORTS":
        subset = subset[
            subset["held_out_cohort"] == cohort
        ]

    for spec in SPECIFICATIONS:

        part = subset[
            subset["specification"] == spec
        ]

        metric_rows.append({
            "cohort": str(cohort),
            "specification": spec,
            "players": part["nfl_id"].nunique(),
            "mae": part["absolute_error"].mean()
        })

metrics = pd.DataFrame(metric_rows)

# ------------------------------------------------------------
# 5. Paired player bootstrap
# Positive improvement means full model is better
# ------------------------------------------------------------

rng = np.random.default_rng(20271118)

bootstrap_rows = []

for cohort in cohorts + ["ALL_COHORTS"]:

    subset = player_predictions.copy()

    if cohort != "ALL_COHORTS":
        subset = subset[
            subset["held_out_cohort"] == cohort
        ]

    wide = subset.pivot(
        index="nfl_id",
        columns="specification",
        values="absolute_error"
    )

    for ablated in [
        "A1_NO_FORTY",
        "A2_NO_VERTICAL",
        "A3_NO_BROAD"
    ]:

        paired = wide[
            ["A0_FULL", ablated]
        ].dropna()

        full_error = paired["A0_FULL"].to_numpy()
        ablated_error = paired[ablated].to_numpy()

        n = len(paired)

        full_mae = full_error.mean()
        ablated_mae = ablated_error.mean()

        difference = ablated_mae - full_mae

        draws = []

        for _ in range(5000):

            idx = rng.integers(0, n, size=n)

            f = full_error[idx].mean()
            a = ablated_error[idx].mean()

            draws.append(a - f)

        draws = np.asarray(draws)

        bootstrap_rows.append({
            "cohort": str(cohort),
            "full_model": "A0_FULL",
            "ablated_model": ablated,
            "players": n,
            "full_mae": full_mae,
            "ablated_mae": ablated_mae,
            "mae_difference": difference,
            "ci_difference_lower": np.percentile(
                draws, 2.5
            ),
            "ci_difference_upper": np.percentile(
                draws, 97.5
            ),
            "probability_full_model_better": np.mean(
                draws > 0
            )
        })

bootstrap = pd.DataFrame(bootstrap_rows)

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("=" * 100)
print("STEP 11F — FIXED ALPHA PLAYER-LEVEL MAE")
print("=" * 100)

print(
    metrics.sort_values(
        ["cohort", "mae"]
    ).to_string(index=False)
)

print("\n" + "=" * 100)
print("STEP 11F — POOLED PAIRED BOOTSTRAP")
print("=" * 100)

print(
    bootstrap[
        bootstrap["cohort"] == "ALL_COHORTS"
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("STEP 11F — MODEL VARIATION DIAGNOSTICS")
print("=" * 100)

print(diagnostics.to_string(index=False))

# ------------------------------------------------------------
# 7. Save and verify
# ------------------------------------------------------------

outputs = {
    "step11f_fixed_alpha_predictions": predictions,
    "step11f_fixed_alpha_player_predictions": player_predictions,
    "step11f_fixed_alpha_metrics": metrics,
    "step11f_fixed_alpha_bootstrap": bootstrap,
    "step11f_fixed_alpha_diagnostics": diagnostics
}

for name, data in outputs.items():

    csv_path = OUT_DIR / f"{name}.csv"
    parquet_path = OUT_DIR / f"{name}.parquet"

    data.to_csv(csv_path, index=False)
    data.to_parquet(parquet_path, index=False)

    check = pd.read_parquet(parquet_path)
    assert check.shape == data.shape

print("\nSUCCESS: Step 11F outputs saved and verified.")

for name, data in outputs.items():
    print(name, data.shape)


STEP 11F — FIXED ALPHA PLAYER-LEVEL MAE
     cohort  specification  players      mae
       2023    A3_NO_BROAD       21 0.410025
       2023        A0_FULL       21 0.414815
       2023    A1_NO_FORTY       21 0.430721
       2023 A2_NO_VERTICAL       21 0.450874
       2024    A1_NO_FORTY       20 0.450192
       2024 A2_NO_VERTICAL       20 0.470661
       2024    A3_NO_BROAD       20 0.475908
       2024        A0_FULL       20 0.484069
       2025 A2_NO_VERTICAL       14 0.271517
       2025    A3_NO_BROAD       14 0.289585
       2025        A0_FULL       14 0.291277
       2025    A1_NO_FORTY       14 0.406671
ALL_COHORTS    A3_NO_BROAD       55 0.403325
ALL_COHORTS        A0_FULL       55 0.408552
ALL_COHORTS A2_NO_VERTICAL       55 0.412415
ALL_COHORTS    A1_NO_FORTY       55 0.431679

STEP 11F — POOLED PAIRED BOOTSTRAP
     cohort full_model  ablated_model  players  full_mae  ablated_mae  mae_difference  ci_difference_lower  ci_difference_upper  probability_full_model_better


In [47]:
import pandas as pd

path = (
    ROOT / "reports" / "model_results" /
    "step11f_fixed_alpha_player_predictions.parquet"
)

p = pd.read_parquet(path)

print("Shape:", p.shape)
print("\nColumns:")
print(p.columns.tolist())

print("\nSample:")
print(p.head(10).to_string(index=False))

print("\nCohort counts:")
print(p.groupby("held_out_cohort").size())

Shape: (220, 7)

Columns:
['held_out_cohort', 'specification', 'nfl_id', 'actual', 'predicted', 'seasons', 'absolute_error']

Sample:
 held_out_cohort specification  nfl_id   actual  predicted  seasons  absolute_error
            2023       A0_FULL   55887 7.126667   6.782123        3        0.344543
            2023       A0_FULL   55904 6.655000   6.739516        2        0.084516
            2023       A0_FULL   55915 7.161667   6.877390        3        0.284276
            2023       A0_FULL   55920 5.338333   6.569605        3        1.231272
            2023       A0_FULL   55928 7.131667   6.917404        3        0.214262
            2023       A0_FULL   55938 7.558333   6.916312        3        0.642022
            2023       A0_FULL   55939 6.721667   6.633005        3        0.088662
            2023       A0_FULL   55959 6.671667   6.497976        3        0.173691
            2023       A0_FULL   55965 7.598333   6.888284        3        0.710050
            2023       A0_

In [49]:

# ============================================================
# STEP 11G — COHORT-STRATIFIED PAIRED PLAYER BOOTSTRAP
# CORRECTED VERSION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

PRED_PATH = (
    ROOT / "reports" / "model_results" /
    "step11f_fixed_alpha_player_predictions.parquet"
)

DATA_PATH = (
    ROOT / "data" / "features" /
    "wr_modeling_dataset_primary_10plus.parquet"
)

OUT_DIR = ROOT / "reports" / "model_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME = "game_peak_speed_raw_median"
N_BOOT = 10000
RANDOM_STATE = 20261118

SPECIFICATIONS = [
    "A0_FULL",
    "A1_NO_FORTY",
    "A2_NO_VERTICAL",
    "A3_NO_BROAD"
]

ABLATIONS = [
    "A1_NO_FORTY",
    "A2_NO_VERTICAL",
    "A3_NO_BROAD"
]

rng = np.random.default_rng(RANDOM_STATE)

# ------------------------------------------------------------
# 1. Load player-level predictions
# ------------------------------------------------------------

pred = pd.read_parquet(PRED_PATH).copy()

required = [
    "held_out_cohort",
    "specification",
    "nfl_id",
    "actual",
    "predicted",
    "absolute_error"
]

missing = [c for c in required if c not in pred.columns]
assert not missing, f"Missing columns: {missing}"

pred["held_out_cohort"] = pd.to_numeric(
    pred["held_out_cohort"]
).astype(int)

pred["nfl_id"] = pd.to_numeric(
    pred["nfl_id"]
).astype("int64")

pred["absolute_error"] = (
    pred["actual"] - pred["predicted"]
).abs()

assert set(pred["specification"].unique()) == set(SPECIFICATIONS)

assert not pred.duplicated(
    ["held_out_cohort", "specification", "nfl_id"]
).any()

# ------------------------------------------------------------
# 2. Reconstruct training-median baseline
# ------------------------------------------------------------

source = pd.read_parquet(DATA_PATH).copy()

source["nfl_id"] = pd.to_numeric(
    source["nfl_id"]
).astype("int64")

source["draft_year"] = pd.to_numeric(
    source["draft_year"]
).astype(int)

source[OUTCOME] = pd.to_numeric(
    source[OUTCOME], errors="coerce"
)

source = source.dropna(
    subset=[
        "nfl_id",
        "draft_year",
        OUTCOME,
        "forty",
        "vertical",
        "broad_jump"
    ]
).copy()

player_actuals = (
    source.groupby(
        ["draft_year", "nfl_id"],
        as_index=False
    )
    .agg(actual=(OUTCOME, "mean"))
)

baseline_rows = []

for cohort in sorted(pred["held_out_cohort"].unique()):

    training = player_actuals[
        player_actuals["draft_year"] != cohort
    ]

    testing = player_actuals[
        player_actuals["draft_year"] == cohort
    ]

    training_median = training["actual"].median()

    for row in testing.itertuples(index=False):

        baseline_rows.append({
            "held_out_cohort": cohort,
            "nfl_id": int(row.nfl_id),
            "baseline_prediction": training_median,
            "actual_source": row.actual
        })

baseline = pd.DataFrame(baseline_rows)

baseline["baseline_absolute_error"] = (
    baseline["actual_source"] -
    baseline["baseline_prediction"]
).abs()

# ------------------------------------------------------------
# 3. Reconcile actual outcomes
# ------------------------------------------------------------

full_actuals = pred[
    pred["specification"] == "A0_FULL"
][
    ["held_out_cohort", "nfl_id", "actual"]
].copy()

reconciliation = full_actuals.merge(
    baseline,
    on=["held_out_cohort", "nfl_id"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert (reconciliation["_merge"] == "both").all()

assert np.allclose(
    reconciliation["actual"],
    reconciliation["actual_source"],
    atol=1e-8
)

# ------------------------------------------------------------
# 4. Create paired error matrix
# ------------------------------------------------------------

wide = pred.pivot(
    index=["held_out_cohort", "nfl_id"],
    columns="specification",
    values="absolute_error"
).reset_index()

wide = wide.merge(
    baseline[
        [
            "held_out_cohort",
            "nfl_id",
            "baseline_absolute_error"
        ]
    ],
    on=["held_out_cohort", "nfl_id"],
    how="inner",
    validate="one_to_one"
)

assert len(wide) == 55
assert wide[SPECIFICATIONS].notna().all().all()

# ------------------------------------------------------------
# 5. Bootstrap helper
# ------------------------------------------------------------

def bootstrap_difference(a, b, rng, n_boot=10000):

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    assert len(a) == len(b)

    n = len(a)

    observed = np.mean(a - b)

    indices = rng.integers(
        0,
        n,
        size=(n_boot, n)
    )

    bootstrap_values = np.mean(
        a[indices] - b[indices],
        axis=1
    )

    return {
        "mae_difference": observed,
        "ci_lower": float(
            np.percentile(bootstrap_values, 2.5)
        ),
        "ci_upper": float(
            np.percentile(bootstrap_values, 97.5)
        ),
        "probability_a_better": float(
            np.mean(bootstrap_values < 0)
        )
    }

# ------------------------------------------------------------
# 6. Cohort-specific MAE and bootstrap
# ------------------------------------------------------------

mae_rows = []
comparison_rows = []

for cohort in sorted(wide["held_out_cohort"].unique()):

    group = wide[
        wide["held_out_cohort"] == cohort
    ].copy()

    n = len(group)

    print(f"Processing cohort {cohort}: {n} players")

    # Individual model MAEs and bootstrap intervals
    for spec in SPECIFICATIONS:

        errors = group[spec].to_numpy()

        indices = rng.integers(
            0, n, size=(N_BOOT, n)
        )

        boot_mae = errors[indices].mean(axis=1)

        mae_rows.append({
            "held_out_cohort": cohort,
            "specification": spec,
            "players": n,
            "mae": errors.mean(),
            "mae_ci_lower": np.percentile(boot_mae, 2.5),
            "mae_ci_upper": np.percentile(boot_mae, 97.5)
        })

    # Training median baseline
    baseline_errors = group[
        "baseline_absolute_error"
    ].to_numpy()

    indices = rng.integers(
        0, n, size=(N_BOOT, n)
    )

    boot_baseline = baseline_errors[indices].mean(axis=1)

    mae_rows.append({
        "held_out_cohort": cohort,
        "specification": "TRAINING_MEDIAN_BASELINE",
        "players": n,
        "mae": baseline_errors.mean(),
        "mae_ci_lower": np.percentile(boot_baseline, 2.5),
        "mae_ci_upper": np.percentile(boot_baseline, 97.5)
    })

    # Full model versus each ablation
    for ablated in ABLATIONS:

        result = bootstrap_difference(
            group["A0_FULL"],
            group[ablated],
            rng,
            N_BOOT
        )

        if result["ci_upper"] < 0:
            interpretation = "FULL_BETTER"
        elif result["ci_lower"] > 0:
            interpretation = "ABLATION_BETTER"
        else:
            interpretation = "NO_CLEAR_DIFFERENCE"

        comparison_rows.append({
            "held_out_cohort": cohort,
            "comparison": f"A0_FULL_vs_{ablated}",
            "players": n,
            "mae_difference": result["mae_difference"],
            "ci_lower": result["ci_lower"],
            "ci_upper": result["ci_upper"],
            "probability_a_better": result["probability_a_better"],
            "interpretation": interpretation
        })

    # Every specification versus training baseline
    for spec in SPECIFICATIONS:

        result = bootstrap_difference(
            group[spec],
            group["baseline_absolute_error"],
            rng,
            N_BOOT
        )

        if result["ci_upper"] < 0:
            interpretation = "MODEL_BETTER"
        elif result["ci_lower"] > 0:
            interpretation = "BASELINE_BETTER"
        else:
            interpretation = "NO_CLEAR_DIFFERENCE"

        comparison_rows.append({
            "held_out_cohort": cohort,
            "comparison": f"{spec}_vs_TRAINING_BASELINE",
            "players": n,
            "mae_difference": result["mae_difference"],
            "ci_lower": result["ci_lower"],
            "ci_upper": result["ci_upper"],
            "probability_a_better": result["probability_a_better"],
            "interpretation": interpretation
        })

# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

mae_results = pd.DataFrame(mae_rows)
comparisons = pd.DataFrame(comparison_rows)

print("\n" + "=" * 100)
print("STEP 11G — COHORT-SPECIFIC PLAYER-LEVEL MAE")
print("=" * 100)

print(
    mae_results.sort_values(
        ["held_out_cohort", "mae"]
    ).to_string(index=False)
)

print("\n" + "=" * 100)
print("STEP 11G — PAIRED BOOTSTRAP COMPARISONS")
print("=" * 100)

print(
    comparisons.sort_values(
        ["held_out_cohort", "comparison"]
    ).to_string(index=False)
)

# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

mae_results.to_csv(
    OUT_DIR / "step11g_cohort_mae_bootstrap.csv",
    index=False
)

mae_results.to_parquet(
    OUT_DIR / "step11g_cohort_mae_bootstrap.parquet",
    index=False
)

comparisons.to_csv(
    OUT_DIR / "step11g_cohort_paired_comparisons.csv",
    index=False
)

comparisons.to_parquet(
    OUT_DIR / "step11g_cohort_paired_comparisons.parquet",
    index=False
)

# ------------------------------------------------------------
# 9. Verify saved artifacts
# ------------------------------------------------------------

mae_check = pd.read_parquet(
    OUT_DIR / "step11g_cohort_mae_bootstrap.parquet"
)

comparison_check = pd.read_parquet(
    OUT_DIR / "step11g_cohort_paired_comparisons.parquet"
)

assert mae_check.shape == mae_results.shape
assert comparison_check.shape == comparisons.shape

assert mae_check["held_out_cohort"].nunique() == 3
assert comparison_check["held_out_cohort"].nunique() == 3

print("\nSUCCESS — STEP 11G COMPLETE")
print("MAE results:", mae_check.shape)
print("Comparison results:", comparison_check.shape)
print("Saved directory:", OUT_DIR)


Processing cohort 2023: 21 players
Processing cohort 2024: 20 players
Processing cohort 2025: 14 players

STEP 11G — COHORT-SPECIFIC PLAYER-LEVEL MAE
 held_out_cohort            specification  players      mae  mae_ci_lower  mae_ci_upper
            2023              A3_NO_BROAD       21 0.410025      0.290215      0.544037
            2023 TRAINING_MEDIAN_BASELINE       21 0.410357      0.262061      0.581755
            2023                  A0_FULL       21 0.414815      0.293360      0.549305
            2023              A1_NO_FORTY       21 0.430721      0.288994      0.591285
            2023           A2_NO_VERTICAL       21 0.450874      0.328916      0.584133
            2024              A1_NO_FORTY       20 0.450192      0.285401      0.642116
            2024           A2_NO_VERTICAL       20 0.470661      0.312778      0.643715
            2024 TRAINING_MEDIAN_BASELINE       20 0.473625      0.281247      0.699509
            2024              A3_NO_BROAD       20 0.47590

In [1]:
from pathlib import Path
import shutil

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")
SOURCE = ROOT / "data/features"
DEST = ROOT / "data/reference"

DEST.mkdir(parents=True, exist_ok=True)

files = [
    "wr_combine_player_drill_profiles.parquet",
    "wr_combine_player_profiles.parquet",
    "wr_game_player_season_profiles.parquet",
]

for filename in files:
    src = SOURCE / filename
    dst = DEST / filename

    if not src.exists():
        print(f"MISSING: {src}")
        continue

    shutil.copy2(src, dst)
    print(f"COPIED: {filename} ({dst.stat().st_size:,} bytes)")

print("\nReference files ready.")

MISSING: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_player_drill_profiles.parquet
MISSING: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_combine_player_profiles.parquet
MISSING: /content/gdrive_new/MyDrive/NFL-BigDataBowl-2027/data/features/wr_game_player_season_profiles.parquet

Reference files ready.


In [2]:
from pathlib import Path

ROOT = Path("/content/gdrive_new/MyDrive/NFL-BigDataBowl-2027")

targets = [
    "wr_combine_player_drill_profiles.parquet",
    "wr_combine_player_profiles.parquet",
    "wr_game_player_season_profiles.parquet",
]

for target in targets:
    matches = list(ROOT.rglob(target))
    print(f"\n{target}")
    if matches:
        for path in matches:
            print(path)
    else:
        print("NOT FOUND")


wr_combine_player_drill_profiles.parquet
NOT FOUND

wr_combine_player_profiles.parquet
NOT FOUND

wr_game_player_season_profiles.parquet
NOT FOUND
